In [ ]:
# Cell 1: Environment setup and data load

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Load Data
import os; RAW_PATH = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit/cs-training.csv"

df_raw = pd.read_csv(RAW_PATH, index_col=0)

print("Shape:", df_raw.shape)
print("\nColumn names (original):")
print(df_raw.columns.tolist())
df_raw.head()

In [ ]:
# Cell 2: Rename columns, check dtypes and missingness

rename_map = {
    'SeriousDlqin2yrs': 'Default',
    'RevolvingUtilizationOfUnsecuredLines': 'Credit Utilisation Rate',
    'age': 'Age',
    'NumberOfTime30-59DaysPastDueNotWorse': '30-59 Days Past Due',
    'DebtRatio': 'Debt Ratio',
    'MonthlyIncome': 'Monthly Income',
    'NumberOfOpenCreditLinesAndLoans': 'Open Credit Lines',
    'NumberOfTimes90DaysLate': '90+ Days Late',
    'NumberRealEstateLoansOrLines': 'Real Estate Loans',
    'NumberOfTime60-89DaysPastDueNotWorse': '60-89 Days Past Due',
    'NumberOfDependents': 'Number of Dependents',
}

df = df_raw.rename(columns=rename_map)

# Reorder: target first, then features in dictionary order
col_order = ['Default', 'Credit Utilisation Rate', 'Age', '30-59 Days Past Due',
             'Debt Ratio', 'Monthly Income', 'Open Credit Lines', '90+ Days Late',
             'Real Estate Loans', '60-89 Days Past Due', 'Number of Dependents']
df = df[col_order]

print("Dtypes:")
print(df.dtypes)

print("\nMissing values (count and %):")
missing_summary = pd.DataFrame({
    'n_missing': df.isnull().sum(),
    'pct_missing': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary)

print(f"\nDefault rate: {df['Default'].mean():.4f}")
print(f"Rows fully complete: {df.dropna().shape[0]}")
print(f"Rows with any missing: {df.shape[0] - df.dropna().shape[0]}")

In [ ]:
# Cell 3: Verify default rate reconciliation and missingness overlap pattern

df_complete = df.dropna()
print(f"Full sample (n={len(df)}) default rate: {df['Default'].mean():.4f}")
print(f"Complete-case (n={len(df_complete)}) default rate: {df_complete['Default'].mean():.4f}")

# Missingness indicator flags
income_missing = df['Monthly Income'].isnull()
deps_missing = df['Number of Dependents'].isnull()

print(f"\nMonthly Income missing: {income_missing.sum()}")
print(f"Number of Dependents missing: {deps_missing.sum()}")
print(f"Both missing: {(income_missing & deps_missing).sum()}")
print(f"Deps missing but Income NOT missing: {(deps_missing & ~income_missing).sum()}")
print(f"Income missing but Deps NOT missing: {(income_missing & ~deps_missing).sum()}")

# Cross-tab
pd.crosstab(income_missing, deps_missing, margins=True, 
            rownames=['Income Missing'], colnames=['Deps Missing'])

In [ ]:
# Cell 4: Distribution summary and value counts for count-like variables

# Continuous variables: describe
continuous_vars = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income']
print("=== Continuous variables: describe() ===")
print(df[continuous_vars].describe())

# Count-like variables: full value_counts to see zero-inflation / discreteness
count_vars = ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due',
              'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents']

print("\n=== Count-like variables: value_counts ===")
for col in count_vars:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().sort_index())

In [ ]:
# Cell 5: Check whether 96/98 sentinel values co-occur in the same rows,
# and inspect the outlier rows in continuous variables

sentinel_mask = (df['30-59 Days Past Due'].isin([96, 98]) |
                  df['90+ Days Late'].isin([96, 98]) |
                  df['60-89 Days Past Due'].isin([96, 98]))
print(f"Rows with sentinel (96/98) in any past-due column: {sentinel_mask.sum()}")

same_rows = ((df['30-59 Days Past Due'].isin([96, 98])) ==
             (df['90+ Days Late'].isin([96, 98]))) & \
            ((df['30-59 Days Past Due'].isin([96, 98])) ==
             (df['60-89 Days Past Due'].isin([96, 98])))
print(f"Sentinel flag identical across all 3 columns for all rows: {same_rows.all()}")

print(f"\nDefault rate among sentinel rows: {df.loc[sentinel_mask, 'Default'].mean():.4f}")
print(f"Default rate among non-sentinel rows: {df.loc[~sentinel_mask, 'Default'].mean():.4f}")

# Age outlier
print(f"\nRows with Age == 0: {(df['Age'] == 0).sum()}")
print(df[df['Age'] == 0])

# Credit Utilisation Rate extreme values
print(f"\nCredit Utilisation Rate > 2: {(df['Credit Utilisation Rate'] > 2).sum()} rows")
print(f"Credit Utilisation Rate > 10: {(df['Credit Utilisation Rate'] > 10).sum()} rows")

# Debt Ratio extreme values
print(f"\nDebt Ratio > 10: {(df['Debt Ratio'] > 10).sum()} rows")
print(f"Debt Ratio > 1000: {(df['Debt Ratio'] > 1000).sum()} rows")

In [ ]:
# Cell 6: Investigate whether extreme Debt Ratio relates to missing/low Monthly Income

high_dr = df['Debt Ratio'] > 10

print("=== Debt Ratio > 10 vs Monthly Income missingness ===")
print(pd.crosstab(high_dr, df['Monthly Income'].isnull(),
                   rownames=['Debt Ratio > 10'], colnames=['Income Missing']))

print(f"\nOf rows with Debt Ratio > 10, income missing: {(high_dr & df['Monthly Income'].isnull()).sum()} / {high_dr.sum()}")

# Among rows where income IS observed, check its value when Debt Ratio is extreme
print("\n=== Monthly Income distribution when Debt Ratio > 10 (income observed) ===")
print(df.loc[high_dr & df['Monthly Income'].notnull(), 'Monthly Income'].describe())

print("\n=== Monthly Income distribution when Debt Ratio <= 10 (income observed) ===")
print(df.loc[~high_dr & df['Monthly Income'].notnull(), 'Monthly Income'].describe())

# Credit Utilisation Rate: how extreme are the top values
print("\n=== Credit Utilisation Rate: top 15 values ===")
print(df['Credit Utilisation Rate'].sort_values(ascending=False).head(15))

In [ ]:
# Cell 7: Apply documented cleaning decisions

df_clean = df.copy()

# 1. Drop the single Age == 0 row (implausible, negligible cost)
df_clean = df_clean[df_clean['Age'] > 0]

# 2. Recode 96/98 sentinel values in past-due variables into the top category
#    (handled properly at the binning stage next — for now, cap at a placeholder
#    high value so they don't distort continuous stats/imputation)
past_due_cols = ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due']
for col in past_due_cols:
    n_sentinel = df_clean[col].isin([96, 98]).sum()
    print(f"{col}: recoding {n_sentinel} sentinel values (96/98)")
    # Cap sentinel values to just above the observed non-sentinel max,
    # so they land in the top bucket rather than distorting scale
    max_valid = df_clean.loc[~df_clean[col].isin([96, 98]), col].max()
    df_clean.loc[df_clean[col].isin([96, 98]), col] = max_valid + 1

# 3. Cap Credit Utilisation Rate at a defensible ceiling
#    Ratios > 2 (i.e. owing 2x+ your total credit limit) are implausible
#    at this frequency; cap rather than drop to preserve sample size
print(f"\nCredit Utilisation Rate > 2: {(df_clean['Credit Utilisation Rate'] > 2).sum()} rows capped")
df_clean['Credit Utilisation Rate'] = df_clean['Credit Utilisation Rate'].clip(upper=2)

print(f"\nFinal shape: {df_clean.shape}")
print(f"\nRe-check describe() after cleaning:")
print(df_clean[['Credit Utilisation Rate', 'Age', 'Debt Ratio']].describe())

In [ ]:
# Cell 8: Histograms of the four candidate equal-frequency variables
# raw scale and log scale where skew is extreme

fig, axes = plt.subplots(4, 2, figsize=(12, 16))

vars_to_plot = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income']

for i, col in enumerate(vars_to_plot):
    data = df_clean[col].dropna()
    axes[i, 0].hist(data, bins=50)
    axes[i, 0].set_title(f'{col} (raw)')
    
    # log1p scale for skewed vars
    axes[i, 1].hist(np.log1p(data), bins=50)
    axes[i, 1].set_title(f'{col} (log1p)')

plt.tight_layout()
plt.show()

# Also check how many unique values each has (relevant for qcut duplicate edges)
for col in vars_to_plot:
    print(f"{col}: {df_clean[col].nunique()} unique values, "
          f"{(df_clean[col] == 0).sum()} zeros ({(df_clean[col]==0).mean()*100:.1f}%)")

In [ ]:
# Cell 9: Equal-frequency (quantile) binning for the four continuous variables

df_binned = df_clean.copy()

equal_freq_vars = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income']
N_BINS = 5

bin_edges = {}  # store for documentation / reuse on train-test consistency later

for col in equal_freq_vars:
    binned_col, edges = pd.qcut(
        df_binned[col], q=N_BINS, duplicates='drop', retbins=True
    )
    df_binned[col + ' (bin)'] = binned_col
    bin_edges[col] = edges
    n_bins_actual = binned_col.nunique(dropna=False) - (1 if binned_col.isnull().any() else 0)
    print(f"{col}: {n_bins_actual} bins created")
    print(binned_col.value_counts(dropna=False).sort_index())
    print()

print("=== Bin edges (for documentation) ===")
for col, edges in bin_edges.items():
    print(f"{col}: {np.round(edges, 4).tolist()}")

In [ ]:
# Cell 10: Natural-category binning for zero-inflated count variables,
# plus equal-frequency binning for Open Credit Lines (should've joined
# the continuous group in Cell 9 — genuinely count-like, not sparse)

def bin_natural_category(series, max_category=3):
    """Bins 0, 1, 2, ..., max_category-1, then max_category+ """
    capped = series.clip(upper=max_category)
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    return pd.Categorical(capped.map(lambda x: labels[int(x)] if x < max_category else labels[-1]),
                           categories=labels, ordered=True)

natural_cat_vars = ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due',
                     'Number of Dependents', 'Real Estate Loans']

for col in natural_cat_vars:
    df_binned[col + ' (bin)'] = bin_natural_category(df_binned[col], max_category=3)
    print(f"--- {col} ---")
    print(df_binned[col + ' (bin)'].value_counts().sort_index())
    print()

# Catch-up: Open Credit Lines via equal-frequency, consistent with the
# other genuinely continuous/count variables
binned_col, edges = pd.qcut(
    df_binned['Open Credit Lines'], q=N_BINS, duplicates='drop', retbins=True
)
df_binned['Open Credit Lines (bin)'] = binned_col
bin_edges['Open Credit Lines'] = edges
print("--- Open Credit Lines ---")
print(binned_col.value_counts(dropna=False).sort_index())
print(f"\nBin edges: {np.round(edges, 4).tolist()}")

In [ ]:
# Cell 10b: Fix bin_natural_category to preserve NaN instead of
# silently routing missing values into the top category

def bin_natural_category(series, max_category=3):
    """Bins 0, 1, 2, ..., max_category-1, then max_category+, preserving NaN."""
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    def map_val(x):
        if pd.isnull(x):
            return np.nan
        return labels[int(x)] if x < max_category else labels[-1]
    mapped = series.map(map_val)
    return pd.Categorical(mapped, categories=labels, ordered=True)

# Re-run only the affected column
df_binned['Number of Dependents (bin)'] = bin_natural_category(
    df_binned['Number of Dependents'], max_category=3
)

print("--- Number of Dependents (fixed) ---")
print(df_binned['Number of Dependents (bin)'].value_counts(dropna=False).sort_index())
print(f"\nTotal (should be 149999): {df_binned['Number of Dependents (bin)'].value_counts(dropna=False).sum()}")
print(f"NaN count (should be 3924): {df_binned['Number of Dependents (bin)'].isnull().sum()}")

In [ ]:
# Cell 11: Consolidate binned dataset, sanity check before moving to
# missing-data handling and BN structure learning

bin_cols = [c for c in df_binned.columns if c.endswith('(bin)')]
model_cols = ['Default'] + bin_cols

df_model = df_binned[model_cols].copy()

print("Columns going into modelling:")
print(df_model.columns.tolist())

print("\nMissingness in final binned dataset:")
print(df_model.isnull().sum())

print(f"\nShape: {df_model.shape}")
print(f"\nDefault rate: {df_model['Default'].mean():.4f}")

# Confirm row-level missingness pattern still matches Cell 3's finding
any_missing = df_model.drop(columns='Default').isnull().any(axis=1)
print(f"\nRows with any missing bin value: {any_missing.sum()} (expect 29731)")

In [ ]:
# Cell 12: Stratified train/test split, seed 42
# Splitting on df_clean (pre-binning, pre-imputation, continuous scale)
# so that ALL downstream fitting (bin edges, imputer, encoder, EM) can be
# done on train only and applied to test, avoiding leakage.

from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df_clean.index,
    test_size=0.2,
    stratify=df_clean['Default'],
    random_state=SEED
)

print(f"Train size: {len(train_idx)}")
print(f"Test size: {len(test_idx)}")
print(f"Train default rate: {df_clean.loc[train_idx, 'Default'].mean():.4f}")
print(f"Test default rate: {df_clean.loc[test_idx, 'Default'].mean():.4f}")

print(f"\nTrain % Monthly Income missing: {df_clean.loc[train_idx, 'Monthly Income'].isnull().mean()*100:.2f}%")
print(f"Test % Monthly Income missing: {df_clean.loc[test_idx, 'Monthly Income'].isnull().mean()*100:.2f}%")
print(f"Train % Number of Dependents missing: {df_clean.loc[train_idx, 'Number of Dependents'].isnull().mean()*100:.2f}%")
print(f"Test % Number of Dependents missing: {df_clean.loc[test_idx, 'Number of Dependents'].isnull().mean()*100:.2f}%")

In [ ]:
# Cell 13: Refit equal-frequency bin edges on TRAIN ONLY, apply same edges to test.
# Supersedes Cell 9, which incorrectly fit on the full dataset.

N_BINS = 5
equal_freq_vars = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines']

df_train = df_clean.loc[train_idx].copy()
df_test = df_clean.loc[test_idx].copy()

bin_edges = {}  # refreshed, train-only edges

for col in equal_freq_vars:
    # Fit edges on train
    _, edges = pd.qcut(df_train[col], q=N_BINS, duplicates='drop', retbins=True)
    bin_edges[col] = edges

    # Apply identical edges to both train and test via pd.cut
    # Extend outer edges slightly to guarantee test extremes aren't dropped as NaN
    edges_adj = edges.copy()
    edges_adj[0] = -np.inf
    edges_adj[-1] = np.inf

    df_train[col + ' (bin)'] = pd.cut(df_train[col], bins=edges_adj, include_lowest=True)
    df_test[col + ' (bin)'] = pd.cut(df_test[col], bins=edges_adj, include_lowest=True)

    print(f"--- {col} ---")
    print("Train bin edges:", np.round(edges, 4).tolist())
    print("Train distribution:")
    print(df_train[col + ' (bin)'].value_counts(dropna=False).sort_index())
    print("Test distribution:")
    print(df_test[col + ' (bin)'].value_counts(dropna=False).sort_index())
    print()

In [ ]:
# Cell 14: Apply natural-category binning (0/1/2/3+) to train and test.
# No leakage risk here since thresholds are fixed integers, not data-derived —
# but we still need these columns present in df_train/df_test for consistency.

def bin_natural_category(series, max_category=3):
    """Bins 0, 1, 2, ..., max_category-1, then max_category+, preserving NaN."""
    labels = [str(i) for i in range(max_category)] + [f"{max_category}+"]
    def map_val(x):
        if pd.isnull(x):
            return np.nan
        return labels[int(x)] if x < max_category else labels[-1]
    mapped = series.map(map_val)
    return pd.Categorical(mapped, categories=labels, ordered=True)

natural_cat_vars = ['30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due',
                     'Number of Dependents', 'Real Estate Loans']

for col in natural_cat_vars:
    df_train[col + ' (bin)'] = bin_natural_category(df_train[col], max_category=3)
    df_test[col + ' (bin)'] = bin_natural_category(df_test[col], max_category=3)

for col in natural_cat_vars:
    print(f"--- {col} ---")
    print("Train:")
    print(df_train[col + ' (bin)'].value_counts(dropna=False).sort_index())
    print("Test:")
    print(df_test[col + ' (bin)'].value_counts(dropna=False).sort_index())
    print()

In [ ]:
# Cell 15: Blocking Check 1 — investigate whether Debt Ratio contains raw
# dollar debt payments (not a true ratio) in income-missing rows

# Split into three groups
group_a = df_raw[df_raw['MonthlyIncome'].isna()]['DebtRatio']
group_b = df_raw[df_raw['MonthlyIncome'].notna() & 
                 (df_raw['MonthlyIncome'] < 100)]['DebtRatio']
group_c = df_raw[df_raw['MonthlyIncome'].notna() & 
                 (df_raw['MonthlyIncome'] > 1000)]['DebtRatio']

for name, group in [('A - Income Missing', group_a), 
                    ('B - Income Near Zero (<$100)', group_b),
                    ('C - Income Normal (>$1000)', group_c)]:
    print(f"\n{name}")
    print(f"Count: {len(group)}")
    print(f"Median: {group.median():.4f}")
    print(f"Mean: {group.mean():.4f}")
    print(f"75th percentile: {group.quantile(0.75):.4f}")
    print(f"90th percentile: {group.quantile(0.90):.4f}")
    print(f"Max: {group.max():.4f}")
    print(f"% values below 5000: {(group < 5000).mean()*100:.1f}%")
    print(f"% values above 10: {(group > 10).mean()*100:.1f}%")

# Side-by-side histogram on log scale
fig, ax = plt.subplots(figsize=(10, 6))
for name, group, color in [('A - Income Missing', group_a, 'red'),
                             ('B - Income Near Zero (<$100)', group_b, 'orange'),
                             ('C - Income Normal (>$1000)', group_c, 'blue')]:
    ax.hist(np.log1p(group), bins=50, alpha=0.5, label=name, color=color, density=True)
ax.set_xlabel('log1p(Debt Ratio)')
ax.set_ylabel('Density')
ax.legend()
ax.set_title('Debt Ratio distribution by income-observation status')
plt.show()

In [ ]:
# Cell 16: Blocking Check 2, updated for pgmpy 1.1.2's DiscreteEM API

import pgmpy
print("pgmpy version:", pgmpy.__version__)

import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.parameter_estimator import DiscreteEM

# Step 1 - Build a 3-node synthetic BN with known CPTs
model = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C')])

cpd_a = TabularCPD('A', 2, [[0.3], [0.7]])
cpd_b = TabularCPD('B', 2, [[0.4], [0.6]])
cpd_c = TabularCPD('C', 2, 
                    [[0.8, 0.6, 0.4, 0.2],
                     [0.2, 0.4, 0.6, 0.8]],
                    evidence=['A', 'B'],
                    evidence_card=[2, 2])

model.add_cpds(cpd_a, cpd_b, cpd_c)
assert model.check_model()

# Step 2 - Generate synthetic complete data from known CPTs
np.random.seed(42)
n = 1000
a_vals = np.random.choice([0, 1], size=n, p=[0.3, 0.7])
b_vals = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
c_probs = np.array([
    [0.8, 0.6, 0.4, 0.2],
    [0.2, 0.4, 0.6, 0.8]
])
c_vals = np.array([
    np.random.choice([0, 1], p=[
        c_probs[0, a*2+b],
        c_probs[1, a*2+b]
    ]) for a, b in zip(a_vals, b_vals)
])

df_toy = pd.DataFrame({'A': a_vals, 'B': b_vals, 'C': c_vals})

# Step 3 - Inject NaN into C for 20% of rows
nan_idx = np.random.choice(n, size=int(0.2*n), replace=False)
df_toy_missing = df_toy.copy()
df_toy_missing.loc[nan_idx, 'C'] = np.nan

print("Toy dataset shape:", df_toy_missing.shape)
print("NaN count in C:", df_toy_missing['C'].isna().sum())

# Step 4 - Run EM using DiscreteEM (initialized instance, per new API)
em_model = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C')])
em_model.fit(df_toy_missing, estimator=DiscreteEM())

# Step 5 - Compare recovered CPTs to known true values
print("\n--- Recovered CPTs vs Known True CPTs ---")
print("\nTrue P(A): [0.3, 0.7]")
print("Recovered P(A):", em_model.get_cpds('A').values.flatten())

print("\nTrue P(B): [0.4, 0.6]")
print("Recovered P(B):", em_model.get_cpds('B').values.flatten())

print("\nTrue P(C|A,B) row 0 (P(C=0)):", [0.8, 0.6, 0.4, 0.2])
print("Recovered P(C|A,B) row 0:", 
      em_model.get_cpds('C').values[0].flatten())

# Step 6 - Compare against listwise deletion baseline
df_complete = df_toy_missing.dropna()
print(f"\nListwise deletion retains {len(df_complete)} of {n} rows")

listwise_model = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C')])
listwise_model.fit(df_complete)

print("\nListwise P(A):", listwise_model.get_cpds('A').values.flatten())
print("Listwise P(C|A,B) row 0:", 
      listwise_model.get_cpds('C').values[0].flatten())

print("\n--- Interpretation ---")
print("If EM recovered CPTs are closer to true values than listwise,")
print("pgmpy EM is handling NaN correctly via marginalisation.")
print("If EM and listwise give identical results, pgmpy is silently")
print("doing listwise deletion and the implementation is broken.")

In [ ]:
# Cell 17: Diagnose why EM output matched listwise deletion exactly

from pgmpy.parameter_estimator import DiscreteEM
import inspect

# Check the signature and docstring
print(inspect.signature(DiscreteEM.__init__))
print(DiscreteEM.__doc__)
print(DiscreteEM.fit.__doc__)

In [ ]:
# Cell 18: Pull the actual DiscreteEM implementation to see how it handles
# partial (row-wise) missingness vs fully-latent columns

import inspect
from pgmpy.parameter_estimator import DiscreteEM

print(inspect.getsource(DiscreteEM.fit))

In [ ]:
# Cell 19: Custom EM implementation — E-step helper
# Computes soft/hard weights for each row given current model CPTs

from pgmpy.inference import VariableElimination
from pgmpy.estimators import MaximumLikelihoodEstimator
import itertools

def compute_row_weights(model, row, missing_vars, infer_engine):
    """
    For a single row, return a list of (full_assignment_dict, weight) tuples.
    - If missing_vars is empty: hard assignment, weight 1.0
    - If missing_vars non-empty: soft weights from VariableElimination posterior
    """
    observed = {var: row[var] for var in row.index if var not in missing_vars and pd.notnull(row[var])}

    if not missing_vars:
        # Complete row: hard weight
        return [(observed, 1.0)]

    # Query joint posterior over missing variables given observed evidence
    result = infer_engine.query(variables=list(missing_vars), evidence=observed, show_progress=False)

    # result is a DiscreteFactor over missing_vars; enumerate all state combinations
    weighted_assignments = []
    state_names = result.state_names
    all_states = [state_names[var] for var in result.variables]

    for combo in itertools.product(*all_states):
        assignment = dict(zip(result.variables, combo))
        prob = result.get_value(**assignment)
        if prob > 0:
            full_assignment = {**observed, **assignment}
            weighted_assignments.append((full_assignment, prob))

    return weighted_assignments

print("compute_row_weights defined.")

In [ ]:
# Cell 20: Custom EM implementation — M-step
# Aggregates weighted assignments into new CPTs for every node in the model

from pgmpy.factors.discrete import TabularCPD
import numpy as np

def compute_new_cpds(model, all_weighted_assignments, state_names):
    """
    all_weighted_assignments: list of (assignment_dict, weight) tuples,
        pooled across ALL rows (each row contributes 1 tuple if complete,
        or multiple tuples summing to 1.0 in weight if it had missing vars)
    state_names: dict of {var: [state0, state1, ...]} — fixed ordering, must
        be consistent across the whole EM run
    Returns: list of TabularCPD, one per node
    """
    new_cpds = []

    for node in model.nodes():
        parents = list(model.get_parents(node))
        node_states = state_names[node]
        n_node_states = len(node_states)

        if not parents:
            # Root node: just need weighted counts of node's own states
            counts = np.zeros(n_node_states)
            for assignment, weight in all_weighted_assignments:
                val = assignment[node]
                counts[node_states.index(val)] += weight

            total = counts.sum()
            probs = counts / total if total > 0 else np.ones(n_node_states) / n_node_states
            cpd = TabularCPD(node, n_node_states, probs.reshape(-1, 1),
                              state_names={node: node_states})
        else:
            parent_states = [state_names[p] for p in parents]
            parent_combos = list(itertools.product(*parent_states))
            n_parent_combos = len(parent_combos)

            counts = np.zeros((n_node_states, n_parent_combos))
            combo_index = {combo: i for i, combo in enumerate(parent_combos)}

            for assignment, weight in all_weighted_assignments:
                node_val = assignment[node]
                parent_vals = tuple(assignment[p] for p in parents)
                row_i = node_states.index(node_val)
                col_i = combo_index[parent_vals]
                counts[row_i, col_i] += weight

            # Normalize each column (parent combo) to sum to 1
            col_sums = counts.sum(axis=0)
            col_sums_safe = np.where(col_sums > 0, col_sums, 1)
            probs = counts / col_sums_safe
            # Where a parent combo had zero total weight, fall back to uniform
            zero_cols = np.where(col_sums == 0)[0]
            if len(zero_cols) > 0:
                probs[:, zero_cols] = 1.0 / n_node_states

            cpd = TabularCPD(node, n_node_states, probs,
                              evidence=parents,
                              evidence_card=[len(state_names[p]) for p in parents],
                              state_names={**{node: node_states}, **{p: state_names[p] for p in parents}})

        new_cpds.append(cpd)

    return new_cpds

print("compute_new_cpds defined.")

In [ ]:
# Cell 21 (corrected): observed-data log-likelihood using VariableElimination's
# joint marginal query directly, rather than an incorrect chained-conditioning shortcut

def compute_log_likelihood(model, data, missing_vars_per_row, infer_engine):
    total_log_lik = 0.0
    eps = 1e-12

    for idx, row in data.iterrows():
        missing_vars = missing_vars_per_row[idx]
        observed = {var: row[var] for var in row.index if var not in missing_vars and pd.notnull(row[var])}

        if len(observed) == 0:
            continue  # edge case, shouldn't occur

        # Query the joint marginal distribution over the OBSERVED variables,
        # with no evidence — this correctly marginalises out missing vars
        # (both structurally latent and row-specific missing).
        marginal = infer_engine.query(variables=list(observed.keys()), evidence=None, show_progress=False)
        prob = marginal.get_value(**observed)

        total_log_lik += np.log(max(prob, eps))

    return total_log_lik

print("compute_log_likelihood defined (corrected).")

In [ ]:
# Cell 22: Full custom EM loop, validated on the Check 2 toy network
# (A -> C, B -> C, with C having 20% row-wise missingness)
# Fix: use DiscreteMLE() as an initialized instance, matching the DiscreteEM
# API convention we already discovered

from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

# Just re-run the fit line with corrected estimator usage:
def run_custom_em(model_structure, data, max_iter=50, tol=1e-4, verbose=True):
    nodes = list(dict.fromkeys([n for edge in model_structure for n in edge]))
    for col in data.columns:
        if col not in nodes:
            nodes.append(col)

    state_names = {var: sorted(data[var].dropna().unique().tolist()) for var in nodes}

    missing_vars_per_row = {
        idx: [var for var in nodes if pd.isnull(row[var])]
        for idx, row in data.iterrows()
    }

    complete_data = data.dropna()
    print(f"Initialising from {len(complete_data)} complete cases out of {len(data)}")

    model = DiscreteBayesianNetwork(model_structure)
    model.fit(complete_data, estimator=DiscreteMLE())  # <-- fixed

    log_lik_history = []

    for iteration in range(max_iter):
        infer_engine = VariableElimination(model)

        all_weighted_assignments = []
        for idx, row in data.iterrows():
            missing_vars = missing_vars_per_row[idx]
            weighted = compute_row_weights(model, row, missing_vars, infer_engine)
            all_weighted_assignments.extend(weighted)

        new_cpds = compute_new_cpds(model, all_weighted_assignments, state_names)
        model.cpds = []
        model.add_cpds(*new_cpds)

        infer_engine = VariableElimination(model)
        log_lik = compute_log_likelihood(model, data, missing_vars_per_row, infer_engine)
        log_lik_history.append(log_lik)

        if verbose:
            print(f"Iteration {iteration+1}: log-likelihood = {log_lik:.4f}")

        if iteration > 0 and abs(log_lik_history[-1] - log_lik_history[-2]) < tol:
            print(f"Converged after {iteration+1} iterations.")
            break

    return model, log_lik_history

# --- Run on toy network ---
toy_model, toy_ll_history = run_custom_em(
    model_structure=[('A', 'C'), ('B', 'C')],
    data=df_toy_missing,
    max_iter=50,
    tol=1e-4
)

print("\n--- Custom EM Recovered CPTs vs Known True CPTs ---")
print("\nTrue P(A): [0.3, 0.7]")
print("Recovered P(A):", toy_model.get_cpds('A').values.flatten())

print("\nTrue P(B): [0.4, 0.6]")
print("Recovered P(B):", toy_model.get_cpds('B').values.flatten())

print("\nTrue P(C|A,B) row 0 (P(C=0)):", [0.8, 0.6, 0.4, 0.2])
print("Recovered P(C|A,B):")
print(toy_model.get_cpds('C'))

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(toy_ll_history)+1), toy_ll_history, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Observed-data log-likelihood')
plt.title('Custom EM convergence — toy network')
plt.grid(True)
plt.show()

diffs = np.diff(toy_ll_history)
print(f"\nAll iterations monotonically non-decreasing: {(diffs >= -1e-8).all()}")
print(f"Number of iterations to converge: {len(toy_ll_history)}")

In [ ]:
# Cell 22a: Find the correct import path for DiscreteMLE (or equivalent)

import pgmpy.estimators
print("Contents of pgmpy.estimators:")
print([x for x in dir(pgmpy.estimators) if not x.startswith('_')])

# Also check parameter_estimator, since DiscreteEM lived there rather than estimators
import pgmpy.parameter_estimator
print("\nContents of pgmpy.parameter_estimator:")
print([x for x in dir(pgmpy.parameter_estimator) if not x.startswith('_')])

In [ ]:
# Cell 23: Extended toy validation — missingness in a MIDDLE node (C),
# which has both a parent structure (A, B -> C) and a child (C -> D).
# This forces genuine soft-evidence propagation in both the E-step (inferring
# C's posterior from A, B, AND D) and the M-step (weighting both C's own CPT
# and D's CPT using non-trivial posteriors) — much closer to Monthly Income's
# actual position in the real 13-node network.

np.random.seed(42)
n2 = 2000

# True CPTs
true_p_a = [0.3, 0.7]
true_p_b = [0.4, 0.6]
true_p_c_given_ab = {  # P(C=1 | A, B)
    (0,0): 0.2, (0,1): 0.5, (1,0): 0.6, (1,1): 0.9
}
true_p_d_given_c = {  # P(D=1 | C)
    0: 0.1, 1: 0.8
}

a2 = np.random.choice([0,1], size=n2, p=true_p_a)
b2 = np.random.choice([0,1], size=n2, p=true_p_b)
c2 = np.array([np.random.choice([0,1], p=[1-true_p_c_given_ab[(a,b)], true_p_c_given_ab[(a,b)]])
               for a, b in zip(a2, b2)])
d2 = np.array([np.random.choice([0,1], p=[1-true_p_d_given_c[c], true_p_d_given_c[c]])
               for c in c2])

df_toy2 = pd.DataFrame({'A': a2, 'B': b2, 'C': c2, 'D': d2})

# Inject 25% missingness into C specifically (the middle node)
nan_idx2 = np.random.choice(n2, size=int(0.25*n2), replace=False)
df_toy2_missing = df_toy2.copy()
df_toy2_missing.loc[nan_idx2, 'C'] = np.nan

print(f"Toy2 shape: {df_toy2_missing.shape}")
print(f"NaN in C: {df_toy2_missing['C'].isna().sum()}")

# --- Run custom EM ---
toy2_model, toy2_ll_history = run_custom_em(
    model_structure=[('A', 'C'), ('B', 'C'), ('C', 'D')],
    data=df_toy2_missing,
    max_iter=50,
    tol=1e-4
)

# --- Listwise deletion baseline for comparison ---
df_toy2_complete = df_toy2_missing.dropna()
listwise_model2 = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C'), ('C', 'D')])
listwise_model2.fit(df_toy2_complete, estimator=DiscreteMLE())

print("\n=== P(C|A,B) comparison ===")
print("True P(C=1|A,B): (0,0)=0.2, (0,1)=0.5, (1,0)=0.6, (1,1)=0.9")
print("\nCustom EM recovered:")
print(toy2_model.get_cpds('C'))
print("\nListwise deletion recovered:")
print(listwise_model2.get_cpds('C'))

print("\n=== P(D|C) comparison ===")
print("True P(D=1|C): C=0 -> 0.1, C=1 -> 0.8")
print("\nCustom EM recovered:")
print(toy2_model.get_cpds('D'))
print("\nListwise deletion recovered:")
print(listwise_model2.get_cpds('D'))

# Plot convergence
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(toy2_ll_history)+1), toy2_ll_history, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Observed-data log-likelihood')
plt.title('Custom EM convergence — extended toy network (missingness in middle node C)')
plt.grid(True)
plt.show()

diffs2 = np.diff(toy2_ll_history)
print(f"\nAll iterations monotonically non-decreasing: {(diffs2 >= -1e-8).all()}")
print(f"Number of iterations to converge: {len(toy2_ll_history)}")

# Quantify: is EM's C|A,B and D|C CPT closer to truth than listwise?
def cpd_l2_dist(cpd, true_dict, parent_order, child_states=(0,1)):
    dist = 0.0
    for parents_vals, true_p1 in true_dict.items():
        if not isinstance(parents_vals, tuple):
            parents_vals = (parents_vals,)
        kwargs = dict(zip(parent_order, parents_vals))
        recovered_p1 = cpd.get_value(**{cpd.variable: 1}, **kwargs)
        dist += (recovered_p1 - true_p1) ** 2
    return np.sqrt(dist)

em_c_dist = cpd_l2_dist(toy2_model.get_cpds('C'), true_p_c_given_ab, ['A', 'B'])
lw_c_dist = cpd_l2_dist(listwise_model2.get_cpds('C'), true_p_c_given_ab, ['A', 'B'])
em_d_dist = cpd_l2_dist(toy2_model.get_cpds('D'), true_p_d_given_c, ['C'])
lw_d_dist = cpd_l2_dist(listwise_model2.get_cpds('D'), true_p_d_given_c, ['C'])

print(f"\nL2 distance from truth — P(C|A,B): Custom EM = {em_c_dist:.4f}, Listwise = {lw_c_dist:.4f}")
print(f"L2 distance from truth — P(D|C): Custom EM = {em_d_dist:.4f}, Listwise = {lw_d_dist:.4f}")
print("(Lower is better; EM should be <= listwise on both if working correctly)")

In [ ]:
# Cell 24: MNAR toy validation — missingness in C depends on C's own (unobserved) value
# P(C missing | C=0) = 0.6, P(C missing | C=1) = 0.1
# Run across 10 seeds, compare average L2 distance from truth: custom EM vs listwise

true_p_a = [0.3, 0.7]
true_p_b = [0.4, 0.6]
true_p_c_given_ab = {(0,0): 0.2, (0,1): 0.5, (1,0): 0.6, (1,1): 0.9}
true_p_d_given_c = {0: 0.1, 1: 0.8}

def generate_mnar_toy(seed, n=2000):
    rng = np.random.RandomState(seed)
    a2 = rng.choice([0,1], size=n, p=true_p_a)
    b2 = rng.choice([0,1], size=n, p=true_p_b)
    c2 = np.array([rng.choice([0,1], p=[1-true_p_c_given_ab[(a,b)], true_p_c_given_ab[(a,b)]])
                   for a, b in zip(a2, b2)])
    d2 = np.array([rng.choice([0,1], p=[1-true_p_d_given_c[c], true_p_d_given_c[c]])
                   for c in c2])

    df = pd.DataFrame({'A': a2, 'B': b2, 'C': c2, 'D': d2})

    # MNAR missingness: P(missing | C=0) = 0.6, P(missing | C=1) = 0.1
    miss_prob = np.where(c2 == 0, 0.6, 0.1)
    is_missing = rng.random(n) < miss_prob
    df_missing = df.copy()
    df_missing.loc[is_missing, 'C'] = np.nan

    return df, df_missing

def cpd_l2_dist(cpd, true_dict, parent_order):
    dist = 0.0
    for parents_vals, true_p1 in true_dict.items():
        if not isinstance(parents_vals, tuple):
            parents_vals = (parents_vals,)
        kwargs = dict(zip(parent_order, parents_vals))
        recovered_p1 = cpd.get_value(**{cpd.variable: 1}, **kwargs)
        dist += (recovered_p1 - true_p1) ** 2
    return np.sqrt(dist)

em_c_dists, lw_c_dists = [], []
em_d_dists, lw_d_dists = [], []

for seed in range(10):
    df_true, df_missing = generate_mnar_toy(seed)
    n_missing = df_missing['C'].isna().sum()

    # Custom EM
    em_model, _ = run_custom_em(
        model_structure=[('A', 'C'), ('B', 'C'), ('C', 'D')],
        data=df_missing, max_iter=50, tol=1e-4, verbose=False
    )

    # Listwise deletion
    df_complete = df_missing.dropna()
    lw_model = DiscreteBayesianNetwork([('A', 'C'), ('B', 'C'), ('C', 'D')])
    lw_model.fit(df_complete, estimator=DiscreteMLE())

    em_c_d = cpd_l2_dist(em_model.get_cpds('C'), true_p_c_given_ab, ['A', 'B'])
    lw_c_d = cpd_l2_dist(lw_model.get_cpds('C'), true_p_c_given_ab, ['A', 'B'])
    em_d_d = cpd_l2_dist(em_model.get_cpds('D'), true_p_d_given_c, ['C'])
    lw_d_d = cpd_l2_dist(lw_model.get_cpds('D'), true_p_d_given_c, ['C'])

    em_c_dists.append(em_c_d)
    lw_c_dists.append(lw_c_d)
    em_d_dists.append(em_d_d)
    lw_d_dists.append(lw_d_d)

    print(f"Seed {seed}: n_missing={n_missing} | EM C-dist={em_c_d:.4f}, LW C-dist={lw_c_d:.4f} | "
          f"EM D-dist={em_d_d:.4f}, LW D-dist={lw_d_d:.4f}")

print("\n=== Summary across 10 seeds ===")
print(f"P(C|A,B) L2 distance — EM: mean={np.mean(em_c_dists):.4f}, std={np.std(em_c_dists):.4f}")
print(f"P(C|A,B) L2 distance — Listwise: mean={np.mean(lw_c_dists):.4f}, std={np.std(lw_c_dists):.4f}")
print(f"P(D|C) L2 distance — EM: mean={np.mean(em_d_dists):.4f}, std={np.std(em_d_dists):.4f}")
print(f"P(D|C) L2 distance — Listwise: mean={np.mean(lw_d_dists):.4f}, std={np.std(lw_d_dists):.4f}")

n_seeds_em_better_c = sum(e < l for e, l in zip(em_c_dists, lw_c_dists))
n_seeds_em_better_d = sum(e < l for e, l in zip(em_d_dists, lw_d_dists))
print(f"\nEM beat listwise on P(C|A,B) in {n_seeds_em_better_c}/10 seeds")
print(f"EM beat listwise on P(D|C) in {n_seeds_em_better_d}/10 seeds")

In [ ]:
# Cell 25: Step 1 — Construct M_Income and M_Dependents indicator columns
# M = 1 means observed, M = 0 means missing. Always fully observed themselves.

df_train['M_Income'] = df_train['Monthly Income (bin)'].notna().astype(int)
df_train['M_Dependents'] = df_train['Number of Dependents (bin)'].notna().astype(int)

df_test['M_Income'] = df_test['Monthly Income (bin)'].notna().astype(int)
df_test['M_Dependents'] = df_test['Number of Dependents (bin)'].notna().astype(int)

print("=== Train ===")
print("M_Income value counts:")
print(df_train['M_Income'].value_counts())
print("M_Dependents value counts:")
print(df_train['M_Dependents'].value_counts())

print("\n=== Test ===")
print("M_Income value counts:")
print(df_test['M_Income'].value_counts())
print("M_Dependents value counts:")
print(df_test['M_Dependents'].value_counts())

# Sanity checks against known missingness counts from earlier cells
print("\n=== Sanity checks ===")
print(f"Train M_Income=0 count (expect ~23748): {(df_train['M_Income']==0).sum()}")
print(f"Test M_Income=0 count (expect ~5983): {(df_test['M_Income']==0).sum()}")
print(f"Train M_Dependents=0 count (expect 3147): {(df_train['M_Dependents']==0).sum()}")
print(f"Test M_Dependents=0 count (expect 777): {(df_test['M_Dependents']==0).sum()}")

# Confirm nested pattern still holds: M_Dependents=0 should always imply M_Income=0
violation_count = ((df_train['M_Dependents']==0) & (df_train['M_Income']==1)).sum()
print(f"\nRows where Dependents missing but Income observed (should be 0): {violation_count}")

In [ ]:
# Cell 26: Step 2a — Extract complete-case subset for structure learning,
# and inspect HillClimbSearch's actual signature before writing the restart loop

# Complete-case subset: all 10 original features + Default observed
# (equivalently: M_Income == 1 AND M_Dependents == 1)
original_11_nodes = ['Default', 'Credit Utilisation Rate (bin)', 'Age (bin)',
                      'Debt Ratio (bin)', 'Monthly Income (bin)',
                      '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
                      '60-89 Days Past Due (bin)', 'Number of Dependents (bin)',
                      'Real Estate Loans (bin)', 'Open Credit Lines (bin)']

df_train_complete = df_train[(df_train['M_Income'] == 1) & (df_train['M_Dependents'] == 1)][original_11_nodes].copy()

print(f"Complete-case training subset: {len(df_train_complete)} rows (expected ~95,830)")
print(f"Columns: {df_train_complete.columns.tolist()}")
print(f"\nAny remaining NaN (should be none):")
print(df_train_complete.isnull().sum())

# Inspect HillClimbSearch API
from pgmpy.estimators import HillClimbSearch, BIC
import inspect
print("\n=== HillClimbSearch.__init__ signature ===")
print(inspect.signature(HillClimbSearch.__init__))
print("\n=== HillClimbSearch.estimate signature ===")
print(inspect.signature(HillClimbSearch.estimate))

In [ ]:
# # Cell 27: Step 2b — Multi-restart hill-climbing + BIC structure learning
# # No built-in restart support in pgmpy's HillClimbSearch, so implemented manually:
# # generate random starting DAGs, run hill-climb from each, keep the best-BIC result.

# from pgmpy.estimators import HillClimbSearch, BIC
# from pgmpy.base import DAG
# import random

# N_RESTARTS = 15
# SEED = 42

# nodes = original_11_nodes

# def random_start_dag(nodes, rng, edge_prob=0.15):
#     """Generate a random DAG by taking a random topological order and
#     randomly including forward-only edges (guarantees acyclicity)."""
#     order = nodes.copy()
#     rng.shuffle(order)
#     dag = DAG()
#     dag.add_nodes_from(order)
#     for i in range(len(order)):
#         for j in range(i+1, len(order)):
#             if rng.random() < edge_prob:
#                 dag.add_edge(order[i], order[j])
#     return dag

# bic_scorer = BIC(df_train_complete)

# results = []
# for restart in range(N_RESTARTS):
#     rng = random.Random(SEED + restart)  # distinct but reproducible seed per restart
#     start = random_start_dag(nodes, rng)

#     hc = HillClimbSearch(df_train_complete)
#     learned_dag = hc.estimate(scoring_method=bic_scorer, start_dag=start, show_progress=False)

#     score = bic_scorer.score(learned_dag)
#     results.append((score, learned_dag, restart))
#     print(f"Restart {restart}: BIC score = {score:.2f}, edges = {len(learned_dag.edges())}")

# # Select best
# results.sort(key=lambda x: x[0], reverse=True)  # higher BIC = better (pgmpy convention: less negative is better)
# best_score, best_dag, best_restart = results[0]

# print(f"\nBest structure: restart {best_restart}, BIC score = {best_score:.2f}")
# print(f"\nLearned edges ({len(best_dag.edges())} total):")
# for edge in best_dag.edges():
#     print(f"  {edge[0]} -> {edge[1]}")

In [ ]:
# # Cell 28 (rebuilt): Step 3 — corrected DAG construction V-structure design.
# # M_Income and M_Dependents are now root nodes, each a direct parent of its own variable,
# # creating a V-structure (collider) wherever that variable already had a learned parent.

# from pgmpy.models import DiscreteBayesianNetwork
# import networkx as nx

# learned_edges = list(best_dag.edges())  # should now be the 22-edge version

# fixed_edges = [
#     ('M_Income', 'Monthly Income (bin)'),
#     ('M_Dependents', 'Number of Dependents (bin)'),
# ]

# full_13_node_edges = learned_edges + fixed_edges

# print(f"Learned edges: {len(learned_edges)}")
# print(f"Fixed edges: {len(fixed_edges)}")
# print(f"Total edges in 13-node DAG: {len(full_13_node_edges)}")

# bn_model = DiscreteBayesianNetwork(full_13_node_edges)

# n_nodes = len(bn_model.nodes())
# print(f"\n[Check 1] Total nodes: {n_nodes} (expect 13) -> {'PASS' if n_nodes == 13 else 'FAIL'}")

# is_dag = nx.is_directed_acyclic_graph(bn_model)
# print(f"[Check 2] Is acyclic: {is_dag} -> {'PASS' if is_dag else 'FAIL'}")

# income_parents = bn_model.get_parents('Monthly Income (bin)')
# print(f"[Check 3] Monthly Income (bin) parents: {income_parents}")
# print(f"          Has M_Income as parent: {'M_Income' in income_parents} -> "
#       f"{'PASS' if 'M_Income' in income_parents and len(income_parents) >= 2 else 'FAIL'}")

# dependents_parents = bn_model.get_parents('Number of Dependents (bin)')
# print(f"[Check 4] Number of Dependents (bin) parents: {dependents_parents}")
# print(f"          Has M_Dependents as parent: {'M_Dependents' in dependents_parents} -> "
#       f"{'PASS' if 'M_Dependents' in dependents_parents else 'FAIL'}")

# m_income_parents = bn_model.get_parents('M_Income')
# m_dependents_parents = bn_model.get_parents('M_Dependents')
# print(f"[Check 5] M_Income parents: {m_income_parents} -> {'PASS' if len(m_income_parents) == 0 else 'FAIL'}")
# print(f"          M_Dependents parents: {m_dependents_parents} -> {'PASS' if len(m_dependents_parents) == 0 else 'FAIL'}")

# print(f"\nFull edge list:")
# for edge in bn_model.edges():
#     print(f"  {edge[0]} -> {edge[1]}")

In [ ]:
# # Cell 29: Step 4 — Combine learned 11-node structure with the two fixed
# # indicator edges to produce the full 13-node DAG
# import os

# fig, ax = plt.subplots(figsize=(24, 15))

# pos = nx.nx_agraph.graphviz_layout(bn_model, prog='dot', args='-Gnodesep=1.0 -Granksep=1.0')

# # Nudge M_Income right, away from the crowded Debt Ratio / Open Credit Lines cluster
# pos['M_Income'] = (700.0, pos['M_Income'][1])

# indicator_nodes = {'M_Income', 'M_Dependents'}
# target_node = {'Default'}
# substantive_nodes = [n for n in bn_model.nodes() if n not in indicator_nodes and n not in target_node]

# node_size = 5000

# nx.draw_networkx_nodes(bn_model, pos, nodelist=substantive_nodes,
#                         node_color='lightblue', node_size=node_size,
#                         edgecolors='black', ax=ax)
# nx.draw_networkx_nodes(bn_model, pos, nodelist=list(target_node),
#                         node_color='gold', node_size=node_size,
#                         edgecolors='black', linewidths=2, ax=ax)
# nx.draw_networkx_nodes(bn_model, pos, nodelist=list(indicator_nodes),
#                         node_color='lightgray', node_size=node_size,
#                         edgecolors='black', node_shape='s', ax=ax)

# nx.draw_networkx_edges(bn_model, pos, edgelist=learned_edges,
#                         edge_color='gray', arrows=True, arrowstyle='-|>',
#                         arrowsize=18, node_size=node_size,
#                         connectionstyle='arc3,rad=0.08', ax=ax)
# nx.draw_networkx_edges(bn_model, pos, edgelist=fixed_edges,
#                         edge_color='dimgray', arrows=True, arrowstyle='-|>',
#                         arrowsize=18, node_size=node_size,
#                         width=2.2, style='dashed', ax=ax)

# labels = {n: n.replace(' (bin)', '') for n in bn_model.nodes()}
# nx.draw_networkx_labels(bn_model, pos, labels=labels, font_size=8, font_weight='bold', ax=ax)

# ax.set_title('13-Node MNAR-Aware Bayesian Network Structure\n'
#               '(Yellow = target (Default), Blue = substantive features, '
#               'Gray squares = missingness indicators (root nodes),\n',
#               fontsize=13)
# ax.axis('off')

# plt.tight_layout()
# plt.savefig('bn_13node_dag_final.png', dpi=300, bbox_inches='tight')
# plt.show()

# print("Saved to:", os.path.abspath('bn_13node_dag_final.png'))

In [ ]:
# # Cell 30: Setup for Step 5 (custom EM on the full 13-node network),
# # plus a corrected, efficient log-likelihood function.
# #
# # BUG HISTORY:
# # v1 (compute_log_likelihood): queried the full 13-variable joint via
# #    VariableElimination even for COMPLETE rows -- the most expensive
# #    possible query, hit on ~80% of rows. Never finished in reasonable time.
# # v2 (compute_log_likelihood_fast): fixed complete rows via chain-rule direct
# #    CPT lookup, but still used VariableElimination for INCOMPLETE rows,
# #    where "observed" = 12 of 13 variables -- almost as expensive. Still hung.
# # v3 (compute_log_likelihood_v2, this version): NO VariableElimination calls
# #    anywhere. Complete rows use direct chain-rule lookup. Incomplete rows
# #    brute-force sum over the missing variable's small state space (max 5
# #    states here), each term also computed via cheap chain-rule lookup.

# import time
# import itertools

# thirteen_node_cols = original_11_nodes + ['M_Income', 'M_Dependents']

# df_train_13 = df_train[thirteen_node_cols].copy()
# df_test_13 = df_test[thirteen_node_cols].copy()

# # Fixed state names across the whole run, from train UNION test, so no state
# # is ever "unknown" to the model during inference on either set
# state_names_13 = {
#     var: sorted(pd.concat([df_train_13[var], df_test_13[var]]).dropna().unique().tolist())
#     for var in thirteen_node_cols
# }
# print("M_Income states:", state_names_13['M_Income'])
# print("M_Dependents states:", state_names_13['M_Dependents'])

# missing_vars_per_row_train = {
#     idx: [var for var in thirteen_node_cols if pd.isnull(row[var])]
#     for idx, row in df_train_13.iterrows()
# }
# missing_vars_per_row_test = {
#     idx: [var for var in thirteen_node_cols if pd.isnull(row[var])]
#     for idx, row in df_test_13.iterrows()
# }

# complete_train_13 = df_train_13.dropna()
# print(f"\nInitialising from {len(complete_train_13)} complete cases out of {len(df_train_13)}")

# full_13_node_edges_final = list(bn_model.edges())
# em_model_13 = DiscreteBayesianNetwork(full_13_node_edges_final)
# em_model_13.fit(complete_train_13, estimator=DiscreteMLE(state_names=state_names_13))

# print(f"\nM_Income CPD states: {em_model_13.get_cpds('M_Income').state_names['M_Income']}")


# def compute_joint_prob_chain_rule(model, assignment, eps=1e-12):
#     """P(full assignment) via chain rule, direct CPT lookups only -- no inference engine."""
#     log_prob = 0.0
#     for node in model.nodes():
#         cpd = model.get_cpds(node)
#         parents = list(model.get_parents(node))
#         node_val = assignment[node]
#         if not parents:
#             p = cpd.get_value(**{node: node_val})
#         else:
#             parent_vals = {p_var: assignment[p_var] for p_var in parents}
#             p = cpd.get_value(**{node: node_val}, **parent_vals)
#         log_prob += np.log(max(p, eps))
#     return log_prob


# def compute_log_likelihood_v2(model, data, missing_vars_per_row, state_names):
#     """Observed-data log-likelihood, no VariableElimination calls anywhere.
#     Complete rows: direct chain-rule lookup.
#     Incomplete rows: brute-force sum over missing variable(s)' state space."""
#     total_log_lik = 0.0
#     eps = 1e-12

#     for idx, row in data.iterrows():
#         missing_vars = missing_vars_per_row[idx]
#         observed = {var: row[var] for var in row.index
#                     if var not in missing_vars and pd.notnull(row[var])}

#         if not missing_vars:
#             total_log_lik += compute_joint_prob_chain_rule(model, observed)
#         else:
#             missing_state_options = [state_names[v] for v in missing_vars]
#             probs = []
#             for combo in itertools.product(*missing_state_options):
#                 full_assignment = {**observed, **dict(zip(missing_vars, combo))}
#                 log_p = compute_joint_prob_chain_rule(model, full_assignment)
#                 probs.append(np.exp(log_p))
#             total_prob = sum(probs)
#             total_log_lik += np.log(max(total_prob, eps))

#     return total_log_lik

# print("\ncompute_log_likelihood_v2 defined (no VariableElimination calls at all).")

In [ ]:
# # Cell 31: CLEAN full 100-iteration EM run. Requires a kernel restart before
# # this cell to guarantee no leftover state from Cell 31/32/33/35's prior runs.
# # Only depends on: df_train_13, df_test_13, state_names_13,
# # missing_vars_per_row_train/test, complete_train_13, full_13_node_edges_final,
# # and the helper functions -- all re-established fresh via Cell 30.

# import copy

# MAX_ITER = 100

# # Fresh initialisation -- explicitly re-created here, not reused from any prior cell
# em_model_clean = DiscreteBayesianNetwork(full_13_node_edges_final)
# em_model_clean.fit(complete_train_13, estimator=DiscreteMLE(state_names=state_names_13))

# infer_engine = VariableElimination(em_model_clean)

# train_ll_clean = []
# test_ll_clean = []
# iter_times_clean = []
# cpds_by_iteration_clean = {}

# overall_start = time.time()

# for iteration in range(1, MAX_ITER + 1):
#     iter_start = time.time()

#     # E-step
#     all_weighted_assignments = []
#     for idx, row in df_train_13.iterrows():
#         missing_vars = missing_vars_per_row_train[idx]
#         weighted = compute_row_weights(em_model_clean, row, missing_vars, infer_engine)
#         all_weighted_assignments.extend(weighted)

#     # M-step
#     new_cpds = compute_new_cpds(em_model_clean, all_weighted_assignments, state_names_13)
#     em_model_clean.cpds = []
#     em_model_clean.add_cpds(*new_cpds)

#     infer_engine = VariableElimination(em_model_clean)

#     # Log-likelihood
#     train_ll = compute_log_likelihood_v2(em_model_clean, df_train_13, missing_vars_per_row_train, state_names_13)
#     test_ll = compute_log_likelihood_v2(em_model_clean, df_test_13, missing_vars_per_row_test, state_names_13)

#     train_ll_clean.append(train_ll)
#     test_ll_clean.append(test_ll)
#     cpds_by_iteration_clean[iteration] = copy.deepcopy(em_model_clean.cpds)

#     iter_time = time.time() - iter_start
#     iter_times_clean.append(iter_time)

#     # Flag timing anomalies immediately (>2x the running median)
#     if iteration > 5:
#         median_time = np.median(iter_times_clean[:-1])
#         if iter_time > 2 * median_time:
#             print(f"  *** TIMING ANOMALY at iteration {iteration}: {iter_time:.1f}s "
#                   f"(median so far: {median_time:.1f}s) ***")

#     print(f"Iteration {iteration}/{MAX_ITER}: train_ll={train_ll:.6f}, test_ll={test_ll:.6f}, time={iter_time:.1f}s")

#     # CRITICAL CHECK: verify iteration 1 matches Cell 35's known-clean iteration 1
#     if iteration == 1:
#         expected_train_iter1 = -1318922.2958
#         expected_test_iter1 = -329661.5031
#         train_match = abs(train_ll - expected_train_iter1) < 0.01
#         test_match = abs(test_ll - expected_test_iter1) < 0.01
#         print(f"\n  >>> INITIALISATION VERIFICATION <<<")
#         print(f"  Expected (Cell 35 iter 1): train={expected_train_iter1}, test={expected_test_iter1}")
#         print(f"  Got:                       train={train_ll:.4f}, test={test_ll:.4f}")
#         print(f"  Train match: {train_match} | Test match: {test_match}")
#         if not (train_match and test_match):
#             print(f"  *** WARNING: INITIALISATION DOES NOT MATCH -- STOP AND INVESTIGATE BEFORE CONTINUING ***\n")
#         else:
#             print(f"  Initialisation verified clean. Continuing full run.\n")

# total_time = time.time() - overall_start

# print(f"\n=== FINAL SUMMARY ===")
# print(f"Total iterations: {len(train_ll_clean)}")
# print(f"Total wall-clock time: {total_time:.1f}s ({total_time/60:.1f} min)")
# print(f"Average iteration time: {np.mean(iter_times_clean):.1f}s")

# peak_idx_clean = int(np.argmax(test_ll_clean))
# peak_iteration_clean = peak_idx_clean + 1
# print(f"\nTrue peak (this clean run): iteration {peak_iteration_clean}")
# print(f"Test log-likelihood at peak: {test_ll_clean[peak_idx_clean]:.6f}")
# print(f"Train log-likelihood at peak: {train_ll_clean[peak_idx_clean]:.6f}")

# train_diffs_clean = np.diff(train_ll_clean)
# print(f"\nTrain log-likelihood monotonically non-decreasing: {(train_diffs_clean >= -1e-6).all()}")

In [ ]:
# # Cell 32: Task 2 — extract iteration-79 CPTs and save as the final model

# PEAK_ITERATION_FINAL = 79
# peak_cpds = cpds_by_iteration_clean[PEAK_ITERATION_FINAL]  # already the correct set (dict keyed 1-100, not 0-indexed)

# # Build the final model with the confirmed 13-node structure and iteration-79 CPTs
# bn_final = DiscreteBayesianNetwork(full_13_node_edges_final)
# bn_final.add_cpds(*peak_cpds)

# # Sanity check before saving
# is_valid = bn_final.check_model()
# print(f"Final model valid (check_model): {is_valid}")
# print(f"Number of nodes: {len(bn_final.nodes())}")
# print(f"Number of CPDs: {len(bn_final.get_cpds())}")

# # Save via pickle (portable, includes state_names and structure implicitly via the CPDs)
# import pickle

# save_path = 'bn_final_model_iteration79.pkl'
# with open(save_path, 'wb') as f:
#     pickle.dump({
#         'model_edges': full_13_node_edges_final,
#         'cpds': bn_final.get_cpds(),
#         'state_names': state_names_13,
#         'peak_iteration': PEAK_ITERATION_FINAL,
#         'train_ll_at_peak': train_ll_clean[PEAK_ITERATION_FINAL - 1],
#         'test_ll_at_peak': test_ll_clean[PEAK_ITERATION_FINAL - 1],
#     }, f)

# print(f"\nSaved to: {os.path.abspath(save_path)}")

# # Verify the save worked by loading it back and checking equivalence
# with open(save_path, 'rb') as f:
#     loaded = pickle.load(f)

# reloaded_model = DiscreteBayesianNetwork(loaded['model_edges'])
# reloaded_model.add_cpds(*loaded['cpds'])
# reload_valid = reloaded_model.check_model()

# print(f"Reload verification -- model valid: {reload_valid}")
# print(f"Reload verification -- peak iteration recorded: {loaded['peak_iteration']}")
# print(f"Reload verification -- train_ll matches: {abs(loaded['train_ll_at_peak'] - train_ll_clean[PEAK_ITERATION_FINAL-1]) < 1e-6}")
# print(f"Reload verification -- test_ll matches: {abs(loaded['test_ll_at_peak'] - test_ll_clean[PEAK_ITERATION_FINAL-1]) < 1e-6}")

In [ ]:
# # Cell 33a: Task 3 -- final convergence plot 

# fig, ax = plt.subplots(figsize=(12, 7))

# iterations = range(1, len(train_ll_clean) + 1)

# # Twin axes since train/test log-likelihood are on very different scales
# ax2 = ax.twinx()

# line1, = ax.plot(iterations, train_ll_clean, color='blue', label='Train log-likelihood', linewidth=1.8)
# line2, = ax2.plot(iterations, test_ll_clean, color='orange', label='Test log-likelihood', linewidth=1.8)

# ax.set_xlabel('Iteration')
# ax.set_ylabel('Train log-likelihood', color='blue')
# ax2.set_ylabel('Test log-likelihood', color='orange')
# ax.tick_params(axis='y', labelcolor='blue')
# ax2.tick_params(axis='y', labelcolor='orange')

# # Vertical dashed line at peak
# ax.axvline(x=PEAK_ITERATION_FINAL, color='black', linestyle='--', linewidth=1.2)
# ax.text(PEAK_ITERATION_FINAL + 1.5, ax.get_ylim()[0] + 0.08 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
#         f'Optimal stopping point (iter. 79)', fontsize=9, color='black')

# ax.set_title('EM Convergence — Train and Test Log-Likelihood', fontsize=13)

# # Combined legend from both axes
# lines = [line1, line2]
# labels = [l.get_label() for l in lines]
# ax.legend(lines, labels, loc='lower right', frameon=False)

# # Clean academic style: no gridlines, minimal spines
# ax.grid(False)
# ax2.grid(False)
# ax.spines['top'].set_visible(False)
# ax2.spines['top'].set_visible(False)

# plt.tight_layout()
# plt.savefig('em_convergence_final.png', dpi=300, bbox_inches='tight')
# plt.show()

# print("Saved to:", os.path.abspath('em_convergence_final.png'))

In [ ]:
# # Cell 33b: Add a zoomed inset (iterations 60-100) to actually show the
# # train-test divergence pattern, which is invisible at full scale

# fig, (ax_main, ax_zoom) = plt.subplots(1, 2, figsize=(16, 7))

# iterations = range(1, len(train_ll_clean) + 1)

# # --- Main plot (full range, as before) ---
# ax2 = ax_main.twinx()
# ax_main.plot(iterations, train_ll_clean, color='blue', label='Train log-likelihood', linewidth=1.8)
# ax2.plot(iterations, test_ll_clean, color='orange', label='Test log-likelihood', linewidth=1.8)
# ax_main.set_xlabel('Iteration')
# ax_main.set_ylabel('Train log-likelihood', color='blue')
# ax2.set_ylabel('Test log-likelihood', color='orange')
# ax_main.tick_params(axis='y', labelcolor='blue')
# ax2.tick_params(axis='y', labelcolor='orange')
# ax_main.axvline(x=PEAK_ITERATION_FINAL, color='black', linestyle='--', linewidth=1.2)
# ax_main.set_title('Full range (iterations 1–100)', fontsize=11)
# ax_main.spines['top'].set_visible(False)
# ax2.spines['top'].set_visible(False)

# # --- Zoomed plot: TEST log-likelihood only, iterations 60-100, where the
# # peak-then-decline is actually visible ---
# zoom_start = 60
# zoom_iters = list(range(zoom_start, 101))
# zoom_test_vals = test_ll_clean[zoom_start-1:100]

# ax_zoom.plot(zoom_iters, zoom_test_vals, color='orange', linewidth=2)
# ax_zoom.axvline(x=PEAK_ITERATION_FINAL, color='black', linestyle='--', linewidth=1.2)
# ax_zoom.scatter([PEAK_ITERATION_FINAL], [test_ll_clean[PEAK_ITERATION_FINAL-1]],
#                  color='red', zorder=5, s=50, label=f'Peak (iter. {PEAK_ITERATION_FINAL})')
# ax_zoom.set_xlabel('Iteration')
# ax_zoom.set_ylabel('Test log-likelihood')
# ax_zoom.set_title(f'Zoomed: iterations {zoom_start}–100 (test log-likelihood only)', fontsize=11)
# ax_zoom.legend(loc='upper right', frameon=False)
# ax_zoom.spines['top'].set_visible(False)
# ax_zoom.spines['right'].set_visible(False)

# fig.suptitle('EM Convergence — Train and Test Log-Likelihood', fontsize=14)
# plt.tight_layout()
# plt.savefig('em_convergence_final_with_zoom.png', dpi=300, bbox_inches='tight')
# plt.show()

# print("Saved to:", os.path.abspath('em_convergence_final_with_zoom.png'))

In [ ]:
# Cell 34: Matrix completion for LR/RF -- IterativeImputer fit on train only,
# transformed onto both train and test. Monthly Income is log1p-transformed
# before imputation (safe for zero values, keeps predictions bounded to a
# small negative range rather than unbounded implausible extrapolation) and
# expm1-transformed back afterward.

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

impute_cols = ['Credit Utilisation Rate', 'Age', 'Debt Ratio', 'Monthly Income',
                'Open Credit Lines', '30-59 Days Past Due', '90+ Days Late',
                '60-89 Days Past Due', 'Number of Dependents', 'Real Estate Loans']

df_train_continuous = df_clean.loc[train_idx, impute_cols].copy()
df_test_continuous = df_clean.loc[test_idx, impute_cols].copy()

# log1p-transform Monthly Income
df_train_continuous['Monthly Income'] = np.log1p(df_train_continuous['Monthly Income'])
df_test_continuous['Monthly Income'] = np.log1p(df_test_continuous['Monthly Income'])

print("Monthly Income (log1p-transformed) summary, train, non-missing rows:")
print(df_train_continuous['Monthly Income'].describe())

print("\nMissing values before imputation:")
print("Train:\n", df_train_continuous.isnull().sum())
print("\nTest:\n", df_test_continuous.isnull().sum())

# Fit on TRAIN only, transform both
imputer = IterativeImputer(random_state=42)
imputer.fit(df_train_continuous)

train_imputed_array = imputer.transform(df_train_continuous)
test_imputed_array = imputer.transform(df_test_continuous)

df_train_imputed = pd.DataFrame(train_imputed_array, columns=impute_cols, index=train_idx)
df_test_imputed = pd.DataFrame(test_imputed_array, columns=impute_cols, index=test_idx)

print("\nMissing values after imputation (should all be 0):")
print("Train:\n", df_train_imputed.isnull().sum())
print("Test:\n", df_test_imputed.isnull().sum())

# expm1 back to original scale
df_train_imputed['Monthly Income'] = np.expm1(df_train_imputed['Monthly Income'])
df_test_imputed['Monthly Income'] = np.expm1(df_test_imputed['Monthly Income'])

# Track which rows were originally missing (needed for plausibility checks below)
train_income_missing_mask = df_clean.loc[train_idx, 'Monthly Income'].isnull()
test_income_missing_mask = df_clean.loc[test_idx, 'Monthly Income'].isnull()

print("\nMonthly Income (back to original scale, pre-clip) summary, train, full column:")
print(df_train_imputed['Monthly Income'].describe())

imputed_income_train_precheck = df_train_imputed.loc[train_income_missing_mask.values, 'Monthly Income']
imputed_income_test_precheck = df_test_imputed.loc[test_income_missing_mask.values, 'Monthly Income']
print(f"\nImputed values (train) below 0, pre-clip: {(imputed_income_train_precheck < 0).sum()}")
print(f"Imputed values (test) below 0, pre-clip: {(imputed_income_test_precheck < 0).sum()}")

# Apply final clip -- all residual negatives are bounded in (-1, 0) by
# construction of the log1p/expm1 round-trip, so this is a safe, minor
# correction rather than papering over unbounded extrapolation.
df_train_imputed['Monthly Income'] = df_train_imputed['Monthly Income'].clip(lower=0)
df_test_imputed['Monthly Income'] = df_test_imputed['Monthly Income'].clip(lower=0)

print(f"\nAfter clip(lower=0):")
print(f"Train Monthly Income min: {df_train_imputed['Monthly Income'].min():.4f}")
print(f"Test Monthly Income min: {df_test_imputed['Monthly Income'].min():.4f}")

In [ ]:
# Cell 35: Discretise the (now non-negative) imputed Monthly Income using the
# existing train-derived bin edges, plus Number of Dependents natural-category
# binning. Full verification checks as originally specified.

# --- Monthly Income: apply existing equal-frequency edges ---
income_edges = bin_edges['Monthly Income'].copy()
income_edges_adj = income_edges.copy()
income_edges_adj[0] = -np.inf
income_edges_adj[-1] = np.inf

df_train_imputed['Monthly Income (bin)'] = pd.cut(df_train_imputed['Monthly Income'],
                                                     bins=income_edges_adj, include_lowest=True)
df_test_imputed['Monthly Income (bin)'] = pd.cut(df_test_imputed['Monthly Income'],
                                                    bins=income_edges_adj, include_lowest=True)

# --- Number of Dependents: apply existing natural-category function ---
df_train_imputed['Number of Dependents (bin)'] = bin_natural_category(
    df_train_imputed['Number of Dependents'], max_category=3)
df_test_imputed['Number of Dependents (bin)'] = bin_natural_category(
    df_test_imputed['Number of Dependents'], max_category=3)

# --- Verification 1: no NaN remain anywhere ---
print("=== NaN check (post-discretisation) ===")
print(f"Train Monthly Income (bin) NaN: {df_train_imputed['Monthly Income (bin)'].isnull().sum()}")
print(f"Test Monthly Income (bin) NaN: {df_test_imputed['Monthly Income (bin)'].isnull().sum()}")
print(f"Train Number of Dependents (bin) NaN: {df_train_imputed['Number of Dependents (bin)'].isnull().sum()}")
print(f"Test Number of Dependents (bin) NaN: {df_test_imputed['Number of Dependents (bin)'].isnull().sum()}")

# --- Verification 2: imputed Monthly Income plausibility (post-clip) ---
imputed_income_train = df_train_imputed.loc[train_income_missing_mask.values, 'Monthly Income']
imputed_income_test = df_test_imputed.loc[test_income_missing_mask.values, 'Monthly Income']

train_income_max_observed = df_clean.loc[train_idx, 'Monthly Income'].max()

print(f"\n=== Monthly Income imputation plausibility (post-clip) ===")
print(f"Training set observed max Monthly Income: {train_income_max_observed:.2f}")
print(f"Imputed values (train) below 0: {(imputed_income_train < 0).sum()} (expect 0)")
print(f"Imputed values (train) above training max: {(imputed_income_train > train_income_max_observed).sum()}")
print(f"Imputed values (test) below 0: {(imputed_income_test < 0).sum()} (expect 0)")
print(f"Imputed values (test) above training max: {(imputed_income_test > train_income_max_observed).sum()}")
print(f"\nImputed Monthly Income (train) distribution:\n{imputed_income_train.describe()}")

# --- Verification 3: imputed Number of Dependents distribution ---
train_dep_missing_mask = df_clean.loc[train_idx, 'Number of Dependents'].isnull()
imputed_dep_train = df_train_imputed.loc[train_dep_missing_mask.values, 'Number of Dependents (bin)']

print(f"\n=== Number of Dependents imputation distribution (train, imputed rows only) ===")
print(imputed_dep_train.value_counts(normalize=True).sort_index())

# --- Full distributions ---
print(f"\n=== Full Monthly Income (bin) distribution — train ===")
print(df_train_imputed['Monthly Income (bin)'].value_counts().sort_index())
print(f"\n=== Full Number of Dependents (bin) distribution — train ===")
print(df_train_imputed['Number of Dependents (bin)'].value_counts().sort_index())

In [ ]:
# Cell 36: Assemble the full 10-feature categorical matrix for LR/RF --
# 8 features that never had missing data (bins from df_train/df_test) +
# Monthly Income (bin) / Number of Dependents (bin) from the imputed
# pipeline (df_train_imputed/df_test_imputed) -- then one-hot encode.

from sklearn.preprocessing import OneHotEncoder

lr_rf_bin_cols = [
    'Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)',
    'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
    '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)',
]

df_train_lr_rf = df_train[lr_rf_bin_cols].copy()
df_test_lr_rf = df_test[lr_rf_bin_cols].copy()

df_train_lr_rf['Monthly Income (bin)'] = df_train_imputed['Monthly Income (bin)'].values
df_train_lr_rf['Number of Dependents (bin)'] = df_train_imputed['Number of Dependents (bin)'].values
df_test_lr_rf['Monthly Income (bin)'] = df_test_imputed['Monthly Income (bin)'].values
df_test_lr_rf['Number of Dependents (bin)'] = df_test_imputed['Number of Dependents (bin)'].values

print(f"df_train_lr_rf shape (pre-encoding): {df_train_lr_rf.shape}")
print(f"df_test_lr_rf shape (pre-encoding): {df_test_lr_rf.shape}")
print(f"\nNaN check pre-encoding:")
print(df_train_lr_rf.isnull().sum())
print(df_test_lr_rf.isnull().sum())

# Convert to string for consistent, hashable OneHotEncoder categories
df_train_lr_rf_str = df_train_lr_rf.astype(str)
df_test_lr_rf_str = df_test_lr_rf.astype(str)

# Fit on TRAIN only, transform both. drop='first' avoids the dummy variable
# trap for LR; same encoder used for RF for pipeline consistency.
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='error')
encoder.fit(df_train_lr_rf_str)

X_train_encoded = encoder.transform(df_train_lr_rf_str)
X_test_encoded = encoder.transform(df_test_lr_rf_str)

encoded_feature_names = encoder.get_feature_names_out(df_train_lr_rf_str.columns)

X_train_df = pd.DataFrame(X_train_encoded, columns=encoded_feature_names, index=train_idx)
X_test_df = pd.DataFrame(X_test_encoded, columns=encoded_feature_names, index=test_idx)

print(f"\n=== ENCODED SHAPES ===")
print(f"X_train_df shape: {X_train_df.shape}")
print(f"X_test_df shape: {X_test_df.shape}")
print(f"\nNumber of encoded features: {X_train_df.shape[1]}")
print(f"\nAll encoded column names:")
print(encoded_feature_names.tolist())

In [ ]:
# Cell 37: Logistic Regression -- GridSearchCV, 5-fold stratified CV
# Per planning instance's exact specification: solver='saga' (supports L1
# and L2 at scale, unlike liblinear which doesn't scale as well on large
# one-hot encoded matrices)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

y_train = df_clean.loc[train_idx, 'Default'].values
y_test = df_clean.loc[test_idx, 'Default'].values

print(f"y_train shape: {y_train.shape}, positive rate: {y_train.mean():.4f}")
print(f"y_test shape: {y_test.shape}, positive rate: {y_test.mean():.4f}")

lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
}

lr_base = LogisticRegression(
    class_weight='balanced',
    solver='saga',
    max_iter=1000,
    random_state=42,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_grid_search = GridSearchCV(
    lr_base, lr_param_grid, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1
)

t0 = time.time()
lr_grid_search.fit(X_train_df, y_train)
lr_tune_time = time.time() - t0

print(f"\n=== LOGISTIC REGRESSION RESULTS ===")
print(f"Training time: {lr_tune_time:.1f}s ({lr_tune_time/60:.1f} min)")
print(f"Best params: {lr_grid_search.best_params_}")
print(f"Best CV AUC: {lr_grid_search.best_score_:.4f}")

lr_final = lr_grid_search.best_estimator_

In [ ]:
# # Cell 38: Random Forest -- GridSearchCV, 5-fold stratified CV
# # Full grid per specification: 3 x 4 x 3 = 36 combinations x 5 folds = 180 fits

# from sklearn.ensemble import RandomForestClassifier

# rf_param_grid = {
#     'n_estimators': [100, 200, 500],
#     'max_depth': [5, 10, 20, None],
#     'min_samples_leaf': [1, 5, 10],
# }

# n_combinations = 3 * 4 * 3
# print(f"Grid: {n_combinations} combinations x 5 folds = {n_combinations*5} total fits")

# rf_base = RandomForestClassifier(
#     class_weight='balanced',
#     random_state=42,
#     n_jobs=-1,
# )

# rf_grid_search = GridSearchCV(
#     rf_base, rf_param_grid, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1
# )

# t0 = time.time()
# rf_grid_search.fit(X_train_df, y_train)
# rf_tune_time = time.time() - t0

# print(f"\n=== RANDOM FOREST RESULTS ===")
# print(f"Training time: {rf_tune_time:.1f}s ({rf_tune_time/60:.1f} min)")
# print(f"Best params: {rf_grid_search.best_params_}")
# print(f"Best CV AUC: {rf_grid_search.best_score_:.4f}")

# rf_final = rf_grid_search.best_estimator_

In [ ]:
# # Cell 39: Diagnostics before trusting RF's CV AUC of 0.9955
# # 1. True held-out test-set performance for both LR and RF
# # 2. RF feature importances -- check for a single feature dominating
# #    suspiciously (a leakage red flag)
# # 3. A deliberately shallower RF, to see if score is robust or an
# #    overfitting artifact of very deep/unrestricted trees

# from sklearn.metrics import roc_auc_score

# # --- 1. Test-set AUC ---
# lr_test_probs = lr_final.predict_proba(X_test_df)[:, 1]
# rf_test_probs = rf_final.predict_proba(X_test_df)[:, 1]

# lr_test_auc = roc_auc_score(y_test, lr_test_probs)
# rf_test_auc = roc_auc_score(y_test, rf_test_probs)

# print("=== TEST-SET AUC (held-out, never touched by CV) ===")
# print(f"LR test AUC: {lr_test_auc:.4f}  (CV was {lr_grid_search.best_score_:.4f})")
# print(f"RF test AUC: {rf_test_auc:.4f}  (CV was {rf_grid_search.best_score_:.4f})")

# # --- 2. Feature importances ---
# importances = pd.Series(rf_final.feature_importances_, index=X_train_df.columns).sort_values(ascending=False)
# print(f"\n=== RF Feature Importances (top 15) ===")
# print(importances.head(15))
# print(f"\nTop feature's share of total importance: {importances.iloc[0]:.4f} "
#       f"({importances.iloc[0]/importances.sum()*100:.1f}% of total)")

# # --- 3. Shallower RF for comparison ---
# rf_shallow = RandomForestClassifier(
#     n_estimators=200, max_depth=5, min_samples_leaf=10,
#     class_weight='balanced', random_state=42, n_jobs=-1
# )
# rf_shallow.fit(X_train_df, y_train)
# rf_shallow_test_probs = rf_shallow.predict_proba(X_test_df)[:, 1]
# rf_shallow_test_auc = roc_auc_score(y_test, rf_shallow_test_probs)

# print(f"\n=== Shallow RF comparison (max_depth=5, min_samples_leaf=10) ===")
# print(f"Shallow RF test AUC: {rf_shallow_test_auc:.4f}")
# print(f"Deep (tuned) RF test AUC: {rf_test_auc:.4f}")
# print(f"Gap: {rf_test_auc - rf_shallow_test_auc:.4f}")

In [ ]:
# Cell 40: RF grid search with a restricted search space designed to avoid
# the overfitting-prone extremes (unbounded depth, min_samples_leaf=1)
# that caused the earlier 0.9955 CV / 0.8590 test gap.

from sklearn.ensemble import RandomForestClassifier

rf_param_grid_restricted = {
    'n_estimators': [100, 200, 500],
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [5, 10, 20],
}

n_combinations_restricted = 3 * 3 * 3
print(f"Restricted grid: {n_combinations_restricted} combinations x 5 folds = "
      f"{n_combinations_restricted*5} total fits")

rf_base_restricted = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

rf_grid_search_restricted = GridSearchCV(
    rf_base_restricted, rf_param_grid_restricted, scoring='roc_auc',
    cv=cv, n_jobs=-1, verbose=1
)

t0 = time.time()
rf_grid_search_restricted.fit(X_train_df, y_train)
rf_tune_time_restricted = time.time() - t0

print(f"\n=== RESTRICTED RF GRID SEARCH RESULTS ===")
print(f"Training time: {rf_tune_time_restricted:.1f}s ({rf_tune_time_restricted/60:.1f} min)")
print(f"Best params: {rf_grid_search_restricted.best_params_}")
print(f"Best CV AUC: {rf_grid_search_restricted.best_score_:.4f}")

rf_final_restricted = rf_grid_search_restricted.best_estimator_

In [ ]:
# Cell 41: Evaluate the CV-selected model on test set, check CV/test gap

from sklearn.metrics import roc_auc_score

rf_restricted_test_probs = rf_final_restricted.predict_proba(X_test_df)[:, 1]
rf_restricted_test_auc = roc_auc_score(y_test, rf_restricted_test_probs)

cv_test_gap = rf_grid_search_restricted.best_score_ - rf_restricted_test_auc

print(f"=== CV vs TEST AUC CHECK ===")
print(f"Best CV AUC: {rf_grid_search_restricted.best_score_:.4f}")
print(f"Test AUC: {rf_restricted_test_auc:.4f}")
print(f"Gap: {cv_test_gap:.4f}")

if abs(cv_test_gap) <= 0.02:
    print(f"\nGap within acceptable range (<=0.02) -- CV and test AUC track closely.")
    print(f"This is a trustworthy result. Recommend locking in as the final RF model.")
else:
    print(f"\nGap exceeds 0.02 -- still a meaningful CV/test discrepancy.")
    print(f"Do NOT lock this in yet -- report back before proceeding further.")

In [ ]:
# Cell 42: Save RF feature importances and LR coefficients now, while both
# models are locked in and in memory. These feed into RQ2's interpretability
# comparison against the BN's sensitivity analysis -- not used in RQ1.

import os

# --- RF feature importances (from the locked-in restricted-search best model) ---
rf_importances = pd.Series(
    rf_final_restricted.feature_importances_,
    index=X_train_df.columns
).sort_values(ascending=False)

print("=== RF Feature Importances (top 15) ===")
print(rf_importances.head(15))

# --- LR coefficients (from the fitted LR model) ---
lr_coefficients = pd.Series(
    lr_final.coef_[0],
    index=X_train_df.columns
).sort_values(key=abs, ascending=False)  # sort by magnitude, preserving sign

print("\n=== LR Coefficients (top 15 by magnitude) ===")
print(lr_coefficients.head(15))

# --- Save both to disk for reliable reuse in RQ2, independent of kernel state ---
import pickle

with open('rq2_feature_importance_artifacts.pkl', 'wb') as f:
    pickle.dump({
        'rf_feature_importances': rf_importances,
        'lr_coefficients': lr_coefficients,
        'rf_best_params': rf_grid_search_restricted.best_params_,
        'lr_best_params': lr_grid_search.best_params_,
        'feature_names': list(X_train_df.columns),
    }, f)

print(f"\nSaved to: {os.path.abspath('rq2_feature_importance_artifacts.pkl')}")

# Verify the save
with open('rq2_feature_importance_artifacts.pkl', 'rb') as f:
    loaded_check = pickle.load(f)
print(f"Reload check -- RF importances match: "
      f"{(loaded_check['rf_feature_importances'] == rf_importances).all()}")
print(f"Reload check -- LR coefficients match: "
      f"{(loaded_check['lr_coefficients'] == lr_coefficients).all()}")

In [ ]:
# Cell 43: Verify df_test is the original pre-imputation data, load the
# saved final BN model, and run a 10-row timing test before committing
# to the full 30,000-row inference run.

# --- Verify df_test integrity ---
print("=== df_test integrity check ===")
print(f"df_test shape: {df_test.shape}")
print(f"Monthly Income (bin) NaN count: {df_test['Monthly Income (bin)'].isnull().sum()} (expect 5983)")
print(f"Number of Dependents (bin) NaN count: {df_test['Number of Dependents (bin)'].isnull().sum()} (expect 777)")

# --- Load the saved final BN model ---
import pickle
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

with open('bn_final_model_iteration79.pkl', 'rb') as f:
    loaded = pickle.load(f)

bn_final = DiscreteBayesianNetwork(loaded['model_edges'])
bn_final.add_cpds(*loaded['cpds'])
print(f"\nBN model loaded. Valid: {bn_final.check_model()}")
print(f"Peak iteration: {loaded['peak_iteration']}")

infer_engine = VariableElimination(bn_final)

# --- Build evidence dict for a single row ---
def build_evidence(row):
    feature_bin_cols = [
        'Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)',
        'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
        '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)',
        'Monthly Income (bin)', 'Number of Dependents (bin)',
    ]
    evidence = {}
    for col in feature_bin_cols:
        val = row[col]
        if pd.notnull(val):
            evidence[col] = val
    evidence['M_Income'] = int(row['M_Income'])
    evidence['M_Dependents'] = int(row['M_Dependents'])
    return evidence

# --- Ensure M_Income / M_Dependents exist on df_test (constructed earlier, but verify) ---
df_test['M_Income'] = df_test['Monthly Income (bin)'].notna().astype(int)
df_test['M_Dependents'] = df_test['Number of Dependents (bin)'].notna().astype(int)

# --- Timing test: first 10 rows ---
import time

test_rows_sample = df_test.iloc[:10]
sample_probs = []
sample_times = []

for idx, row in test_rows_sample.iterrows():
    t0 = time.time()
    evidence = build_evidence(row)
    result = infer_engine.query(variables=['Default'], evidence=evidence, show_progress=False)
    prob_default_1 = result.get_value(Default=1)
    t1 = time.time()

    sample_probs.append(prob_default_1)
    sample_times.append(t1 - t0)

    print(f"Row {idx}: evidence keys={len(evidence)}, P(Default=1)={prob_default_1:.4f}, time={t1-t0:.3f}s")

print(f"\n=== TIMING TEST SUMMARY (10 rows) ===")
print(f"Mean per-row time: {np.mean(sample_times):.3f}s")
print(f"Max per-row time: {np.max(sample_times):.3f}s")
print(f"Estimated total time for 30,000 rows: {np.mean(sample_times)*30000/60:.1f} min")

if np.mean(sample_times) > 5:
    print("\n*** Mean per-row time exceeds 5s -- STOP, do not proceed to full run, report back first ***")
else:
    print("\nPer-row time under 5s threshold -- safe to proceed to full run.")

In [ ]:
# Cell 44: Full BN inference on all 30,000 test rows, using real observed
# evidence (pre-imputation), letting junction tree marginalise over any
# missing feature bins automatically.

t0 = time.time()

bn_test_probs = []
errors_or_out_of_range = []

for i, (idx, row) in enumerate(df_test.iterrows()):
    evidence = build_evidence(row)
    try:
        result = infer_engine.query(variables=['Default'], evidence=evidence, show_progress=False)
        prob = result.get_value(Default=1)
        if not (0 <= prob <= 1):
            errors_or_out_of_range.append((idx, 'out_of_range', prob))
            prob = np.nan
    except Exception as e:
        errors_or_out_of_range.append((idx, 'error', str(e)))
        prob = np.nan
    bn_test_probs.append(prob)

    if (i + 1) % 5000 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/30000 rows done, {elapsed:.1f}s elapsed")

total_inference_time = time.time() - t0

bn_test_probs = np.array(bn_test_probs)

print(f"\n=== FULL INFERENCE RUN COMPLETE ===")
print(f"Total time: {total_inference_time:.1f}s ({total_inference_time/60:.1f} min)")
print(f"Rows with errors or out-of-range probabilities: {len(errors_or_out_of_range)}")
if errors_or_out_of_range:
    print("First 10 flagged rows:")
    for item in errors_or_out_of_range[:10]:
        print(f"  {item}")

print(f"\nNaN count in bn_test_probs: {np.isnan(bn_test_probs).sum()}")
print(f"Min probability: {np.nanmin(bn_test_probs):.4f}")
print(f"Max probability: {np.nanmax(bn_test_probs):.4f}")
print(f"Mean probability: {np.nanmean(bn_test_probs):.4f}")

In [ ]:
# Investigate why rows 125238 and 7312 produced impossible (zero-probability) evidence

for problem_idx in [125238, 7312]:
    row = df_test.loc[problem_idx]
    evidence = build_evidence(row)
    print(f"\n=== Row {problem_idx} ===")
    print(f"Evidence: {evidence}")
    print(f"Actual Default label: {row['Default']}")

    # Check if this EXACT evidence combination (excluding Default) appears
    # anywhere in the training data at all
    mask = pd.Series(True, index=df_train.index)
    for k, v in evidence.items():
        if k in ['M_Income', 'M_Dependents']:
            mask &= (df_train[k] == v)
        else:
            mask &= (df_train[k] == v)
    n_matching_train_rows = mask.sum()
    print(f"Matching rows in training data (same evidence combination): {n_matching_train_rows}")

In [ ]:
# Cell 45: Compute BN test-set AUC, Gini, KS -- excluding the 2 rows with
# genuinely zero-probability evidence (documented, expected edge case,
# not a bug -- confirmed via zero matching training rows for either).

valid_mask = ~np.isnan(bn_test_probs)
n_excluded = (~valid_mask).sum()

print(f"Rows excluded from metrics (zero-probability evidence): {n_excluded}")
print(f"Rows used for metrics: {valid_mask.sum()} / {len(bn_test_probs)}")

y_test_valid = y_test[valid_mask]
bn_probs_valid = bn_test_probs[valid_mask]

bn_test_auc = roc_auc_score(y_test_valid, bn_probs_valid)
bn_gini = 2 * bn_test_auc - 1

# KS statistic: max separation between cumulative distributions of
# defaulters vs non-defaulters' predicted probabilities
from scipy.stats import ks_2samp
ks_stat, ks_pvalue = ks_2samp(
    bn_probs_valid[y_test_valid == 1],
    bn_probs_valid[y_test_valid == 0]
)

print(f"\n=== BN TEST-SET RESULTS ===")
print(f"BN Test AUC: {bn_test_auc:.4f}")
print(f"BN Gini: {bn_gini:.4f}")
print(f"BN KS statistic: {ks_stat:.4f}")
print(f"Total inference time (30,000 rows): {total_inference_time:.1f}s")

# --- Save full probability vector (with NaN preserved for the 2 excluded
# rows) to disk for RQ1 charts, RQ2, and EVOI extension ---
bn_probs_save = pd.Series(bn_test_probs, index=df_test.index, name='bn_pred_prob')

with open('bn_test_predictions.pkl', 'wb') as f:
    pickle.dump({
        'bn_test_probs': bn_probs_save,
        'bn_test_auc': bn_test_auc,
        'bn_gini': bn_gini,
        'bn_ks_statistic': ks_stat,
        'excluded_row_indices': [125238, 7312],
        'total_inference_time_seconds': total_inference_time,
    }, f)

print(f"\nSaved to: {os.path.abspath('bn_test_predictions.pkl')}")

In [ ]:
# Rebuild LR test-set predictions and AUC -- lr_final is already the fitted
# model from Cell 37, this just re-generates predictions on test, cheap and instant

lr_test_probs = lr_final.predict_proba(X_test_df)[:, 1]
lr_test_auc = roc_auc_score(y_test, lr_test_probs)

print(f"LR test AUC: {lr_test_auc:.4f}")

In [ ]:
# Cell 46: Complete metrics for LR and RF, matching what's already computed for BN

from scipy.stats import ks_2samp

def compute_ks(probs, y_true):
    return ks_2samp(probs[y_true == 1], probs[y_true == 0]).statistic

def cost_sensitive_loss(probs, y_true, threshold=0.5, fn_cost=5, fp_cost=1):
    preds = (probs >= threshold).astype(int)
    fn = ((preds == 0) & (y_true == 1)).sum()
    fp = ((preds == 1) & (y_true == 0)).sum()
    total_cost = fn * fn_cost + fp * fp_cost
    return total_cost, fn, fp

# --- LR ---
lr_gini = 2 * lr_test_auc - 1
lr_ks = compute_ks(lr_test_probs, y_test)
lr_cost, lr_fn, lr_fp = cost_sensitive_loss(lr_test_probs, y_test)

# --- RF (restricted-search, locked-in model) ---
rf_gini = 2 * rf_restricted_test_auc - 1
rf_ks = compute_ks(rf_restricted_test_probs, y_test)
rf_cost, rf_fn, rf_fp = cost_sensitive_loss(rf_restricted_test_probs, y_test)

# --- BN (already have AUC, Gini, KS from Cell 45 -- recompute cost loss consistently) ---
bn_cost, bn_fn, bn_fp = cost_sensitive_loss(bn_probs_valid, y_test_valid)

print("=== PROVISIONAL cost-sensitive loss (threshold=0.5, 5:1 FN:FP ratio) ===")
print("FLAGGED AS PROVISIONAL -- threshold 0.5 is a placeholder until the")
print("cost-sensitive threshold curve determines the actual optimal value.\n")

print(f"LR:  cost={lr_cost}, FN={lr_fn}, FP={lr_fp}")
print(f"RF:  cost={rf_cost}, FN={rf_fn}, FP={rf_fp}")
print(f"BN:  cost={bn_cost}, FN={bn_fn}, FP={bn_fp}  (n=29,998, 2 rows excluded)")

print("\n=== COMPLETE METRICS TABLE ===")
metrics_table = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Bayesian Network'],
    'AUC': [lr_test_auc, rf_restricted_test_auc, bn_test_auc],
    'Gini': [lr_gini, rf_gini, bn_gini],
    'KS': [lr_ks, rf_ks, ks_stat],
    'Cost (provisional, thr=0.5)': [lr_cost, rf_cost, bn_cost],
})
print(metrics_table.to_string(index=False))

In [ ]:
# Cell 47: Bootstrap-based pairwise AUC comparison (DeLong alternative,
# chosen because BN's prediction vector excludes 2 rows the others don't,
# complicating a clean paired analytic DeLong test)

N_BOOTSTRAP = 2000
rng = np.random.RandomState(42)

def bootstrap_auc_diff(probs_a, probs_b, y_true, n_boot=N_BOOTSTRAP, seed=42):
    """Bootstrap the difference in AUC (A - B) over shared rows only."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    diffs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        y_b = y_true[idx]
        if len(np.unique(y_b)) < 2:
            continue  # skip degenerate resamples with only one class
        auc_a = roc_auc_score(y_b, probs_a[idx])
        auc_b = roc_auc_score(y_b, probs_b[idx])
        diffs.append(auc_a - auc_b)
    diffs = np.array(diffs)
    ci_lower, ci_upper = np.percentile(diffs, [2.5, 97.5])
    # two-sided bootstrap p-value: proportion of resamples where sign flips relative to observed
    observed_diff = np.mean(diffs)
    p_value = 2 * min((diffs > 0).mean(), (diffs < 0).mean())
    return observed_diff, ci_lower, ci_upper, p_value

# --- LR vs RF (all 30,000 rows, both fully defined) ---
diff_lr_rf, ci_lo, ci_hi, p_lr_rf = bootstrap_auc_diff(lr_test_probs, rf_restricted_test_probs, y_test)
print(f"LR vs RF:  AUC diff (LR-RF) = {diff_lr_rf:.4f}, 95% CI [{ci_lo:.4f}, {ci_hi:.4f}], p = {p_lr_rf:.4f}")

# --- LR vs BN and RF vs BN: restrict to the 29,998 rows BN could score ---
valid_idx_bool = valid_mask  # from Cell 45, True for the 29,998 usable rows
lr_probs_valid = lr_test_probs[valid_idx_bool]
rf_probs_valid_for_bn = rf_restricted_test_probs[valid_idx_bool]

diff_lr_bn, ci_lo2, ci_hi2, p_lr_bn = bootstrap_auc_diff(lr_probs_valid, bn_probs_valid, y_test_valid)
print(f"LR vs BN:  AUC diff (LR-BN) = {diff_lr_bn:.4f}, 95% CI [{ci_lo2:.4f}, {ci_hi2:.4f}], p = {p_lr_bn:.4f}")

diff_rf_bn, ci_lo3, ci_hi3, p_rf_bn = bootstrap_auc_diff(rf_probs_valid_for_bn, bn_probs_valid, y_test_valid)
print(f"RF vs BN:  AUC diff (RF-BN) = {diff_rf_bn:.4f}, 95% CI [{ci_lo3:.4f}, {ci_hi3:.4f}], p = {p_rf_bn:.4f}")

print(f"\n(n_bootstrap = {N_BOOTSTRAP}, seed = 42)")

In [ ]:
# Cell 48: Diagnostic check (not a dissertation figure) -- compare the SHAPE
# of predicted probability distributions across all three models, to
# understand the FN/FP asymmetry seen at threshold 0.5.

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

axes[0].hist(lr_test_probs, bins=50, color='tab:blue', alpha=0.8)
axes[0].set_title('LR predicted P(Default=1)')
axes[0].set_xlabel('Predicted probability')
axes[0].set_ylabel('Count')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1)

axes[1].hist(rf_restricted_test_probs, bins=50, color='tab:orange', alpha=0.8)
axes[1].set_title('RF predicted P(Default=1)')
axes[1].set_xlabel('Predicted probability')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1)

axes[2].hist(bn_probs_valid, bins=50, color='tab:green', alpha=0.8)
axes[2].set_title('BN predicted P(Default=1)')
axes[2].set_xlabel('Predicted probability')
axes[2].axvline(0.5, color='black', linestyle='--', linewidth=1)

plt.suptitle('Diagnostic: predicted probability distributions (test set)', fontsize=13)
plt.tight_layout()
plt.savefig('diagnostic_prob_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Distribution summary statistics ===")
for name, probs in [('LR', lr_test_probs), ('RF', rf_restricted_test_probs), ('BN', bn_probs_valid)]:
    print(f"\n{name}:")
    print(f"  Min: {probs.min():.4f}, Max: {probs.max():.4f}")
    print(f"  Mean: {probs.mean():.4f}, Median: {np.median(probs):.4f}")
    print(f"  % of predictions >= 0.5: {(probs >= 0.5).mean()*100:.2f}%")
    print(f"  Unique values: {len(np.unique(probs))}")

# --- Confirm cost ratio consistency ---
print(f"\n=== Cost ratio confirmation ===")
print(f"All three models' Cost(thr=0.5) figures use the same 5:1 FN:FP cost ratio "
      f"(fn_cost=5, fp_cost=1), applied identically via the same cost_sensitive_loss() function.")

In [ ]:
# Cell 49: Read-only diagnostic -- extract Default's CPT, compute marginal
# P(Default=1) via inference, compare against empirical rates. No changes
# to any saved artifacts, no EM re-run.

# --- Step 1: Confirm Default's parents, print raw CPT ---
default_parents = bn_final.get_parents('Default')
print(f"Default's parents (from learned DAG): {default_parents}")
print(f"Default is a root node: {len(default_parents) == 0}\n")

default_cpd = bn_final.get_cpds('Default')
print("=== Raw CPT for Default ===")
print(default_cpd)

# --- Marginal P(Default=1) via inference, no evidence ---
infer_engine_check = VariableElimination(bn_final)
default_marginal = infer_engine_check.query(variables=['Default'], evidence=None, show_progress=False)
print(f"\n=== Marginal P(Default) via VariableElimination, no evidence ===")
print(default_marginal)
p_default_1_bn = default_marginal.get_value(Default=1)
print(f"\nBN marginal P(Default=1): {p_default_1_bn:.4f} ({p_default_1_bn*100:.2f}%)")

# --- Step 2: Empirical rates ---
full_dataset_rate = df_clean['Default'].mean()
complete_case_rate = df_clean.dropna().loc[df_clean.dropna().index.intersection(train_idx.union(test_idx))]['Default'].mean() \
    if False else df_train_complete['Default'].mean()  # use the actual complete-case subset used for structure learning
df_train_actual_rate = df_train['Default'].mean()

print(f"\n=== Empirical rates ===")
print(f"Full dataset (150,000 rows... 149,999 post-cleaning): {full_dataset_rate:.4f} ({full_dataset_rate*100:.2f}%)")
print(f"Complete-case subset used for Phase 1 structure learning (df_train_complete, "
      f"n={len(df_train_complete)}): {complete_case_rate:.4f} ({complete_case_rate*100:.2f}%)")
print(f"Actual EM training set (df_train, n={len(df_train)}): {df_train_actual_rate:.4f} ({df_train_actual_rate*100:.2f}%)")

# --- Step 3: Comparison ---
print(f"\n=== Comparison ===")
print(f"BN marginal P(Default=1): {p_default_1_bn:.4f}")
print(f"vs full dataset:          {full_dataset_rate:.4f}  (diff: {p_default_1_bn - full_dataset_rate:+.4f})")
print(f"vs complete-case subset:  {complete_case_rate:.4f}  (diff: {p_default_1_bn - complete_case_rate:+.4f})")
print(f"vs actual training set:   {df_train_actual_rate:.4f}  (diff: {p_default_1_bn - df_train_actual_rate:+.4f})")

In [ ]:
# Cell 50: Cost-sensitive threshold optimization -- reusable function,
# per-model curves, and a cost-ratio sensitivity check

def find_optimal_threshold(y_true, y_scores, fn_cost=5, fp_cost=1, thresholds=None):
    """Sweep thresholds, return (optimal_threshold, min_cost, full_cost_array, thresholds)."""
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)
    costs = np.array([
        cost_sensitive_loss(y_scores, y_true, threshold=t, fn_cost=fn_cost, fp_cost=fp_cost)[0]
        for t in thresholds
    ])
    best_idx = np.argmin(costs)
    return thresholds[best_idx], costs[best_idx], costs, thresholds

# --- Primary 5:1 ratio: compute curves for all three models ---
thresholds_sweep = np.arange(0.01, 1.00, 0.01)

tau_lr, cost_lr, costs_lr_arr, _ = find_optimal_threshold(y_test, lr_test_probs, fn_cost=5, fp_cost=1, thresholds=thresholds_sweep)
tau_rf, cost_rf, costs_rf_arr, _ = find_optimal_threshold(y_test, rf_restricted_test_probs, fn_cost=5, fp_cost=1, thresholds=thresholds_sweep)
tau_bn, cost_bn, costs_bn_arr, _ = find_optimal_threshold(y_test_valid, bn_probs_valid, fn_cost=5, fp_cost=1, thresholds=thresholds_sweep)

print("=== Primary results (5:1 FN:FP cost ratio) ===")
print(f"LR:  tau = {tau_lr:.2f}, min cost = {cost_lr}")
print(f"RF:  tau = {tau_rf:.2f}, min cost = {cost_rf}")
print(f"BN:  tau = {tau_bn:.2f}, min cost = {cost_bn}")

# --- Plot all three threshold-cost curves ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds_sweep, costs_lr_arr, color='tab:blue', label=f'LR (τ={tau_lr:.2f})')
ax.plot(thresholds_sweep, costs_rf_arr, color='tab:orange', label=f'RF (τ={tau_rf:.2f})')
ax.plot(thresholds_sweep, costs_bn_arr, color='tab:green', label=f'BN (τ={tau_bn:.2f})')

ax.axvline(tau_lr, color='tab:blue', linestyle='--', alpha=0.5)
ax.axvline(tau_rf, color='tab:orange', linestyle='--', alpha=0.5)
ax.axvline(tau_bn, color='tab:green', linestyle='--', alpha=0.5)

ax.set_xlabel('Decision threshold')
ax.set_ylabel('Total cost (FN cost = 5x FP cost)')
ax.set_title('Cost-sensitive threshold curves (5:1 FN:FP ratio)\n'
              'per-model optimal threshold marked')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('rq1_chart1_threshold_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Sensitivity check: fn_cost in {5, 10, 15, 20}, fp_cost=1 ---
sensitivity_rows = []
for fn_c in [5, 10, 15, 20]:
    tau_lr_s, cost_lr_s, _, _ = find_optimal_threshold(y_test, lr_test_probs, fn_cost=fn_c, fp_cost=1, thresholds=thresholds_sweep)
    tau_rf_s, cost_rf_s, _, _ = find_optimal_threshold(y_test, rf_restricted_test_probs, fn_cost=fn_c, fp_cost=1, thresholds=thresholds_sweep)
    tau_bn_s, cost_bn_s, _, _ = find_optimal_threshold(y_test_valid, bn_probs_valid, fn_cost=fn_c, fp_cost=1, thresholds=thresholds_sweep)
    sensitivity_rows.append({
        'fn:fp ratio': f'{fn_c}:1',
        'LR tau': round(tau_lr_s, 2), 'LR cost': cost_lr_s,
        'RF tau': round(tau_rf_s, 2), 'RF cost': cost_rf_s,
        'BN tau': round(tau_bn_s, 2), 'BN cost': cost_bn_s,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print("\n=== Cost-ratio sensitivity table ===")
print(sensitivity_df.to_string(index=False))

In [ ]:
# Cell 51: 3-panel confusion matrices, each at its own optimal threshold

from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_data = [
    ('LR', y_test, lr_test_probs, tau_lr, 'Blues'),
    ('RF', y_test, rf_restricted_test_probs, tau_rf, 'Oranges'),
    ('BN', y_test_valid, bn_probs_valid, tau_bn, 'Greens'),
]

conf_matrix_results = {}

for ax, (name, y_true, probs, tau, cmap) in zip(axes, model_data):
    preds = (probs >= tau).astype(int)
    cm = confusion_matrix(y_true, preds)
    conf_matrix_results[name] = cm

    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax, cbar=False,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    ax.set_title(f'{name} (τ={tau:.2f})')

    tn, fp, fn, tp = cm.ravel()
    print(f"{name} (τ={tau:.2f}): TN={tn}, FP={fp}, FN={fn}, TP={tp}")

fig.suptitle('Confusion matrices at per-model cost-minimizing threshold (5:1 FN:FP ratio)', fontsize=13)
plt.tight_layout()
plt.savefig('rq1_chart2_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 52: ROC curves, all three models, threshold-independent

from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 8))

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_restricted_test_probs)
fpr_bn, tpr_bn, _ = roc_curve(y_test_valid, bn_probs_valid)

ax.plot(fpr_lr, tpr_lr, color='tab:blue', label=f'LR (AUC={lr_test_auc:.4f})')
ax.plot(fpr_rf, tpr_rf, color='tab:orange', label=f'RF (AUC={rf_restricted_test_auc:.4f})')
ax.plot(fpr_bn, tpr_bn, color='tab:green', label=f'BN (AUC={bn_test_auc:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Chance')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('rq1_chart3_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 53: Precision-Recall curves, all three models

from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 8))

prec_lr, rec_lr, _ = precision_recall_curve(y_test, lr_test_probs)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, rf_restricted_test_probs)
prec_bn, rec_bn, _ = precision_recall_curve(y_test_valid, bn_probs_valid)

ap_lr = average_precision_score(y_test, lr_test_probs)
ap_rf = average_precision_score(y_test, rf_restricted_test_probs)
ap_bn = average_precision_score(y_test_valid, bn_probs_valid)

ax.plot(rec_lr, prec_lr, color='tab:blue', label=f'LR (AP={ap_lr:.4f})')
ax.plot(rec_rf, prec_rf, color='tab:orange', label=f'RF (AP={ap_rf:.4f})')
ax.plot(rec_bn, prec_bn, color='tab:green', label=f'BN (AP={ap_bn:.4f})')

baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color='gray', linestyle=':', label=f'Baseline (prevalence={baseline_rate:.4f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('rq1_chart4_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 54: KS plot -- cumulative distributions of defaulters vs non-defaulters,
# per model, max-separation point marked

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

model_data_ks = [
    ('LR', y_test, lr_test_probs, lr_ks),
    ('RF', y_test, rf_restricted_test_probs, rf_ks),
    ('BN', y_test_valid, bn_probs_valid, ks_stat),
]

for ax, (name, y_true, probs, ks_val) in zip(axes, model_data_ks):
    probs_pos = np.sort(probs[y_true == 1])
    probs_neg = np.sort(probs[y_true == 0])

    x_grid = np.linspace(0, 1, 500)
    cdf_pos = np.searchsorted(probs_pos, x_grid, side='right') / len(probs_pos)
    cdf_neg = np.searchsorted(probs_neg, x_grid, side='right') / len(probs_neg)

    diffs = np.abs(cdf_pos - cdf_neg)
    max_idx = np.argmax(diffs)

    ax.plot(x_grid, cdf_pos, color='tab:red', label='Defaulters (y=1)')
    ax.plot(x_grid, cdf_neg, color='tab:gray', label='Non-defaulters (y=0)')
    ax.vlines(x_grid[max_idx], cdf_neg[max_idx], cdf_pos[max_idx], color='black', linestyle='--')
    ax.set_title(f'{name} (KS={ks_val:.4f})')
    ax.set_xlabel('Predicted probability')
    ax.set_ylabel('Cumulative proportion')
    ax.legend(loc='lower right', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('KS plots — cumulative distributions, defaulters vs non-defaulters', fontsize=13)
plt.tight_layout()
plt.savefig('rq1_chart5_ks_plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 55: Row-level agreement check, verifying RQ1's confusion-matrix-level
# finding actually holds borrower-by-borrower, not just in aggregate totals.

# --- Step 1: per-borrower decisions, restricted to the shared 29,998-row set ---

# Confirm row alignment explicitly before comparing anything.
# lr_test_probs and rf_restricted_test_probs are full 30,000-row arrays (positional,
# aligned to df_test's iteration order). bn_probs_valid is already restricted to the
# 29,998 valid rows via valid_mask. Need to apply the SAME mask to LR/RF to guarantee
# identical row alignment across all three.

lr_probs_shared = lr_test_probs[valid_mask]
rf_probs_shared = rf_restricted_test_probs[valid_mask]
bn_probs_shared = bn_probs_valid  # already the 29,998-row valid subset
y_shared = y_test_valid            # already aligned to the same valid_mask subset

print(f"Shared row count check: LR={len(lr_probs_shared)}, RF={len(rf_probs_shared)}, "
      f"BN={len(bn_probs_shared)}, y={len(y_shared)} (all should be 29998)")

# Alignment sanity check: df_test.index[valid_mask] should be the same index
# order used to build y_test_valid and bn_probs_valid back in Cell 45
shared_index = df_test.index[valid_mask]
print(f"Index alignment check -- first 5 shared indices: {shared_index[:5].tolist()}")
print(f"bn_probs_valid (as Series, if applicable) index matches: "
      f"{'Confirmed positional alignment via valid_mask -- same boolean mask applied to all arrays' }")

decision_LR = (lr_probs_shared >= tau_lr).astype(int)
decision_RF = (rf_probs_shared >= tau_rf).astype(int)
decision_BN = (bn_probs_shared >= tau_bn).astype(int)

print(f"\nDecision counts -- LR: {decision_LR.sum()} predicted default, "
      f"RF: {decision_RF.sum()}, BN: {decision_BN.sum()} (out of {len(decision_LR)})")

# --- Step 2: Pairwise and three-way agreement ---

agree_lr_rf = (decision_LR == decision_RF).mean()
agree_lr_bn = (decision_LR == decision_BN).mean()
agree_rf_bn = (decision_RF == decision_BN).mean()
agree_all_three = ((decision_LR == decision_RF) & (decision_RF == decision_BN)).mean()

print(f"\n=== Overall agreement ===")
print(f"LR vs RF: {agree_lr_rf*100:.2f}%")
print(f"LR vs BN: {agree_lr_bn*100:.2f}%")
print(f"RF vs BN: {agree_rf_bn*100:.2f}%")
print(f"All three simultaneously: {agree_all_three*100:.2f}%")

# Conditional on true label
for label_val, label_name in [(1, 'Default=1 (actual defaulters)'), (0, 'Default=0 (actual non-defaulters)')]:
    mask_label = (y_shared == label_val)
    n_label = mask_label.sum()
    agree_all_label = ((decision_LR[mask_label] == decision_RF[mask_label]) &
                        (decision_RF[mask_label] == decision_BN[mask_label])).mean()
    agree_lr_rf_label = (decision_LR[mask_label] == decision_RF[mask_label]).mean()
    agree_lr_bn_label = (decision_LR[mask_label] == decision_BN[mask_label]).mean()
    agree_rf_bn_label = (decision_RF[mask_label] == decision_BN[mask_label]).mean()

    print(f"\n=== Agreement conditional on {label_name} (n={n_label}) ===")
    print(f"LR vs RF: {agree_lr_rf_label*100:.2f}%")
    print(f"LR vs BN: {agree_lr_bn_label*100:.2f}%")
    print(f"RF vs BN: {agree_rf_bn_label*100:.2f}%")
    print(f"All three simultaneously: {agree_all_label*100:.2f}%")

# --- Step 3: Disagreement inspection ---

unanimous = (decision_LR == decision_RF) & (decision_RF == decision_BN)
disagreement_mask = ~unanimous
n_disagree = disagreement_mask.sum()

print(f"\n=== Disagreement breakdown ===")
print(f"Total disagreement rows: {n_disagree} ({n_disagree/len(decision_LR)*100:.2f}%)")

# Among disagreement rows: 2-agree-1-differs vs all-three-differ
lr_rf_agree_disagree_subset = (decision_LR[disagreement_mask] == decision_RF[disagreement_mask])
lr_bn_agree_disagree_subset = (decision_LR[disagreement_mask] == decision_BN[disagreement_mask])
rf_bn_agree_disagree_subset = (decision_RF[disagreement_mask] == decision_BN[disagreement_mask])

# With binary decisions, "all three differ" is impossible (only 2 possible values),
# so every disagreement row is necessarily a 2-agree-1-outlier case. Identify the outlier.
outlier_is_lr = lr_disagree_from_pair = (~lr_rf_agree_disagree_subset) & (~lr_bn_agree_disagree_subset) & rf_bn_agree_disagree_subset
outlier_is_rf = (~lr_rf_agree_disagree_subset) & lr_bn_agree_disagree_subset & (~rf_bn_agree_disagree_subset)
outlier_is_bn = lr_rf_agree_disagree_subset & (~lr_bn_agree_disagree_subset) & (~rf_bn_agree_disagree_subset)

print(f"Note: with binary (0/1) decisions, 'all three differ' is mathematically impossible "
      f"(only 2 possible values) -- every disagreement is necessarily a 2-vs-1 split.")
print(f"LR is the outlier: {outlier_is_lr.sum()} rows ({outlier_is_lr.sum()/n_disagree*100:.1f}% of disagreements)")
print(f"RF is the outlier: {outlier_is_rf.sum()} rows ({outlier_is_rf.sum()/n_disagree*100:.1f}% of disagreements)")
print(f"BN is the outlier: {outlier_is_bn.sum()} rows ({outlier_is_bn.sum()/n_disagree*100:.1f}% of disagreements)")

# --- Disagreement rate: incomplete-evidence rows vs complete-evidence rows ---
# Need M_Income/M_Dependents for the shared row set
m_income_shared = df_test['M_Income'].values[valid_mask]
m_dependents_shared = df_test['M_Dependents'].values[valid_mask]
incomplete_evidence_mask = (m_income_shared == 0) | (m_dependents_shared == 0)

n_incomplete = incomplete_evidence_mask.sum()
n_complete = (~incomplete_evidence_mask).sum()

disagree_rate_incomplete = disagreement_mask[incomplete_evidence_mask].mean()
disagree_rate_complete = disagreement_mask[~incomplete_evidence_mask].mean()

print(f"\n=== Disagreement rate by evidence completeness ===")
print(f"Incomplete-evidence rows (n={n_incomplete}): disagreement rate = {disagree_rate_incomplete*100:.2f}%")
print(f"Complete-evidence rows (n={n_complete}): disagreement rate = {disagree_rate_complete*100:.2f}%")
print(f"Ratio (incomplete/complete): {disagree_rate_incomplete/disagree_rate_complete:.2f}x" 
      if disagree_rate_complete > 0 else "N/A")

In [ ]:
# Cell 56: RQ2 / VoI -- Step 0 checks (read-only; nothing is modified or saved)

import os, copy, pickle, time
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print("PYTHONHASHSEED:", os.environ.get('PYTHONHASHSEED'), "(expect '0')")

FEATURE_BINS = [
    'Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)',
    'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
    '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)',
    'Monthly Income (bin)', 'Number of Dependents (bin)',
]
missing_cols = [c for c in FEATURE_BINS + ['Default'] if c not in df_test.columns]
assert not missing_cols, f"df_test is missing {missing_cols} -- run the data-prep cells first"

# Indicator columns, built exactly as in RQ1
df_test['M_Income'] = df_test['Monthly Income (bin)'].notna().astype(int)
df_test['M_Dependents'] = df_test['Number of Dependents (bin)'].notna().astype(int)

# --- Load final model (deep-copied CPDs, never modified) and saved RQ1 predictions ---
with open('bn_final_model_iteration79.pkl', 'rb') as f:
    saved_model = pickle.load(f)
bn = DiscreteBayesianNetwork(saved_model['model_edges'])
bn.add_cpds(*copy.deepcopy(saved_model['cpds']))
assert bn.check_model()
infer = VariableElimination(bn)

with open('bn_test_predictions.pkl', 'rb') as f:
    bn_saved = pickle.load(f)['bn_test_probs']
assert bn_saved.index.equals(df_test.index), "saved prediction index != df_test index"

# --- Same evidence convention as the RQ1 scoring run ---
def build_evidence(row):
    ev = {col: row[col] for col in FEATURE_BINS if pd.notnull(row[col])}
    ev['M_Income'] = int(row['M_Income'])
    ev['M_Dependents'] = int(row['M_Dependents'])
    return ev

cache = {}
def posterior_default(evidence):
    """P(Default=1 | evidence); NaN if the evidence has zero probability. Cached by evidence tuple."""
    key = tuple(sorted(evidence.items(), key=lambda kv: kv[0]))
    if key in cache:
        return cache[key]
    with np.errstate(invalid='ignore', divide='ignore'):
        try:
            p = infer.query(variables=['Default'], evidence=evidence,
                            show_progress=False).get_value(Default=1)
        except Exception:
            p = np.nan
    if not np.isfinite(p) or not (0 <= p <= 1):
        p = np.nan
    cache[key] = p
    return p

# ===== (a) Evidence convention for rows with missing income / dependents =====
print("\n=== (a) Evidence convention ===")
ex_inc = df_test[(df_test['M_Income'] == 0) & (df_test['M_Dependents'] == 1)].iloc[0]
ex_both = df_test[(df_test['M_Income'] == 0) & (df_test['M_Dependents'] == 0)].iloc[0]
for name, r in [('Income missing only', ex_inc), ('Income + Dependents missing', ex_both)]:
    ev = build_evidence(r)
    print(f"{name}: {len(ev)} evidence entries; "
          f"M_Income={ev['M_Income']}, M_Dependents={ev['M_Dependents']}; "
          f"'Monthly Income (bin)' in evidence: {'Monthly Income (bin)' in ev}; "
          f"'Number of Dependents (bin)' in evidence: {'Number of Dependents (bin)' in ev}")

# ===== (b) Baseline reproduction against the saved RQ1 predictions =====
t0 = time.time()
new_probs = pd.Series([posterior_default(build_evidence(r)) for _, r in df_test.iterrows()],
                      index=df_test.index)
print(f"\nRecomputed all {len(new_probs)} test posteriors in {time.time()-t0:.1f}s "
      f"({len(cache)} distinct evidence tuples)")

complete_mask = ((df_test['M_Income'] == 1) & (df_test['M_Dependents'] == 1)
                 & df_test[FEATURE_BINS].notna().all(axis=1))
both_valid = new_probs.notna() & bn_saved.notna()

diff_complete = (new_probs[complete_mask & both_valid] - bn_saved[complete_mask & both_valid]).abs().max()
diff_incomplete = (new_probs[~complete_mask & both_valid] - bn_saved[~complete_mask & both_valid]).abs().max()
print("\n=== (b) Baseline reproduction ===")
print(f"Max abs diff, COMPLETE rows (required < 1e-8): {diff_complete:.3e} "
      f"-> {'PASS' if diff_complete < 1e-8 else '*** FAIL -- STOP AND REPORT ***'}")
print(f"Max abs diff, INCOMPLETE rows (confirms convention a): {diff_incomplete:.3e}")
print(f"NaN positions identical to saved vector: {(new_probs.isna() == bn_saved.isna()).all()}")

# ===== (c) Learned CPD columns for M = 0 =====
def cpt_df(model, variable):
    cpd = model.get_cpds(variable)
    parents = list(cpd.variables[1:])
    cols = pd.MultiIndex.from_product([cpd.state_names[p] for p in parents], names=parents)
    return pd.DataFrame(cpd.get_values(),
                        index=pd.Index(cpd.state_names[variable], name=variable), columns=cols)

print("\n=== (c) Learned CPDs under M = 0 (rows = parent combination, columns = child state) ===")
for child, m_var in [('Monthly Income (bin)', 'M_Income'),
                     ('Number of Dependents (bin)', 'M_Dependents')]:
    cpt = cpt_df(bn, child)
    m0 = cpt.xs(0, axis=1, level=m_var)
    m1 = cpt.xs(1, axis=1, level=m_var)
    k = cpt.shape[0]
    dev0 = (m0 - 1.0 / k).abs().max(axis=0)
    dev1 = (m1 - 1.0 / k).abs().max(axis=0)
    print(f"\n--- {child} | {m_var} = 0  ({m0.shape[1]} parent combinations) ---")
    print(m0.T.round(3))
    print(f"Columns exactly uniform (max dev from 1/{k} < 1e-6): "
          f"M=0: {(dev0 < 1e-6).sum()}/{m0.shape[1]}, M=1: {(dev1 < 1e-6).sum()}/{m1.shape[1]}")
    print(f"Mean max-probability per column: M=0: {m0.max(axis=0).mean():.3f}, "
          f"M=1: {m1.max(axis=0).mean():.3f}   (uniform would be {1.0/k:.3f})")

# ===== (d) Complete test rows =====
print("\n=== (d) Complete test rows ===")
known_zero = new_probs[new_probs.isna()].index.tolist()
print(f"Test rows: {len(df_test)}; M_Income=0: {(df_test['M_Income']==0).sum()}, "
      f"M_Dependents=0: {(df_test['M_Dependents']==0).sum()}")
print(f"Complete rows (M_Income=1, M_Dependents=1, all 10 bins non-null): {complete_mask.sum()}")
print(f"Zero-probability rows (NaN): {known_zero}; "
      f"of these in the complete set: {[i for i in known_zero if complete_mask[i]]}")

complete_rows_idx = df_test.index[complete_mask & new_probs.notna()]
print(f"Analysis set after exclusion: {len(complete_rows_idx)} rows")

In [ ]:
# Cell 57: Verify the Markov-blanket explanation before any RQ2 analysis (read-only, nothing saved)

parents = set(bn.get_parents('Default'))
children = set(bn.successors('Default'))
co_parents = set().union(*[set(bn.get_parents(c)) for c in children]) - {'Default'}
mb = sorted(parents | children | co_parents)
print("Default parents :", sorted(parents))
print("Default children:", sorted(children))
print("Markov blanket  :", mb)

# 1. Do complete rows with identical blanket values always get an identical posterior?
sub = df_test.loc[complete_rows_idx, mb].copy()
sub['p'] = new_probs.loc[complete_rows_idx].values
g = sub.groupby(mb, observed=True)['p']
spread = g.max() - g.min()
print(f"\nDistinct blanket combinations (complete rows): {g.ngroups}")
print(f"Max within-combination spread of P(Default=1 | e): {spread.max():.3e}   (0 if the claim holds)")

n_possible = int(np.prod([len(bn.get_cpds(v).state_names[v]) for v in mb]))
print(f"Possible blanket combinations: {n_possible}; "
      f"distinct in full test set: {len(df_test[mb].drop_duplicates())}; "
      f"unique saved RQ1 predictions: {bn_saved.nunique()}")

# 2. Direct perturbation on 200 random complete rows (diagnostic only, not the RQ2 run)
rng = np.random.RandomState(42)
sample_idx = rng.choice(complete_rows_idx.values, size=200, replace=False)

def max_shift_for(var):
    worst = 0.0
    for idx in sample_idx:
        ev = build_evidence(df_test.loc[idx])
        base = posterior_default(ev)
        if var in ('M_Income', 'M_Dependents'):
            value_var = 'Monthly Income (bin)' if var == 'M_Income' else 'Number of Dependents (bin)'
            alts = [{**{k: v for k, v in ev.items() if k != value_var}, var: 0}]
        else:
            alts = [{**ev, var: s} for s in bn.get_cpds(var).state_names[var] if s != ev[var]]
        for ev2 in alts:
            p2 = posterior_default(ev2)
            if np.isfinite(p2):
                worst = max(worst, abs(p2 - base))
    return worst

rows = [{'variable': v, 'in blanket': v in mb, 'max |shift|, 200 rows': max_shift_for(v)}
        for v in FEATURE_BINS + ['M_Income', 'M_Dependents']]
print("\n" + pd.DataFrame(rows).sort_values('max |shift|, 200 rows', ascending=False)
      .to_string(index=False, float_format=lambda x: f"{x:.3e}"))

In [ ]:
# Cell 58: Step 1 -- final EM CPDs of Default, 30-59, 60-89 vs plain DiscreteMLE on the full df_train (read-only)
from pgmpy.parameter_estimator import DiscreteMLE

targets = ['Default', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)']
family = sorted(set(targets).union(*[set(bn.get_parents(t)) for t in targets]))
print("Targets + their parents:", family)
assert df_train[family].notna().all().all(), "NaN in family columns -- plain MLE would not be comparable"

# Plain MLE on the induced sub-network (same parents for the 3 targets), all 119,999 rows
sub_edges = [(u, v) for (u, v) in bn.edges() if u in family and v in family]
sub_model = DiscreteBayesianNetwork(sub_edges)
sub_model.fit(df_train[family], estimator=DiscreteMLE(
    state_names={v: saved_model['state_names'][v] for v in family}))
for t in targets:
    assert sorted(sub_model.get_parents(t)) == sorted(bn.get_parents(t))

def canonical_cpt(model, var, state_names):
    """CPT as a DataFrame with parents in sorted order and states in saved order (so two models align)."""
    cpd = model.get_cpds(var)
    parents = list(cpd.variables[1:])
    idx = pd.Index(cpd.state_names[var], name=var)
    cols = pd.MultiIndex.from_product([cpd.state_names[p] for p in parents], names=parents)
    df = pd.DataFrame(cpd.get_values(), index=idx, columns=cols)
    order = sorted(parents)
    df = df.reorder_levels(order, axis=1)
    target_cols = pd.MultiIndex.from_product([state_names[p] for p in order], names=order)
    return df.reindex(index=state_names[var], columns=target_cols)

rows = []
for t in targets:
    em = canonical_cpt(bn, t, saved_model['state_names'])
    mle = canonical_cpt(sub_model, t, saved_model['state_names'])
    diff = (em - mle).abs()
    rows.append({'node': t, 'n_parent_combinations': em.shape[1],
                 'combinations_NaN_in_MLE': int(mle.isna().any(axis=0).sum()),
                 'max_abs_diff (finite entries)': float(np.nanmax(diff.values))})
print("\n" + pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Cell 59: Step 2 -- rescore rows 125238 and 7312 using only the Markov-blanket evidence (read-only)
parents_d = set(bn.get_parents('Default')); children_d = set(bn.successors('Default'))
co_parents = set().union(*[set(bn.get_parents(c)) for c in children_d]) - {'Default'}
mb = sorted(parents_d | children_d | co_parents)
print("Markov blanket:", mb)

for idx in [125238, 7312]:
    row = df_test.loc[idx]
    ev_b = {v: row[v] for v in mb}
    p_post = posterior_default(ev_b)
    with np.errstate(invalid='ignore', divide='ignore'):
        p_blanket = infer.query(variables=mb, show_progress=False).get_value(**ev_b)
    mask = np.ones(len(df_train), dtype=bool)
    for k, v in ev_b.items():
        mask &= (df_train[k] == v).values
    print(f"\nRow {idx}: blanket evidence = {ev_b}")
    print(f"  P(blanket evidence) under the BN : {p_blanket:.6g}")
    print(f"  P(Default=1 | blanket evidence)  : {p_post}  "
          f"({'still undefined/zero' if np.isnan(p_post) else 'defined'})")
    print(f"  Training rows with this blanket combination: {mask.sum()}")
    print(f"  Actual Default = {row['Default']}", end="")
    if not np.isnan(p_post):
        print(f"; decision at tau_BN=0.17: {int(p_post >= 0.17)}")
    else:
        print()

In [ ]:
# Cell 60: Step 3 -- RQ2 one-way evidence sensitivity, all test rows except the 2 zero-probability rows
TAU_BN = 0.17
VALUE_OF = {'M_Income': 'Monthly Income (bin)', 'M_Dependents': 'Number of Dependents (bin)'}
VARS12 = FEATURE_BINS + ['M_Income', 'M_Dependents']
STATES = {v: bn.get_cpds(v).state_names[v] for v in FEATURE_BINS}
analysis_idx = new_probs.index[new_probs.notna()]
print(f"Analysis rows: {len(analysis_idx)} (expect 29998)")

def row_sensitivity(ev, base):
    """[(variable, S, n_skipped_states)] for every variable defined for this row."""
    out = []
    for var in FEATURE_BINS:
        if var not in ev:
            continue
        diffs, skipped = [], 0
        for s in STATES[var]:
            if s == ev[var]:
                continue
            p2 = posterior_default({**ev, var: s})
            if np.isnan(p2):
                skipped += 1
            else:
                diffs.append(abs(p2 - base))
        out.append((var, max(diffs) if diffs else np.nan, skipped))
    for m, val_var in VALUE_OF.items():
        if ev[m] == 1:
            ev2 = {k: v for k, v in ev.items() if k != val_var}
            ev2[m] = 0
            p2 = posterior_default(ev2)
            out.append((m, np.nan if np.isnan(p2) else abs(p2 - base), int(np.isnan(p2))))
    return out

def summary_table(df, col, skip_col):
    rows = []
    for var, g in df.groupby('variable', sort=False):
        x = g[col].dropna().values
        if len(x) == 0:
            continue
        rows.append({'variable': var, 'n_rows': len(x), 'mean': x.mean(), 'median': np.median(x),
                     'std': x.std(), 'p90': np.percentile(x, 90), 'p95': np.percentile(x, 95),
                     'p99': np.percentile(x, 99), 'max': x.max(),
                     'share>0.01': (x > 0.01).mean(), 'share>0.05': (x > 0.05).mean(),
                     'share>0.10': (x > 0.10).mean(), 'share_zero': (x < 1e-12).mean(),
                     'skipped_states': int(g[skip_col].sum())})
    return pd.DataFrame(rows).sort_values('mean', ascending=False)

# Timing test on 10 rows
t0 = time.time()
for idx in analysis_idx[:10]:
    row_sensitivity(build_evidence(df_test.loc[idx]), new_probs[idx])
per_row = (time.time() - t0) / 10
projected_min = per_row * len(analysis_idx) / 60
print(f"Timing: {per_row*1000:.1f} ms/row; projected full run (no cross-row cache hits assumed): {projected_min:.1f} min")
assert projected_min < 30, "Projected > 30 min -- stop and report before the full run"

# Full run
records, t0 = [], time.time()
for i, idx in enumerate(analysis_idx):
    ev = build_evidence(df_test.loc[idx]); base = new_probs[idx]
    for var, S, skipped in row_sensitivity(ev, base):
        records.append((idx, var, S, base, skipped))
    if (i + 1) % 5000 == 0:
        print(f"  {i+1}/{len(analysis_idx)} rows, {time.time()-t0:.0f}s, cache={len(cache)}")
rq2_df = pd.DataFrame(records, columns=['row_index', 'variable', 'S', 'baseline_prob', 'skipped_states'])
print(f"Full run: {time.time()-t0:.0f}s, {len(rq2_df)} row-variable records")

pd.set_option('display.float_format', lambda x: f"{x:.4f}")
print("\n=== RQ2: blanket variables (ranked by mean S) ===")
print(summary_table(rq2_df[rq2_df.variable.isin(mb)], 'S', 'skipped_states').to_string(index=False))

nb = rq2_df[~rq2_df.variable.isin(mb)]
print("\n=== RQ2: non-blanket variables (expected exactly zero) ===")
print(nb.groupby('variable').agg(n_rows=('S', 'count'), max_S=('S', 'max'),
                                 skipped_states=('skipped_states', 'sum')).to_string())
rq2_df.to_pickle('rq2_sensitivity_per_row.pkl')
print("Saved:", os.path.abspath('rq2_sensitivity_per_row.pkl'))
assert nb['S'].max() < 1e-12, "non-blanket variable shows non-zero shift"

In [ ]:
# Cell 61: Step 4 -- Extension Part A: retract one blanket variable at a time, flip rate at tau_BN = 0.17
rows = []
for i, idx in enumerate(analysis_idx):
    ev = build_evidence(df_test.loc[idx]); base = new_probs[idx]
    for var in mb:
        p2 = posterior_default({k: v for k, v in ev.items() if k != var})
        if np.isnan(p2):
            rows.append((idx, var, np.nan, base, np.nan, np.nan, 1))
        else:
            rows.append((idx, var, abs(p2 - base), base, p2,
                         int((base >= TAU_BN) != (p2 >= TAU_BN)), 0))
    if (i + 1) % 10000 == 0:
        print(f"  {i+1}/{len(analysis_idx)} rows")
partA = pd.DataFrame(rows, columns=['row_index', 'variable', 'delta', 'baseline_prob',
                                    'retracted_prob', 'flip', 'skipped'])

tbl = summary_table(partA, 'delta', 'skipped')
tbl['flip_rate'] = tbl['variable'].map(partA.groupby('variable')['flip'].mean())
print("\n=== Part A: delta = |P(Default=1|e) - P(Default=1|e without Xj)|, tau_BN = 0.17 ===")
print(tbl.to_string(index=False))
partA.to_pickle('vo_partA_retraction_per_row.pkl')
print("Saved:", os.path.abspath('vo_partA_retraction_per_row.pkl'))

# Part B: one-line confirmation of zero. Observe the missing value (M=1, value=x) on naturally incomplete rows.
def partB_check(m_var, val_var):
    rows_idx = [i for i in analysis_idx if df_test.at[i, m_var] == 0]
    worst, flips, skipped = 0.0, 0, 0
    for idx in rows_idx:
        ev = build_evidence(df_test.loc[idx]); base = new_probs[idx]
        for s in STATES[val_var]:
            p2 = posterior_default({**ev, m_var: 1, val_var: s})
            if np.isnan(p2):
                skipped += 1
                continue
            worst = max(worst, abs(p2 - base))
            flips += int((base >= TAU_BN) != (p2 >= TAU_BN))
    return len(rows_idx), worst, flips, skipped

for m_var, val_var in VALUE_OF.items():
    n, worst, flips, skipped = partB_check(m_var, val_var)
    print(f"Part B check, {val_var}: {n} incomplete rows, max shift over states = {worst:.3e}, "
          f"decision flips = {flips}, skipped states = {skipped}  ->  EPS = EVOI = {'0' if worst < 1e-12 else 'NOT zero'}")

In [ ]:
# Cell 61b: diagnose the 0.99 plateau in 60-89 S (read-only)
d60 = rq2_df[rq2_df.variable == '60-89 Days Past Due (bin)']
share_at_cap = (d60['S'] > 0.989).mean()
print(f"60-89: share of rows with S > 0.989: {share_at_cap:.3f}")
print("Baseline probs on those rows (value counts, top 5):")
print(d60.loc[d60['S'] > 0.989, 'baseline_prob'].round(4).value_counts().head(5))

# Which swapped-to state produces the jump? Look at one example row.
ex_idx = d60.loc[d60['S'] > 0.989, 'row_index'].iloc[0]
ev = build_evidence(df_test.loc[ex_idx])
print("\nExample row:", {k: ev[k] for k in mb})
for s in STATES['60-89 Days Past Due (bin)']:
    p2 = posterior_default({**ev, '60-89 Days Past Due (bin)': s})
    print(f"  60-89 = {s}: P(Default=1|e') = {p2}")

# How many training rows sit in the swapped-to state, with Default = 0?
print("\nTraining rows by 60-89 state and Default:")
print(pd.crosstab(df_train['60-89 Days Past Due (bin)'], df_train['Default']))

In [ ]:
# Cell 62: Step 5 -- ablations with the tuned settings (test AUC vs locked LR 0.8618 / RF 0.8603)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

assert 'X_train_df' in globals() and 'X_test_df' in globals(), \
    "Run the matrix-completion and encoding cells first (X_train_df / X_test_df)"
assert X_train_df.shape == (119999, 35) and X_test_df.shape == (30000, 35)
y_train = df_clean.loc[train_idx, 'Default'].values
y_test = df_clean.loc[test_idx, 'Default'].values

def cols_for(vars_):
    return [c for c in X_train_df.columns if any(c.startswith(v + '_') for v in vars_)]
blanket_cols = cols_for(mb)
no_inc_dep = [c for c in X_train_df.columns
              if c not in cols_for(['Monthly Income (bin)', 'Number of Dependents (bin)'])]
print(f"Blanket-only columns: {len(blanket_cols)} (expect 13); without Income/Dependents: {len(no_inc_dep)} (expect 28)")

make_lr = lambda: LogisticRegression(C=100, penalty='l2', solver='saga', class_weight='balanced',
                                     max_iter=1000, random_state=42)
make_rf = lambda: RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=10,
                                         class_weight='balanced', random_state=42, n_jobs=-1)
LR_BASE, RF_BASE = 0.8618, 0.8603

configs = [
    ('Control: LR, all 35 features (refit)', make_lr, list(X_train_df.columns), LR_BASE),
    ('Control: RF, all 35 features (refit)', make_rf, list(X_train_df.columns), RF_BASE),
    ('(a) LR, 4 blanket variables only',      make_lr, blanket_cols, LR_BASE),
    ('(b) LR, without Income + Dependents',   make_lr, no_inc_dep, LR_BASE),
    ('(c) RF, without Income + Dependents',   make_rf, no_inc_dep, RF_BASE),
]
results = []
for name, factory, cols, base_auc in configs:
    t0 = time.time()
    model = factory().fit(X_train_df[cols], y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_df[cols])[:, 1])
    results.append({'model': name, 'n_features': len(cols), 'test_AUC': auc,
                    'diff_vs_locked': auc - base_auc, 'fit_s': time.time() - t0})
    print(f"  done: {name} ({time.time()-t0:.0f}s)")

bn_ref_auc = pickle.load(open('bn_test_predictions.pkl', 'rb'))['bn_test_auc']
print("\n" + pd.DataFrame(results).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nReference: locked LR {LR_BASE}, locked RF {RF_BASE}, BN (saved) {bn_ref_auc:.4f}")

In [ ]:
# Cell 63: Step 1 -- isolate the zero-cell mechanisms (read-only; nothing saved)
from collections import Counter

A30, B60, C90, CUR = ('30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)',
                      '90+ Days Late (bin)', 'Credit Utilisation Rate (bin)')
short = lambda n: n.replace(' (bin)', '')
assert set(bn.get_parents('Default')) == {C90, CUR}
assert set(bn.get_parents(B60)) == {CUR, C90, 'Default'}
assert set(bn.get_parents(A30)) == {CUR, 'Default', B60}
print("Parent sets match the factorisation used below.")

def cpd_val(model, var, asg):
    cpd = model.get_cpds(var)
    kw = {var: asg[var]}
    for p in cpd.variables[1:]:
        kw[p] = asg[p]
    return float(cpd.get_value(**kw))

def cpd_column(model, var, asg):
    cpd = model.get_cpds(var)
    out = {}
    for s in cpd.state_names[var]:
        kw = {var: s}
        for p in cpd.variables[1:]:
            kw[p] = asg[p]
        out[str(s)] = round(float(cpd.get_value(**kw)), 6)
    return out

FACTORS = ['P(Default|90+,CUR)', 'P(60-89|Default,CUR,90+)', 'P(30-59|Default,CUR,60-89)']

def fam_terms(model, a, b, c, u):
    """The three Default-dependent factors for d=0 and d=1; T[d] is their product."""
    f = {}
    for d in (0, 1):
        asg = {A30: a, B60: b, C90: c, CUR: u, 'Default': d}
        f[d] = [cpd_val(model, 'Default', asg), cpd_val(model, B60, asg), cpd_val(model, A30, asg)]
    T = {d: float(np.prod(f[d])) for d in (0, 1)}
    tot = T[0] + T[1]
    return f, T, (T[1] / tot if tot > 0 else np.nan)

def count_rows(conds):
    m = np.ones(len(df_train), dtype=bool)
    for k, v in conds.items():
        m &= (df_train[k] == v).values
    return int(m.sum())

def support(i, d, a, b2, c, u):
    """(rows in the parent configuration, rows in the cell) in df_train for factor i under Default=d."""
    if i == 0:
        par = {C90: c, CUR: u};                 cell = {**par, 'Default': d}
    elif i == 1:
        par = {'Default': d, CUR: u, C90: c};   cell = {**par, B60: b2}
    else:
        par = {'Default': d, CUR: u, B60: b2};  cell = {**par, A30: a}
    return count_rows(par), count_rows(cell)

def zero_nodes(model, asg):
    return tuple(sorted(n for n in model.nodes() if cpd_val(model, n, asg) == 0.0))

# --- Validate the decomposition against the engine on 500 random rows ---
rng = np.random.RandomState(42)
gaps = []
for idx in rng.choice(analysis_idx.values, size=500, replace=False):
    r = df_test.loc[idx]
    _, _, p_dec = fam_terms(bn, r[A30], r[B60], r[C90], r[CUR])
    gaps.append(abs(p_dec - new_probs[idx]))
print(f"Decomposition vs engine posterior, 500 rows: max abs diff = {np.nanmax(gaps):.2e}")

# --- (1a) the 60-89 plateau ---
n60 = int((rq2_df.variable == B60).sum())
hi = rq2_df[(rq2_df.variable == B60) & (rq2_df.S > 0.989)]
print(f"\n(1a) 60-89 rows with S > 0.989: {len(hi)} of {n60} ({len(hi)/n60*100:.1f}%)")
print("Baseline P(Default=1|e) on those rows, 6 decimals:")
print(hi['baseline_prob'].round(6).value_counts().head(8).to_string())

combo_counts = (df_test.loc[hi.row_index.values, mb].groupby(mb, observed=True)
                .size().sort_values(ascending=False))
print(f"Distinct blanket combinations among them: {len(combo_counts)}")

cause = Counter()
for rank, (key, n) in enumerate(combo_counts.items()):
    k = dict(zip(mb, key)); a, b, c, u = k[A30], k[B60], k[C90], k[CUR]
    f_base, T_base, base = fam_terms(bn, a, b, c, u)
    best = None
    for b2 in STATES[B60]:
        if b2 == b:
            continue
        f2, T2, p2 = fam_terms(bn, a, b2, c, u)
        if np.isnan(p2):
            continue
        if best is None or abs(p2 - base) > best['shift']:
            best = dict(shift=abs(p2 - base), b2=b2, p2=p2, f=f2, T=T2)
    zf0 = tuple(FACTORS[i] for i, v in enumerate(best['f'][0]) if v == 0.0)
    zf1 = tuple(FACTORS[i] for i, v in enumerate(best['f'][1]) if v == 0.0)
    cause[(str(best['b2']), zf0, zf1)] += n
    if rank < 6:
        print(f"\n  #{rank+1}: {n} rows | a={a}, b={b}, c={c}, u={u}")
        print(f"     baseline {base:.6f}; T0={T_base[0]:.3e}, T1={T_base[1]:.3e}")
        print(f"     factors d=0: {[f'{v:.4g}' for v in f_base[0]]}   d=1: {[f'{v:.4g}' for v in f_base[1]]}")
        print(f"     max-shift swap 60-89: {b} -> {best['b2']}, P(Default=1) = {best['p2']:.6f}; "
              f"T0={best['T'][0]:.3e}, T1={best['T'][1]:.3e}")
        print(f"     swapped factors d=0: {[f'{v:.4g}' for v in best['f'][0]]}   d=1: {[f'{v:.4g}' for v in best['f'][1]]}")
        for d in (0, 1):
            for i, v in enumerate(best['f'][d]):
                if v == 0.0:
                    pn, cn = support(i, d, a, best['b2'], c, u)
                    print(f"     ZERO factor d={d}: {FACTORS[i]} -> parent rows {pn}, cell rows {cn}")

print("\n(1a) Cause summary over all rows with S > 0.989 (swap state, zero factors d=0, zero factors d=1):")
for (b2, z0, z1), n in sorted(cause.items(), key=lambda kv: -kv[1]):
    print(f"  {n:5d} rows | swap to {b2} | d=0 zero: {list(z0)} | d=1 zero: {list(z1)}")

# CPD columns for the largest combination
key, n = next(iter(combo_counts.items()))
k = dict(zip(mb, key)); a, b, c, u = k[A30], k[B60], k[C90], k[CUR]
print(f"\nCPD cells for the largest combination (c={c}, u={u}):")
print("  P(Default | 90+, CUR):", cpd_column(bn, 'Default', {C90: c, CUR: u}))
for d in (0, 1):
    print(f"  P(60-89 | Default={d}, CUR, 90+):", cpd_column(bn, B60, {'Default': d, CUR: u, C90: c}))
    print(f"  P(30-59 | Default={d}, CUR, 60-89={b}):", cpd_column(bn, A30, {'Default': d, CUR: u, B60: b}))
print("  60-89 states by Default in df_train:")
print(pd.crosstab(df_train[B60], df_train['Default']).to_string())

# --- (1b) skipped states for Credit Utilisation Rate ---
sk = rq2_df[(rq2_df.variable == CUR) & (rq2_df.skipped_states > 0)]
print(f"\n(1b) CUR: {len(sk)} rows with skipped states; total skipped = {int(sk.skipped_states.sum())}")
sig, n_pairs, n_incomplete = Counter(), 0, 0
for idx in sk.row_index.values:
    ev = build_evidence(df_test.loc[idx])
    for u2 in STATES[CUR]:
        if u2 == ev[CUR]:
            continue
        ev2 = {**ev, CUR: u2}
        if not np.isnan(posterior_default(ev2)):
            continue
        n_pairs += 1
        if len(ev) != 12:      # missing Income/Dependents: decomposition not attempted
            n_incomplete += 1
            continue
        _, _, p_blk = fam_terms(bn, ev2[A30], ev2[B60], ev2[C90], u2)
        z0 = tuple(short(x) for x in zero_nodes(bn, {**ev2, 'Default': 0}))
        z1 = tuple(short(x) for x in zero_nodes(bn, {**ev2, 'Default': 1}))
        sig[('Default-family zero' if np.isnan(p_blk) else 'blanket defined, other CPD zero', z0, z1)] += 1
print(f"NaN (row, swapped state) pairs found: {n_pairs}; on incomplete-evidence rows (not decomposed): {n_incomplete}")
for (kind, z0, z1), n in sorted(sig.items(), key=lambda kv: -kv[1]):
    print(f"  {n:4d} | {kind} | zero CPD entries if Default=0: {list(z0)} | if Default=1: {list(z1)}")

In [ ]:
# Cell 64: Step 2 -- BDeu-smoothed copies of Default, 30-59 and 60-89 CPDs (ESS 1 and 10). The pickle is never modified.
import itertools
from pgmpy.factors.discrete import TabularCPD
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

TARGETS = ['Default', A30, B60]

def bdeu_cpd(cpd, ess):
    var = cpd.variable
    parents = list(cpd.variables[1:])
    states = list(cpd.state_names[var])
    pstates = [list(cpd.state_names[p]) for p in parents]
    combos = list(itertools.product(*pstates))
    counts = df_train.groupby(parents + [var], observed=True).size().to_dict()
    alpha = ess / (len(states) * len(combos))
    vals = np.zeros((len(states), len(combos)))
    for j, combo in enumerate(combos):
        n_s = np.array([counts.get(tuple(combo) + (s,), 0) for s in states], dtype=float)
        vals[:, j] = (n_s + alpha) / (n_s.sum() + len(states) * alpha)
    return TabularCPD(var, len(states), vals, evidence=parents,
                      evidence_card=[len(p) for p in pstates],
                      state_names={var: states, **dict(zip(parents, pstates))})

# Validate the manual formula: with a vanishing prior it must reproduce the EM CPDs
for var in TARGETS:
    orig = bn.get_cpds(var)
    diff = np.abs(bdeu_cpd(orig, 1e-9).get_values() - orig.get_values())
    print(f"{short(var)}: manual BDeu (ESS->0) vs saved CPD: max abs diff = {np.nanmax(diff):.2e}, "
          f"columns differing > 1e-6: {int((diff > 1e-6).any(axis=0).sum())}/{diff.shape[1]}")

def smoothed_model(ess):
    cpds = [bdeu_cpd(c, ess) if c.variable in TARGETS else copy.deepcopy(c) for c in saved_model['cpds']]
    m = DiscreteBayesianNetwork(saved_model['model_edges'])
    m.add_cpds(*cpds)
    assert m.check_model()
    return m

def make_posterior_fn(model):
    eng = VariableElimination(model)
    cache_m = {}
    def f(evidence):
        key = tuple(sorted(evidence.items(), key=lambda kv: kv[0]))
        if key in cache_m:
            return cache_m[key]
        with np.errstate(invalid='ignore', divide='ignore'):
            try:
                p = eng.query(variables=['Default'], evidence=evidence, show_progress=False).get_value(Default=1)
            except Exception:
                p = np.nan
        if not np.isfinite(p) or not (0 <= p <= 1):
            p = np.nan
        cache_m[key] = p
        return p
    return f

evid_all = {idx: build_evidence(df_test.loc[idx]) for idx in df_test.index}   # reused in Cells 65-66
y_all = df_test['Default'].values
valid = new_probs.notna().values            # the 29,998 rows the original BN could score
ref_auc = pickle.load(open('bn_test_predictions.pkl', 'rb'))['bn_test_auc']
zero_rows = [125238, 7312]

def eval_probs(probs):
    m = valid & np.isfinite(probs)
    p, y = probs[m], y_all[m]
    auc = roc_auc_score(y, p)
    return auc, 2 * auc - 1, ks_2samp(p[y == 1], p[y == 0]).statistic, int(m.sum())

smoothed, smoothed_probs = {}, {}
rows_out = []
auc0, gini0, ks0, n0 = eval_probs(new_probs.values)
rows_out.append({'model': 'original (EM)', 'n_NaN_of_30000': int(new_probs.isna().sum()),
                 'AUC': auc0, 'Gini': gini0, 'KS': ks0, 'dAUC_vs_saved': auc0 - ref_auc,
                 'unique_preds': new_probs.nunique(), 'max_abs_diff_vs_orig': 0.0})
for ess in (1, 10):
    t0 = time.time()
    smoothed[ess] = smoothed_model(ess)
    post = make_posterior_fn(smoothed[ess])
    probs = np.array([post(evid_all[i]) for i in df_test.index])
    smoothed_probs[ess] = probs
    auc, gini, ks, n_used = eval_probs(probs)
    m = valid & np.isfinite(probs)
    rows_out.append({'model': f'BDeu ESS={ess}', 'n_NaN_of_30000': int(np.isnan(probs).sum()),
                     'AUC': auc, 'Gini': gini, 'KS': ks, 'dAUC_vs_saved': auc - ref_auc,
                     'unique_preds': len(np.unique(probs[~np.isnan(probs)])),
                     'max_abs_diff_vs_orig': float(np.abs(probs[m] - new_probs.values[m]).max())})
    print(f"ESS={ess}: check_model True; scored 30,000 rows in {time.time()-t0:.0f}s")

res_tbl = pd.DataFrame(rows_out)
print("\n" + res_tbl.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"(AUC on the 29,998 shared rows; saved RQ1 BN AUC = {ref_auc:.4f})")
for r in rows_out[1:]:
    if abs(r['dAUC_vs_saved']) > 0.001:
        print(f"*** FLAG: {r['model']} moves AUC by {r['dAUC_vs_saved']:+.4f} (> 0.001) ***")

print("\nFootnote: the two zero-probability rows")
for ess in (1, 10):
    post = make_posterior_fn(smoothed[ess])
    for idx in zero_rows:
        p_full = post(evid_all[idx])
        p_blk = post({m_: df_test.at[idx, m_] for m_ in mb})
        dec = lambda p: 'n/a' if np.isnan(p) else int(p >= 0.17)
        print(f"  ESS={ess}, row {idx}: P(Default=1 | full evidence) = {p_full:.4f} (decision {dec(p_full)}), "
              f"| blanket-only = {p_blk:.4f} (decision {dec(p_blk)}), actual Default = {df_test.at[idx, 'Default']}")

In [ ]:
# Cell 65: Step 3 -- S_max and S_exp on the original model (also computes Delta and flip, reused in Cell 66)
TAU_BN = 0.17

def analyze_blanket(model, rows):
    post = make_posterior_fn(model)
    eng = VariableElimination(model)
    wcache = {}
    def pred_dist(var, ev_rest):
        key = (var, tuple(sorted(ev_rest.items(), key=lambda kv: kv[0])))
        if key in wcache:
            return wcache[key]
        with np.errstate(invalid='ignore', divide='ignore'):
            try:
                f = eng.query(variables=[var], evidence=ev_rest, show_progress=False)
                vals = np.asarray(f.values, dtype=float)
                w = dict(zip(f.state_names[var], vals)) if np.all(np.isfinite(vals)) else None
            except Exception:
                w = None
        wcache[key] = w
        return w
    recs, t0 = [], time.time()
    for i, idx in enumerate(rows):
        ev = evid_all[idx]
        base = post(ev)
        if np.isnan(base):
            continue
        for var in mb:
            xo = ev[var]
            e_rest = {k: v for k, v in ev.items() if k != var}
            shifts, skipped = {}, 0
            for s in STATES[var]:
                if s == xo:
                    continue
                p2 = post({**ev, var: s})
                if np.isnan(p2):
                    skipped += 1
                else:
                    shifts[s] = abs(p2 - base)
            S_max = max(shifts.values()) if shifts else np.nan
            w = pred_dist(var, e_rest)
            if w is None:
                S_exp, w_obs = np.nan, np.nan
            else:
                S_exp = sum(ws * shifts[s] for s, ws in w.items() if s != xo and ws > 0 and s in shifts)
                w_obs = w.get(xo, np.nan)
            p_r = post(e_rest)
            delta = np.nan if np.isnan(p_r) else abs(p_r - base)
            flip = np.nan if np.isnan(p_r) else int((base >= TAU_BN) != (p_r >= TAU_BN))
            recs.append((idx, var, S_max, S_exp, delta, flip, base, skipped, w_obs))
        if (i + 1) % 10000 == 0:
            print(f"  {i+1}/{len(rows)} rows, {time.time()-t0:.0f}s")
    return pd.DataFrame(recs, columns=['row_index', 'variable', 'S_max', 'S_exp', 'delta', 'flip',
                                       'baseline_prob', 'skipped', 'w_obs'])

def summarize(res):
    out = []
    for var, g in res.groupby('variable', sort=False):
        out.append({'variable': short(var), 'n': len(g),
                    'Smax_mean': g.S_max.mean(), 'Smax_med': g.S_max.median(), 'Smax_p95': g.S_max.quantile(.95),
                    'Sexp_mean': g.S_exp.mean(), 'Sexp_med': g.S_exp.median(), 'Sexp_p95': g.S_exp.quantile(.95),
                    'Sexp_p99': g.S_exp.quantile(.99), 'Sexp_max': g.S_exp.max(),
                    'Sexp>0.01': (g.S_exp > 0.01).mean(), 'Sexp>0.05': (g.S_exp > 0.05).mean(),
                    'w_obs_mean': g.w_obs.mean(),
                    'D_mean': g.delta.mean(), 'D_med': g.delta.median(), 'D_p95': g.delta.quantile(.95),
                    'flip': g.flip.mean(), 'skipped_Smax': int(g.skipped.sum())})
    return pd.DataFrame(out).sort_values('Smax_mean', ascending=False)

def show(summ):
    pd.set_option('display.float_format', lambda x: f"{x:.4f}")
    print(summ[['variable', 'n', 'Smax_mean', 'Smax_med', 'Smax_p95',
                'Sexp_mean', 'Sexp_med', 'Sexp_p95', 'Sexp_p99', 'Sexp_max',
                'Sexp>0.01', 'Sexp>0.05', 'w_obs_mean']].to_string(index=False))
    print(summ[['variable', 'D_mean', 'D_med', 'D_p95', 'flip', 'skipped_Smax']].to_string(index=False))

t0 = time.time()
res_orig = analyze_blanket(bn, analysis_idx)
print(f"Original model: {len(res_orig)} records in {time.time()-t0:.0f}s")

# Self-checks against Cells 60 and 61
c60 = res_orig.merge(rq2_df[['row_index', 'variable', 'S']], on=['row_index', 'variable'], how='left')
print(f"S_max vs Cell 60: max abs diff = {np.nanmax(np.abs(c60.S_max - c60.S)):.2e}; "
      f"NaN positions match: {(c60.S_max.isna() == c60.S.isna()).all()}")
c61 = res_orig.merge(partA[['row_index', 'variable', 'delta', 'flip']], on=['row_index', 'variable'], suffixes=('', '_A'))
print(f"Delta vs Cell 61: max abs diff = {np.nanmax(np.abs(c61.delta - c61.delta_A)):.2e}; "
      f"flip mismatches: {int((c61.flip.fillna(-1) != c61.flip_A.fillna(-1)).sum())}")

summ_orig = summarize(res_orig)
print("\n=== Original model: S_max and S_exp (blanket variables) ===")
show(summ_orig)
rank = lambda s, col: ' > '.join(s.sort_values(col, ascending=False).variable)
print("\nRank by S_max mean:", rank(summ_orig, 'Smax_mean'))
print("Rank by S_exp mean:", rank(summ_orig, 'Sexp_mean'))
print("Rank by Delta mean:", rank(summ_orig, 'D_mean'))
res_orig.to_pickle('vo_blanket_original_per_row.pkl')

In [ ]:
# Cell 66: Step 4 -- RQ2 (S_max, S_exp) and Part A (Delta, flip at 0.17) on both smoothed copies
ba = {'orig': res_orig}
for ess in (1, 10):
    t0 = time.time()
    ba[f'ess{ess}'] = analyze_blanket(smoothed[ess], analysis_idx)
    ba[f'ess{ess}'].to_pickle(f'vo_blanket_ess{ess}_per_row.pkl')
    print(f"\nESS={ess}: {len(ba[f'ess{ess}'])} records in {time.time()-t0:.0f}s")
    show(summarize(ba[f'ess{ess}']))

print("\n=== Stability across original / ESS 1 / ESS 10 ===")
metric_cols = [('S_max', 'S_max'), ('S_exp', 'S_exp'), ('delta', 'Delta (retraction)'), ('flip', 'flip rate')]
for col, label in metric_cols:
    wide = pd.DataFrame({name: res.groupby('variable')[col].mean() for name, res in ba.items()})
    wide.index = [short(v) for v in wide.index]
    orders = {name: ' > '.join(wide[name].sort_values(ascending=False).index) for name in wide.columns}
    print(f"\n{label} (mean per variable)")
    print(wide.to_string(float_format=lambda x: f"{x:.4f}"))
    for name, o in orders.items():
        print(f"  rank {name:6s}: {o}")
    print(f"  identical ranking across all three: {len(set(orders.values())) == 1}")

print("\n60-89: share of rows with S_max > 0.989 and total skipped states, by model")
for name, res in ba.items():
    g = res[res.variable == B60]
    print(f"  {name:6s}: {(g.S_max > 0.989).mean()*100:5.1f}% of rows | skipped (S_max) all four variables = {int(res.skipped.sum())}")

In [ ]:
# Cell 67: Step 5 -- paired bootstrap on test AUC (2,000 resamples, seed 42)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

assert X_train_df.shape == (119999, 35) and X_test_df.shape == (30000, 35)
y_train = df_clean.loc[train_idx, 'Default'].values
y_test = df_clean.loc[test_idx, 'Default'].values
assert X_test_df.index.equals(df_test.index) and (df_test['Default'].values == y_test).all()

def cols_for(vars_):
    return [c for c in X_train_df.columns if any(c.startswith(v + '_') for v in vars_)]
all_cols = list(X_train_df.columns)
blanket_cols = cols_for(mb)
no_inc_dep = [c for c in all_cols if c not in cols_for(['Monthly Income (bin)', 'Number of Dependents (bin)'])]

make_lr = lambda: LogisticRegression(C=100, penalty='l2', solver='saga', class_weight='balanced',
                                     max_iter=1000, random_state=42)
make_rf = lambda: RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=10,
                                         class_weight='balanced', random_state=42, n_jobs=-1)
spec = {'LR_all': (make_lr, all_cols, 0.8618), 'RF_all': (make_rf, all_cols, 0.8603),
        'LR_4var': (make_lr, blanket_cols, 0.8508), 'LR_noID': (make_lr, no_inc_dep, 0.8612),
        'RF_noID': (make_rf, no_inc_dep, 0.8599)}
preds = {}
for name, (factory, cols, expected) in spec.items():
    model = factory().fit(X_train_df[cols], y_train)
    preds[name] = model.predict_proba(X_test_df[cols])[:, 1]
    auc = roc_auc_score(y_test, preds[name])
    print(f"{name}: AUC {auc:.4f} (Cell 62: {expected}) -> reproduced: {abs(auc - expected) < 1e-4}")
preds['BN'] = new_probs.values
pickle.dump(preds, open('ablation_predictions.pkl', 'wb'))

def paired_bootstrap(y, pa, pb, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y)
    obs = roc_auc_score(y, pa) - roc_auc_score(y, pb)
    diffs = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yb = y[idx]
        if yb.min() == yb.max():
            continue
        diffs.append(roc_auc_score(yb, pa[idx]) - roc_auc_score(yb, pb[idx]))
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = max(2 * min((diffs > 0).mean(), (diffs < 0).mean()), 1.0 / len(diffs))
    return obs, lo, hi, p, len(diffs)

v = valid   # the 29,998 rows where the BN is involved
comparisons = [
    ('LR_all - LR_4var (30,000 rows)',     y_test, preds['LR_all'], preds['LR_4var']),
    ('LR_all - LR_noID (30,000 rows)',     y_test, preds['LR_all'], preds['LR_noID']),
    ('RF_all - RF_noID (30,000 rows)',     y_test, preds['RF_all'], preds['RF_noID']),
    ('BN - LR_4var (29,998 shared rows)',  y_test[v], preds['BN'][v], preds['LR_4var'][v]),
]
rows_b = []
for label, y, pa, pb in comparisons:
    t0 = time.time()
    obs, lo, hi, p, nb = paired_bootstrap(y, pa, pb)
    rows_b.append({'comparison': label, 'AUC diff': obs, 'CI low': lo, 'CI high': hi, 'p': p, 'n_boot': nb})
    print(f"  done: {label} ({time.time()-t0:.0f}s)")
print("\n" + pd.DataFrame(rows_b).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Cell 68: RQ2 finish -- (1) delta <= S_exp check, (2) 12-variable table with bootstrap CIs, (4) coverage checks.
# Read-only on the locked model. Reads saved per-row results; writes rq2_sensitivity_table.csv
# and rq2_pattern_level_check.csv.
import warnings

short = lambda n: n.replace(' (bin)', '')
pd.set_option('display.width', 250)
if 'res_orig' not in globals():
    res_orig = pd.read_pickle('vo_blanket_original_per_row.pkl')
if 'rq2_df' not in globals():
    rq2_df = pd.read_pickle('rq2_sensitivity_per_row.pkl')

BLANKET = sorted(res_orig.variable.unique())
assert len(BLANKET) == 4, BLANKET
FEATURE_BINS = [
    'Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)',
    'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
    '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)',
    'Monthly Income (bin)', 'Number of Dependents (bin)',
]
VARS12 = FEATURE_BINS + ['M_Income', 'M_Dependents']

# ---------- 1. delta <= S_exp + 1e-12, row by row, 4 blanket variables ----------
rows = []
for var, g in res_orig.groupby('variable', sort=False):
    both = g[['S_exp', 'delta']].dropna()
    gap = both['S_exp'] - both['delta']                 # expected >= 0
    rows.append({'variable': short(var), 'rows': len(g),
                 'NaN S_exp': int(g.S_exp.isna().sum()), 'NaN delta': int(g.delta.isna().sum()),
                 'compared': len(both),
                 'max violation (delta - S_exp)': float((-gap).max()),
                 'rows violating (>1e-12)': int((gap < -1e-12).sum()),
                 'rows equal (<=1e-12)': int((gap.abs() <= 1e-12).sum()),
                 'share equal': float((gap.abs() <= 1e-12).mean()),
                 'mean gap (S_exp - delta)': float(gap.mean())})
chk = pd.DataFrame(rows)
print("=== 1. Row-level check: delta <= S_exp + 1e-12 ===")
print(chk.to_string(index=False, formatters={
    'max violation (delta - S_exp)': lambda x: f"{x:.3e}",
    'share equal': lambda x: f"{x:.4f}",
    'mean gap (S_exp - delta)': lambda x: f"{x:.5f}"}))
worst = chk['max violation (delta - S_exp)'].max()
print(f"Worst violation over the four variables: {worst:.3e} (negative = never exceeded)")
assert worst <= 1e-12, "delta exceeds S_exp -- stop and report"

# ---------- 2. 12-variable table ----------
analysis_rows = pd.Index(res_orig.row_index.unique())
n = len(analysis_rows)
print(f"\nAnalysis rows: {n} (expect 29998)")
S_exp_w = pd.DataFrame(np.nan, index=analysis_rows, columns=VARS12)
S_max_w = pd.DataFrame(np.nan, index=analysis_rows, columns=VARS12)
for var in BLANKET:
    g = res_orig[res_orig.variable == var].set_index('row_index')
    S_exp_w.loc[g.index, var] = g['S_exp']
    S_max_w.loc[g.index, var] = g['S_max']

nbv = [v for v in VARS12 if v not in BLANKET]
nb = rq2_df[rq2_df.variable.isin(nbv)]
print(f"Non-blanket: max S_max over all rows and variables = {nb['S'].max():.3e}; NaN count = {int(nb['S'].isna().sum())}")
assert nb['S'].max() < 1e-12 and nb['S'].notna().all()
for var in nbv:
    g = nb[nb.variable == var].set_index('row_index')
    S_max_w.loc[g.index, var] = g['S']
    S_exp_w.loc[g.index, var] = 0.0          # S_exp <= S_max = 0 on every applicable row

rng = np.random.RandomState(42)
B = 2000
Xe, Xm = S_exp_w.values, S_max_w.values
boot_e = np.empty((B, len(VARS12))); boot_m = np.empty((B, len(VARS12)))
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    for i in range(B):
        ix = rng.randint(0, n, n)                 # resample borrowers (rows), shared across variables
        boot_e[i] = np.nanmean(Xe[ix], axis=0)
        boot_m[i] = np.nanmean(Xm[ix], axis=0)
assert np.isfinite(boot_e).all() and np.isfinite(boot_m).all()
print(f"Bootstrap done in {time.time()-t0:.0f}s ({B} row resamples, seed 42)")

def stat_row(j, var, X, boot, measure):
    x = X[:, j]; x = x[~np.isnan(x)]
    lo, hi = np.percentile(boot[:, j], [2.5, 97.5])
    return {'variable': var, 'in_blanket': var in BLANKET, 'measure': measure, 'n_rows': len(x),
            'mean': x.mean(), 'ci_low': lo, 'ci_high': hi, 'median': np.median(x),
            'p90': np.percentile(x, 90), 'p95': np.percentile(x, 95), 'p99': np.percentile(x, 99),
            'share_gt_0.01': (x > 0.01).mean(), 'share_gt_0.05': (x > 0.05).mean()}

recs = []
for j, var in enumerate(VARS12):
    recs.append(stat_row(j, var, Xe, boot_e, 'S_exp'))
    recs.append(stat_row(j, var, Xm, boot_m, 'S_max'))
tbl = pd.DataFrame(recs)
order = (tbl[tbl.measure == 'S_exp'].sort_values(['mean', 'variable'], ascending=[False, True])
         .variable.tolist())
tbl['variable'] = pd.Categorical(tbl.variable, categories=order, ordered=True)
tbl = tbl.sort_values(['measure', 'variable'], ascending=[True, True]).reset_index(drop=True)
tbl.assign(variable=tbl.variable.astype(str)).to_csv('rq2_sensitivity_table.csv', index=False)
print("Saved:", os.path.abspath('rq2_sensitivity_table.csv'))

show_cols = ['variable', 'n_rows', 'mean', 'ci_low', 'ci_high', 'median', 'p90', 'p95', 'p99',
             'share_gt_0.01', 'share_gt_0.05']
for meas in ['S_exp', 'S_max']:
    print(f"\n=== 2. {meas}: all 12 variables (sorted by S_exp mean) ===")
    print(tbl[tbl.measure == meas][show_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---------- 4. coverage, complete-case comparison, pattern-level check ----------
print("\n=== 4. Applicable rows per variable ===")
print(tbl[tbl.measure == 'S_exp'][['variable', 'n_rows']].to_string(index=False))

complete = ((df_test.loc[analysis_rows, 'M_Income'] == 1)
            & (df_test.loc[analysis_rows, 'M_Dependents'] == 1)).values
print(f"\nComplete-evidence rows within the analysis set: {complete.sum()} (expect 24015)")
cc = []
for var in BLANKET:
    j = VARS12.index(var)
    cc.append({'variable': short(var),
               'S_exp mean (all 29,998)': np.nanmean(Xe[:, j]), 'S_exp mean (complete)': np.nanmean(Xe[complete, j]),
               'S_max mean (all 29,998)': np.nanmean(Xm[:, j]), 'S_max mean (complete)': np.nanmean(Xm[complete, j])})
print(pd.DataFrame(cc).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

pat = (df_test.loc[analysis_rows, BLANKET].astype(str)
       .apply(lambda r: '|'.join(r), axis=1).values)
n_pat = len(set(pat))
base = (res_orig[res_orig.variable == BLANKET[0]].set_index('row_index')['baseline_prob']
        .reindex(analysis_rows).values)
d0 = pd.DataFrame({'pat': pat, 'base': base}).groupby('pat')['base'].agg(lambda s: s.max() - s.min())
print(f"\nDistinct blanket patterns among the {n} analysis rows: {n_pat} (207 was counted over all 30,000 test rows)")
print(f"Max within-pattern spread of the baseline P(Default=1 | e), all rows incl. incomplete: {d0.max():.3e}")

prow = []
for var in BLANKET:
    j = VARS12.index(var)
    d = pd.DataFrame({'pat': pat, 'e': Xe[:, j], 'm': Xm[:, j]})
    ge = d.groupby('pat')['e'].agg(lambda s: s.max() - s.min())
    gm = d.groupby('pat')['m'].agg(lambda s: s.max() - s.min())
    prow.append({'variable': var, 'n_patterns': n_pat,
                 'S_max_within_pattern_spread': gm.max(), 'S_exp_within_pattern_spread': ge.max(),
                 'distinct_S_exp_values': int(pd.Series(Xe[:, j]).round(10).nunique()),
                 'S_exp_pattern_level': bool(ge.max() < 1e-12)})
prow = pd.DataFrame(prow)
prow.to_csv('rq2_pattern_level_check.csv', index=False)
print("\n=== Is the S measure a function of the blanket pattern alone? ===")
print(prow.assign(variable=prow.variable.map(short)).to_string(
    index=False, formatters={'S_max_within_pattern_spread': lambda x: f"{x:.3e}",
                             'S_exp_within_pattern_spread': lambda x: f"{x:.3e}"}))

In [ ]:
# Cell 69 (revised): RQ2 figures, ranking chart without error bars. Replaces the previous Cell 69. Read-only; no numbers change.
import textwrap
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator

short = lambda n: n.replace(' (bin)', '')
tbl = pd.read_csv('rq2_sensitivity_table.csv')
pl = pd.read_csv('rq2_pattern_level_check.csv')

# ---------- 3a. ranking ----------
te = tbl[tbl.measure == 'S_exp'].copy()
blanket = te[te.in_blanket].sort_values('mean', ascending=False)
zero_order = ['Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', 'Real Estate Loans (bin)',
              'Monthly Income (bin)', 'M_Income', 'Number of Dependents (bin)', 'M_Dependents']
assert len(blanket) == 4 and set(zero_order) == set(te[~te.in_blanket].variable)
zeros = te.set_index('variable').loc[zero_order].reset_index()
te = pd.concat([blanket, zeros], ignore_index=True)
is_b = te['in_blanket'].values.astype(bool)

y = np.arange(len(te))
means = te['mean'].values
xmax = float(means.max())

fig, ax = plt.subplots(figsize=(10.5, 6.5))
ax.barh(y[is_b], means[is_b], color='tab:blue', height=0.6)
for yi, m in zip(y[is_b], means[is_b]):
    ax.text(m + 0.015 * xmax, yi, f"{m:.4f}", va='center', ha='left', fontsize=9)
ax.plot(np.zeros((~is_b).sum()), y[~is_b], linestyle='none', marker='o', markersize=6,
        markerfacecolor='white', markeredgecolor='dimgray', clip_on=False, zorder=5)
ax.set_yticks(y)
ax.set_yticklabels([f"{short(v)}  (n={int(k):,})" for v, k in zip(te['variable'], te['n_rows'])])
ax.invert_yaxis()
y0, y1 = y[~is_b].min() - 0.4, y[~is_b].max() + 0.4
xb = 0.07 * xmax
ax.plot([xb * 0.6, xb, xb, xb * 0.6], [y0, y0, y1, y1], color='dimgray', lw=1.2)
ax.text(xb * 1.6, (y0 + y1) / 2,
        "outside Default's Markov blanket:\nstructural zero (S_exp = 0 on every row)",
        va='center', ha='left', color='dimgray', fontsize=10)
ax.set_xlim(0, xmax * 1.22)
ax.set_xlabel('Mean S_exp (probability-weighted shift in P(Default=1))')
ax.set_title("RQ2: one-way evidence sensitivity of the BN's P(Default=1)\n"
             "all 29,998 test rows, locked (unsmoothed) model")
ax.grid(False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.text(0.01, 0.005, textwrap.fill(
    "S_exp = sum over x of P(X_j = x | e without X_j) * |P(Default=1 | e, X_j = x) - P(Default=1 | e)|. "
    "Bootstrap 95% confidence intervals for the means are in rq2_sensitivity_table.csv. "
    "n = rows on which the variable is observed.", 140), fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.savefig('rq2_sensitivity_ranking.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", os.path.abspath('rq2_sensitivity_ranking.png'))

# ---------- 3b. distributions ----------
res_o = pd.read_pickle('vo_blanket_original_per_row.pkl')
med = res_o.groupby('variable')['S_exp'].median()
order = med.sort_values(ascending=False).index.tolist()
data = [res_o.loc[res_o.variable == v, 'S_exp'].dropna().values for v in order]
lvl = pl.set_index('variable')['S_exp_pattern_level']
n_pat = int(pl['n_patterns'].iloc[0])

fig, ax = plt.subplots(figsize=(10.5, 6.5))
bp = ax.boxplot(data, whis=(5, 95), showfliers=False, patch_artist=True, widths=0.55,
                medianprops=dict(color='black', lw=1.5))
for box in bp['boxes']:
    box.set_facecolor('lightsteelblue')
xs = np.arange(1, len(data) + 1)
ax.scatter(xs, [d.mean() for d in data], marker='D', color='tab:red', zorder=3, label='mean')
ax.scatter(xs, [np.percentile(d, 99) for d in data], marker='x', color='black', zorder=3,
           label='99th percentile')
ax.set_yscale('symlog', linthresh=1e-3)
ax.set_ylim(0, 1)
ax.set_yticks([0, 0.001, 0.01, 0.1, 1])
ax.set_yticklabels(['0', '0.001', '0.01', '0.1', '1'])
ax.yaxis.set_minor_locator(NullLocator())
ax.set_xticks(xs)
ax.set_xticklabels([f"{short(v)}\nmedian {m:.4f}\n({'pattern-level' if lvl[v] else 'row-level'})"
                    for v, m in zip(order, med[order])])
ax.set_ylabel('S_exp (symlog axis; linear below 0.001)', labelpad=10)
ax.legend(loc='upper left', frameon=False)
ax.set_title("Distribution of S_exp across test rows: Default's Markov-blanket variables")
ax.grid(False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

maxes = ', '.join(f"{short(v)} {d.max():.3f}" for v, d in zip(order, data))
cap = ("Box = interquartile range, line = median, whiskers = 5th to 95th percentile (outliers not drawn), "
       "diamond = mean, x = 99th percentile. "
       f"Maximum S_exp (not drawn): {maxes}. "
       f"{n_pat} distinct blanket patterns in the 29,998 rows. "
       "Pattern-level: every row sharing a blanket pattern has the same S_exp. "
       "Row-level: S_exp also depends on evidence outside the blanket.")
fig.text(0.01, 0.005, textwrap.fill(cap, 140), fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.12, 1, 1))
plt.savefig('rq2_sensitivity_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved:", os.path.abspath('rq2_sensitivity_distributions.png'))

In [ ]:
# Cell 69b: why does 30-59's lower whisker sit on the box edge? (read-only)
res_o = pd.read_pickle('vo_blanket_original_per_row.pkl')
BL = sorted(res_o.variable.unique())
g = res_o[res_o.variable == '30-59 Days Past Due (bin)'][['row_index', 'S_exp']].copy()
g['S_r'] = g.S_exp.round(8)
print("S_exp percentiles (30-59):", g.S_exp.quantile([0.05, 0.10, 0.20, 0.25]).round(5).to_dict())
top = g.groupby('S_r').size().sort_values(ascending=False).head(5)
print("\nMost common S_exp values (value -> rows, share):")
for v, n in top.items():
    print(f"  {v:.5f} -> {n} rows ({n/len(g)*100:.1f}%)")
rows0 = g.loc[g.S_r == top.index[0], 'row_index']
print("\nBlanket patterns among the rows at the most common value:")
print(df_test.loc[rows0, BL].astype(str).value_counts().head(5))

In [ ]:
# Cell 70: RQ2 extension Part A -- Step 0 (from scratch), Step 1 (table), Step 5 (sentences)
# Read-only on the locked model. Reads vo_blanket_original_per_row.pkl. Writes rq2_ext_partA_table.csv.
import copy, pickle, os, time
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

TAU_BN = 0.17
short = lambda n: n.replace(' (bin)', '')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
FEATURE_BINS = [
    'Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)',
    'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)',
    '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)',
    'Monthly Income (bin)', 'Number of Dependents (bin)',
]

res_orig = pd.read_pickle('vo_blanket_original_per_row.pkl')
BLANKET = sorted(res_orig.variable.unique())
assert len(BLANKET) == 4, BLANKET
analysis_rows = pd.Index(res_orig.row_index.unique())
assert len(analysis_rows) == 29998 and res_orig.groupby('variable').size().eq(29998).all()
n = len(analysis_rows)

try:
    _t = pd.read_csv('rq2_sensitivity_table.csv')
    _t = _t[(_t.measure == 'S_exp') & (_t['in_blanket'].astype(str).str.lower() == 'true')]
    ORDER = _t.sort_values('mean', ascending=False).variable.tolist()
except FileNotFoundError:
    ORDER = res_orig.groupby('variable')['S_exp'].mean().sort_values(ascending=False).index.tolist()
assert sorted(ORDER) == BLANKET
print("Variable order (RQ2 S_exp mean):", [short(v) for v in ORDER])

# ===================== Step 0: reproducibility from scratch =====================
with open('bn_final_model_iteration79.pkl', 'rb') as f:
    _sm = pickle.load(f)
bn0 = DiscreteBayesianNetwork(_sm['model_edges'])
bn0.add_cpds(*copy.deepcopy(_sm['cpds']))
assert bn0.check_model()
eng0 = VariableElimination(bn0)            # new engine, no shared cache

def evidence_row(idx):
    r = df_test.loc[idx]
    ev = {c: r[c] for c in FEATURE_BINS if pd.notnull(r[c])}
    ev['M_Income'] = int(pd.notnull(r['Monthly Income (bin)']))
    ev['M_Dependents'] = int(pd.notnull(r['Number of Dependents (bin)']))
    return ev

def p_default0(ev):
    with np.errstate(invalid='ignore', divide='ignore'):
        try:
            p = eng0.query(variables=['Default'], evidence=ev, show_progress=False).get_value(Default=1)
        except Exception:
            p = np.nan
    return float(p) if np.isfinite(p) and 0 <= p <= 1 else np.nan

keys = list(zip(res_orig.row_index, res_orig.variable))
saved_delta = dict(zip(keys, res_orig.delta)); saved_flip = dict(zip(keys, res_orig.flip))
saved_base = dict(zip(keys, res_orig.baseline_prob))

rng = np.random.RandomState(42)
sample = rng.choice(analysis_rows.values, size=200, replace=False)
max_dd = max_db = 0.0
n_cmp = n_nan = flip_mismatch = 0
for idx in sample:
    ev = evidence_row(idx)
    base = p_default0(ev)
    for var in BLANKET:
        p_r = p_default0({k: v for k, v in ev.items() if k != var})
        if np.isnan(p_r) or np.isnan(base):
            n_nan += 1
            continue
        flip = int((base >= TAU_BN) != (p_r >= TAU_BN))
        max_dd = max(max_dd, abs(abs(p_r - base) - saved_delta[(idx, var)]))
        max_db = max(max_db, abs(base - saved_base[(idx, var)]))
        flip_mismatch += int(flip != saved_flip[(idx, var)])
        n_cmp += 1
print("\n=== Step 0: recompute from scratch (fresh model load, fresh engine, no cache) ===")
print(f"200 random rows (seed 42) x 4 variables: {n_cmp} comparisons, {n_nan} NaN")
print(f"Max abs diff, Delta vs saved: {max_dd:.3e}; baseline vs saved: {max_db:.3e}; flip mismatches: {flip_mismatch}")
assert n_nan == 0 and max_dd < 1e-12 and max_db < 1e-12 and flip_mismatch == 0

exp_mean = {'Credit Utilisation Rate (bin)': 0.0234, '30-59 Days Past Due (bin)': 0.0226,
            '90+ Days Late (bin)': 0.0214, '60-89 Days Past Due (bin)': 0.0106}
exp_flip = {'Credit Utilisation Rate (bin)': 0.0458, '30-59 Days Past Due (bin)': 0.0463,
            '90+ Days Late (bin)': 0.0309, '60-89 Days Past Due (bin)': 0.0140}
print("Full-set means vs the earlier run:")
for var in ORDER:
    g = res_orig[res_orig.variable == var]
    m, fr = g.delta.mean(), g.flip.mean()
    ok_m = abs(round(m, 4) - exp_mean[var]) < 1e-9
    ok_f = abs(round(fr, 4) - exp_flip[var]) < 1e-9
    print(f"  {short(var):26s} mean Delta {m:.4f} (earlier {exp_mean[var]:.4f}) {'OK' if ok_m else 'MISMATCH'} | "
          f"flip {fr*100:.2f}% (earlier {exp_flip[var]*100:.2f}%) {'OK' if ok_f else 'MISMATCH'}")
    assert ok_m and ok_f

# ===================== Step 1: Part A table =====================
D = pd.DataFrame(np.nan, index=analysis_rows, columns=ORDER)
F = pd.DataFrame(np.nan, index=analysis_rows, columns=ORDER)
for var in ORDER:
    g = res_orig[res_orig.variable == var].set_index('row_index')
    D[var] = g['delta'].reindex(analysis_rows).values
    F[var] = g['flip'].reindex(analysis_rows).values
assert D.notna().all().all() and F.notna().all().all()
BASE_all = res_orig.pivot(index='row_index', columns='variable', values='baseline_prob').reindex(analysis_rows)
assert (BASE_all.max(axis=1) - BASE_all.min(axis=1)).max() < 1e-12
BASE = BASE_all[ORDER[0]]

known_reject = BASE.values >= TAU_BN
Fi = F.values.astype(int)
to_reject = Fi * known_reject[:, None]       # known = reject, without = approve
to_approve = Fi * (~known_reject)[:, None]   # known = approve, without = reject
assert (to_reject + to_approve == Fi).all()

pat = (df_test.loc[analysis_rows, BLANKET].astype(str)
       .apply(lambda r: '|'.join(r), axis=1).values)
n_pat = len(set(pat))

rng = np.random.RandomState(42)
B = 2000
Dv, Fv = D.values, F.values
boot_D = np.empty((B, 4)); boot_F = np.empty((B, 4))
for i in range(B):
    ix = rng.randint(0, n, n)
    boot_D[i] = Dv[ix].mean(axis=0)
    boot_F[i] = Fv[ix].mean(axis=0)

rows = []
for j, var in enumerate(ORDER):
    x = Dv[:, j]
    gp = pd.DataFrame({'pat': pat, 'd': x}).groupby('pat')['d']
    spread = float((gp.max() - gp.min()).max())
    lo, hi = np.percentile(boot_D[:, j], [2.5, 97.5])
    flo, fhi = np.percentile(boot_F[:, j], [2.5, 97.5])
    nf, nr, na = int(Fi[:, j].sum()), int(to_reject[:, j].sum()), int(to_approve[:, j].sum())
    rows.append({
        'variable': var, 'level': 'pattern-level' if spread < 1e-12 else 'row-level', 'n_rows': n,
        'mean_delta': x.mean(), 'ci_low': lo, 'ci_high': hi, 'median': np.median(x),
        'p90': np.percentile(x, 90), 'p95': np.percentile(x, 95), 'p99': np.percentile(x, 99), 'max': x.max(),
        'share_gt_0.01': (x > 0.01).mean(), 'share_gt_0.05': (x > 0.05).mean(),
        'n_flips': nf, 'flip_rate': nf / n, 'flip_ci_low': flo, 'flip_ci_high': fhi,
        'n_flip_to_reject': nr, 'n_flip_to_approve': na, 'share_flips_to_reject': nr / nf if nf else np.nan,
        'n_patterns': n_pat, 'delta_within_pattern_spread': spread,
        'distinct_delta_values': int(pd.Series(x).round(10).nunique())})
tblA = pd.DataFrame(rows)
tblA.to_csv('rq2_ext_partA_table.csv', index=False)
print("\nSaved:", os.path.abspath('rq2_ext_partA_table.csv'))

fmt = lambda v: f"{v:.4f}"
t = tblA.assign(variable=tblA.variable.map(short))
print(f"\n=== Part A: Delta = |P(Default=1|e) - P(Default=1|e without X_j)|, tau_BN = {TAU_BN}, {n} rows ===")
print(t[['variable', 'level', 'n_rows', 'mean_delta', 'ci_low', 'ci_high', 'median', 'p90', 'p95', 'p99',
         'max', 'share_gt_0.01', 'share_gt_0.05']].to_string(index=False, float_format=fmt))
print("\n=== Part A: decision flips at tau = 0.17 (direction of the effect of KNOWING the variable) ===")
print(t[['variable', 'n_flips', 'flip_rate', 'flip_ci_low', 'flip_ci_high', 'n_flip_to_reject',
         'n_flip_to_approve', 'share_flips_to_reject']].to_string(index=False, float_format=fmt))
print(f"\n=== Is Delta a function of the blanket pattern alone? ({n_pat} distinct patterns in {n} rows) ===")
print(t[['variable', 'level', 'delta_within_pattern_spread', 'distinct_delta_values']].to_string(
    index=False, formatters={'delta_within_pattern_spread': lambda v: f"{v:.3e}"}))

print("\n=== Extra (not requested): paired bootstrap of adjacent differences, same 2,000 resamples ===")
for j in range(3):
    a, b = ORDER[j], ORDER[j + 1]
    dd, ff = boot_D[:, j] - boot_D[:, j + 1], boot_F[:, j] - boot_F[:, j + 1]
    print(f"  {short(a)} minus {short(b)}: mean Delta {dd.mean():+.4f} [{np.percentile(dd, 2.5):+.4f}, "
          f"{np.percentile(dd, 97.5):+.4f}]; flip rate {ff.mean()*100:+.2f} pp "
          f"[{np.percentile(ff, 2.5)*100:+.2f}, {np.percentile(ff, 97.5)*100:+.2f}]")

# ===================== Step 5: draft descriptive sentences (numbers copied from the table) =====================
print("\n=== Step 5: draft sentences ===")
for _, r in tblA.iterrows():
    print(f"- {short(r['variable'])}: Retracting it changes the BN's P(Default=1) by a median of {r['median']:.4f} "
          f"(mean {r['mean_delta']:.4f}, 95% CI [{r['ci_low']:.4f}, {r['ci_high']:.4f}]; p90 {r['p90']:.4f}, "
          f"p95 {r['p95']:.4f}, p99 {r['p99']:.4f}, max {r['max']:.4f}); {r['share_gt_0.01']*100:.1f}% of the "
          f"{int(r['n_rows']):,} test rows shift by more than 0.01 and {r['share_gt_0.05']*100:.1f}% by more than "
          f"0.05, and the decision at tau = 0.17 flips on {r['flip_rate']*100:.2f}% of rows "
          f"(95% CI [{r['flip_ci_low']*100:.2f}%, {r['flip_ci_high']*100:.2f}%]; {int(r['n_flip_to_reject']):,} "
          f"toward reject, {int(r['n_flip_to_approve']):,} toward approve). Delta is {r['level']}.")

In [ ]:
# Cell 71: Part B -- naturally incomplete rows (read-only). Needs bn0, eng0, p_default0, evidence_row, BASE, ORDER from Cell 70.
VALUE_OF = [('M_Income', 'Monthly Income (bin)', 'Income-missing rows'),
            ('M_Dependents', 'Number of Dependents (bin)', 'Dependents-missing rows')]
cacheB, wcache = {}, {}

def post_B(ev):
    key = tuple(sorted(ev.items(), key=lambda kv: kv[0]))
    if key not in cacheB:
        cacheB[key] = p_default0(ev)
    return cacheB[key]

def weights(var, ev):
    """P(X_j = x | e) with the M indicator set to 0 and the value unobserved."""
    key = (var, tuple(sorted(ev.items(), key=lambda kv: kv[0])))
    if key not in wcache:
        with np.errstate(invalid='ignore', divide='ignore'):
            f = eng0.query(variables=[var], evidence=ev, show_progress=False)
        vals = np.asarray(f.values, dtype=float)
        assert np.isfinite(vals).all()
        wcache[key] = dict(zip(f.state_names[var], vals))
    return wcache[key]

t0 = time.time()
out, row_sets = [], {}
for m_var, val_var, label in VALUE_OF:
    rows_b = [i for i in analysis_rows if pd.isnull(df_test.at[i, val_var])]
    row_sets[label] = set(rows_b)
    max_shift = eps_sum = evoi_sum = 0.0
    flips = skipped = evaluated = 0
    for i in rows_b:
        ev = evidence_row(i)
        base = BASE[i]
        assert ev[m_var] == 0 and val_var not in ev
        assert abs(post_B(ev) - base) < 1e-12
        for s, w in weights(val_var, ev).items():
            if w <= 0:
                continue
            p2 = post_B({**ev, m_var: 1, val_var: s})    # observe the missing value: M = 1 and value = s
            if np.isnan(p2):
                skipped += 1
                continue
            shift = abs(p2 - base)
            flip = int((base >= TAU_BN) != (p2 >= TAU_BN))
            max_shift = max(max_shift, shift)
            eps_sum += w * shift
            evoi_sum += w * flip
            flips += flip
            evaluated += 1
    out.append({'group': label, 'n_rows': len(rows_b), 'states_evaluated': evaluated, 'max_shift': max_shift,
                'mean_EPS': eps_sum / len(rows_b), 'mean_EVOI': evoi_sum / len(rows_b),
                'decision_flips': flips, 'skipped_states': skipped})
print(f"Part B done in {time.time()-t0:.0f}s")
tblB = pd.DataFrame(out)
print("\n=== Part B (blanket variables observed on every row; value of observing the missing variable) ===")
print(tblB.to_string(index=False, formatters={'max_shift': lambda v: f"{v:.3e}",
      'mean_EPS': lambda v: f"{v:.3e}", 'mean_EVOI': lambda v: f"{v:.3e}"}))
assert (tblB.n_rows.values == [5983, 777]).all()
assert (tblB.max_shift < 1e-12).all() and (tblB.mean_EPS < 1e-12).all() and (tblB.mean_EVOI < 1e-12).all()
assert (tblB.decision_flips == 0).all() and (tblB.skipped_states == 0).all()
assert row_sets['Dependents-missing rows'] <= row_sets['Income-missing rows']
print("Assertions passed: all values are exactly zero.")
print("All 777 Dependents-missing rows are also Income-missing, so the two rows are nested, not independent samples, "
      "and are not ranked against each other.")
tblB.to_csv('rq2_ext_partB_table.csv', index=False)

In [ ]:
# Cell 72: robustness footnote (read-only; reads the per-row pickles saved by Cell 66)
ref = res_orig
for ess in (1, 10):
    r = pd.read_pickle(f'vo_blanket_ess{ess}_per_row.pkl')
    m = ref.merge(r, on=['row_index', 'variable'], suffixes=('_o', '_s'))
    assert len(m) == len(ref) == len(r)
    mo = ref.groupby('variable')[['delta', 'flip']].mean()
    ms = r.groupby('variable')[['delta', 'flip']].mean()
    rank = lambda s: list(s.sort_values(ascending=False).index)
    print(f"ESS={ess}: mean Delta max abs diff {np.abs(mo.delta - ms.delta).max():.2e}, "
          f"Delta ranking identical: {rank(mo.delta) == rank(ms.delta)}; "
          f"flip rate max abs diff {np.abs(mo.flip - ms.flip).max():.2e}, "
          f"flip ranking identical: {rank(mo.flip) == rank(ms.flip)}; "
          f"flip decision changed on {int((m.flip_o != m.flip_s).sum())} of {len(m):,} row-variable records; "
          f"largest single-row Delta change {np.abs(m.delta_o - m.delta_s).max():.4f}")

In [ ]:
# Cell 73 (revised): RQ2 extension figures -- layout fixes only. Replaces the previous Cell 73.
# Read-only; no numbers change. Needs Cell 70's ORDER, D, F, BASE, pat, n, TAU_BN, short.
import textwrap
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullLocator

tblA = pd.read_csv('rq2_ext_partA_table.csv').set_index('variable').loc[ORDER]
level = tblA['level'].to_dict()
SYM_TICKS, SYM_LABELS = [0, 0.001, 0.01, 0.1, 1], ['0', '0.001', '0.01', '0.1', '1']
PAL = dict(zip(ORDER, ['#0072B2', '#E69F00', '#009E73', '#CC79A8']))
REJ, APP = '#D55E00', '#56B4E9'

def clean(ax):
    ax.grid(False)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ---------- 3a. ECDF of Delta ----------
fig, ax = plt.subplots(figsize=(10.5, 6.8))
for var in ORDER:
    x = np.sort(D[var].values)
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), drawstyle='steps-post', color=PAL[var], lw=1.8,
            label=f"{short(var)} ({level[var]})")
ax.set_xscale('symlog', linthresh=1e-3)
ax.set_xlim(0, 1.0); ax.set_ylim(0, 1.0)
ax.set_xticks(SYM_TICKS); ax.set_xticklabels(SYM_LABELS); ax.xaxis.set_minor_locator(NullLocator())
for xv in (0.01, 0.05):
    ax.axvline(xv, color='gray', lw=0.9, ls=':')
    ax.annotate(f"Delta = {xv}", xy=(xv, 1.0), xycoords=('data', 'axes fraction'),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom',
                fontsize=9, color='dimgray')
ax.set_xlabel('Delta = |P(Default=1 | e) - P(Default=1 | e without X_j)|   (symlog axis; linear below 0.001)')
ax.set_ylabel('Cumulative share of test rows', labelpad=10)
ax.set_title("RQ2 extension, Part A: effect of retracting each blanket variable on the BN's P(Default=1)", pad=26)
ax.legend(loc='upper left', frameon=False, fontsize=9.5)
clean(ax)
cap = ("ECDF of Delta over all 29,998 test rows. Pattern-level: every row sharing a blanket pattern has the same Delta "
       "(hence the steps). Row-level: Delta also depends on evidence outside the blanket. "
       f"Locked (unsmoothed) model; {pd.Series(pat).nunique()} distinct blanket patterns.")
fig.text(0.01, 0.005, textwrap.fill(cap, 140), fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig('rq2_ext_delta_distributions.png', dpi=300, bbox_inches='tight'); plt.show()
print("Saved:", os.path.abspath('rq2_ext_delta_distributions.png'))

# ---------- 3b. flip rates, stacked by direction ----------
fig, ax = plt.subplots(figsize=(11, 6.5))
xs = np.arange(4)
rej = tblA['n_flip_to_reject'].values / n * 100
app = tblA['n_flip_to_approve'].values / n * 100
tot = tblA['flip_rate'].values * 100
lo, hi = tblA['flip_ci_low'].values * 100, tblA['flip_ci_high'].values * 100
ax.bar(xs, rej, color=REJ, width=0.6, label='Knowing it pushes to reject (known: reject, without: approve)')
ax.bar(xs, app, bottom=rej, color=APP, width=0.6, label='Knowing it pushes to approve (known: approve, without: reject)')
ax.errorbar(xs + 0.40, tot, yerr=[tot - lo, hi - tot], fmt='o', color='black', markersize=4, capsize=4, lw=1.2,
            label='Total flip rate with bootstrap 95% CI')
for x_, r_, a_, t_ in zip(xs, rej, app, tot):
    ax.text(x_, t_ + 0.1, f"{t_:.2f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.text(x_, r_ / 2, f"{r_:.2f}%", ha='center', va='center', fontsize=9, color='white')
    if a_ >= 0.8:
        ax.text(x_, r_ + a_ / 2, f"{a_:.2f}%", ha='center', va='center', fontsize=9)
    else:   # thin segment: label beside the bar instead of inside it
        ax.text(x_ - 0.33, r_ + a_ / 2, f"{a_:.2f}%", ha='right', va='center', fontsize=9, color='#2b7bb9')
ax.set_xlim(-0.6, 3.75)
ax.set_xticks(xs)
ax.set_xticklabels([f"{short(v)}\n{int(tblA.loc[v, 'n_flips']):,} flips of {n:,}\n({level[v]})" for v in ORDER])
ax.set_ylabel('Decision flips at tau = 0.17 (% of test rows)', labelpad=10)
ax.set_ylim(0, max(hi) * 1.22)
ax.set_title('RQ2 extension, Part A: how often retracting a variable changes the lending decision')
ax.legend(loc='upper right', frameon=False, fontsize=9)
clean(ax)
fig.text(0.01, 0.005, textwrap.fill(
    "Decision = reject if P(Default=1) >= 0.17 (tau_BN, 5:1 FN:FP cost ratio). Error bars (offset to the right of each bar): "
    "bootstrap 95% CI of the total flip rate (2,000 row resamples, seed 42; test rows resampled, model fixed).", 140),
    fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig('rq2_ext_flip_rates.png', dpi=300, bbox_inches='tight'); plt.show()
print("Saved:", os.path.abspath('rq2_ext_flip_rates.png'))

# ---------- 3c. Delta vs baseline, 2x2 ----------
XFLOOR = 1e-4
SZ = lambda k: 8 + 3 * np.sqrt(np.asarray(k, dtype=float))
print(f"\nRows with baseline < {XFLOOR}: {int((BASE.values < XFLOOR).sum())} (drawn at x = {XFLOOR})")
xline = np.logspace(np.log10(XFLOOR * 0.8), 0, 800)
vline = np.abs(xline - TAU_BN)
fig, axes = plt.subplots(2, 2, figsize=(13, 11), sharex=True, sharey=True)
print("Figure 3c per-variable counts (rows): flips | non-flip above V-line (shift away from tau) | non-flip below V-line")
for ax, var in zip(axes.ravel(), ORDER):
    d = pd.DataFrame({'pat': pat, 'base': BASE.values, 'delta': D[var].values, 'flip': F[var].values.astype(int)})
    d['dr'] = d['delta'].round(10)
    g = d.groupby(['pat', 'dr', 'flip'], as_index=False).agg(base=('base', 'first'), delta=('delta', 'first'), n=('delta', 'size'))
    g['x'] = g['base'].clip(lower=XFLOOR)
    g['above'] = g['delta'] >= (g['base'] - TAU_BN).abs() - 1e-12
    assert not ((g['flip'] == 1) & ~g['above']).any(), f"{var}: a flip lies below the V-line"
    fm = g['flip'] == 1
    am = (g['flip'] == 0) & g['above']
    bm = (g['flip'] == 0) & ~g['above']
    ax.plot(xline, vline, color='black', lw=1.1, zorder=1)
    ax.axvline(TAU_BN, color='black', lw=0.9, ls='--', zorder=1)
    ax.scatter(g.x[bm], g.delta[bm], s=SZ(g.n[bm].values), facecolors='lightgray', edgecolors='none', alpha=0.55, zorder=2)
    ax.scatter(g.x[am], g.delta[am], s=SZ(g.n[am].values), facecolors='none', edgecolors='#0072B2', linewidth=1.0, zorder=3)
    ax.scatter(g.x[fm], g.delta[fm], s=SZ(g.n[fm].values), facecolors=REJ, edgecolors='black', linewidth=0.3, zorder=4)
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=1e-3)
    ax.set_xlim(XFLOOR * 0.7, 1.4); ax.set_ylim(0, 1.05)
    ax.set_xticks([1e-4, 1e-3, 1e-2, 0.1, 1]); ax.set_xticklabels(['<=0.0001', '0.001', '0.01', '0.1', '1'])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_yticks(SYM_TICKS); ax.set_yticklabels(SYM_LABELS); ax.yaxis.set_minor_locator(NullLocator())
    ax.annotate('tau = 0.17', xy=(TAU_BN, 1.0), xycoords=('data', 'axes fraction'),
                xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=8.5)
    rf_, ra_, rb_ = int(g.n[fm].sum()), int(g.n[am].sum()), int(g.n[bm].sum())
    ax.set_title(f"{short(var)} ({level[var]})\n{rf_:,} flips ({rf_/n*100:.2f}%)", fontsize=11, pad=16)
    clean(ax)
    print(f"  {short(var):26s} {rf_:6,} | {ra_:6,} | {rb_:6,}   (unique markers: {len(g)})")
for ax in axes[1]:
    ax.set_xlabel('Baseline P(Default=1 | e)  (log axis)')
for ax in axes[:, 0]:
    ax.set_ylabel('Delta (symlog; linear below 0.001)')
handles = [
    Line2D([0], [0], marker='o', ls='', markerfacecolor=REJ, markeredgecolor='black', markersize=8, label='Decision flip'),
    Line2D([0], [0], marker='o', ls='', markerfacecolor='none', markeredgecolor='#0072B2', markersize=8,
           label='No flip, Delta >= |x - tau| (shift moved away from tau)'),
    Line2D([0], [0], marker='o', ls='', markerfacecolor='lightgray', markeredgecolor='gray', markersize=8,
           label='No flip, Delta < |x - tau|'),
    Line2D([0], [0], color='black', lw=1.1, label='Delta = |x - 0.17|'),
] + [Line2D([0], [0], marker='o', ls='', markerfacecolor='dimgray', markeredgecolor='dimgray',
            markersize=np.sqrt(SZ(k)), label=f"{k:,} rows") for k in (1, 100, 1000)]
fig.suptitle('RQ2 extension, Part A: why retraction rarely changes the decision', fontsize=14)
fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.045), fontsize=9)
fig.text(0.01, 0.005, textwrap.fill(
    "Each marker is a unique (blanket pattern, Delta, flip) combination; area scales with sqrt(row count). A flip needs Delta >= |x - 0.17|, "
    "but a shift away from the threshold can also exceed that distance without changing the decision. On a log x-axis the line "
    "Delta = |x - 0.17| is nearly flat at about 0.17 for small x and drops to zero at x = 0.17. Baselines below 1e-4 are drawn at 1e-4. "
    "Vertical stripes in row-level panels are rows sharing a baseline (one blanket pattern) with different Delta.", 170),
    fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.11, 1, 0.97))
plt.savefig('rq2_ext_delta_vs_baseline.png', dpi=300, bbox_inches='tight'); plt.show()
print("Saved:", os.path.abspath('rq2_ext_delta_vs_baseline.png'))

# ---------- 3d. heatmap by baseline band ----------
edges = [0, 0.02, 0.05, 0.10, 0.17, 0.30, 0.50, 1.0000001]
bands = ['<0.02', '0.02-0.05', '0.05-0.10', '0.10-0.17', '0.17-0.30', '0.30-0.50', '>=0.50']
band = pd.cut(BASE, bins=edges, labels=bands, right=False, include_lowest=True)
cnt = band.value_counts().reindex(bands)
mean_tbl = pd.DataFrame({short(v): D[v].groupby(band, observed=False).mean() for v in ORDER}).reindex(bands)
flip_tbl = pd.DataFrame({short(v): F[v].groupby(band, observed=False).mean() for v in ORDER}).reindex(bands)
fig, ax = plt.subplots(figsize=(11, 6.5))
M = np.ma.masked_invalid(mean_tbl.values.astype(float))
im = ax.imshow(M, aspect='auto', cmap='Blues', vmin=0)
vmax = np.nanmax(mean_tbl.values)
for i in range(len(bands)):
    for j in range(4):
        v, fr = mean_tbl.values[i, j], flip_tbl.values[i, j]
        txt = '-' if np.isnan(v) else f"{v:.4f}\nflip {fr*100:.1f}%"
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='white' if (not np.isnan(v) and v > 0.6 * vmax) else 'black')
ax.set_xticks(range(4)); ax.set_xticklabels([f"{short(v)}\n({level[v]})" for v in ORDER])
ax.set_yticks(range(len(bands))); ax.set_yticklabels([f"{b}  (n={int(c):,})" for b, c in zip(bands, cnt)])
ax.axhline(3.5, color='black', lw=1.8)
ax.annotate('tau = 0.17', xy=(1.0, 3.5), xycoords=('axes fraction', 'data'),
            xytext=(6, 0), textcoords='offset points', ha='left', va='center', fontsize=9)
ax.set_ylabel('Baseline P(Default=1 | e) band', labelpad=10)
ax.set_title('RQ2 extension, Part A: mean Delta (colour) and flip rate by baseline risk band')
fig.colorbar(im, ax=ax, label='Mean Delta', pad=0.14)
for s in ax.spines.values():
    s.set_visible(False)
fig.text(0.01, 0.005, textwrap.fill(
    "Bands are fixed ranges of the baseline P(Default=1 | e) on all 29,998 test rows. Cell colour = mean Delta; text = mean Delta and flip rate at tau = 0.17. "
    "The blanket variables are observed on every row, so the nested-missingness caveat does not affect this figure.", 150),
    fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.savefig('rq2_ext_heatmap.png', dpi=300, bbox_inches='tight'); plt.show()
print("Saved:", os.path.abspath('rq2_ext_heatmap.png'))

In [ ]:
# Cell 73b: reprint the two counts from the Figure c cell (read-only, no plotting)
# Needs Cell 70's ORDER, D, F, BASE, TAU_BN, short.
XFLOOR = 1e-4
base = BASE.values
low = base < XFLOOR
print(f"Rows with baseline < {XFLOOR}: {int(low.sum())}")

print("\nFigure c counts, in rows: flips | non-flip above V-line (shift moved away from tau) | non-flip below V-line")
for var in ORDER:
    flip = F[var].values.astype(int)
    delta = D[var].values
    above = delta >= np.abs(base - TAU_BN) - 1e-12
    assert (flip[~above] == 0).all(), f"{short(var)}: a flip lies below the V-line"
    n_flip = int(flip.sum())
    n_ab = int(((flip == 0) & above).sum())
    n_be = int(((flip == 0) & ~above).sum())
    assert n_flip + n_ab + n_be == len(base)
    print(f"  {short(var):26s} {n_flip:6,} | {n_ab:6,} | {n_be:6,}   "
          f"(flips among the {int(low.sum())} baseline<1e-4 rows: {int(flip[low].sum())})")

In [ ]:
# Cell 73d: flip-rate figure without error bars. Read-only; no numbers change.
import textwrap
import matplotlib.pyplot as plt

REJ, APP = '#D55E00', '#56B4E9'
tblA = pd.read_csv('rq2_ext_partA_table.csv').set_index('variable').loc[ORDER]
level = tblA['level'].to_dict()

fig, ax = plt.subplots(figsize=(11, 6.5))
xs = np.arange(4)
rej = tblA['n_flip_to_reject'].values / n * 100
app = tblA['n_flip_to_approve'].values / n * 100
tot = tblA['flip_rate'].values * 100

ax.bar(xs, rej, color=REJ, width=0.6, label='Knowing it pushes to reject (known: reject, without: approve)')
ax.bar(xs, app, bottom=rej, color=APP, width=0.6, label='Knowing it pushes to approve (known: approve, without: reject)')
for x_, r_, a_, t_ in zip(xs, rej, app, tot):
    ax.text(x_, t_ + 0.08, f"{t_:.2f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.text(x_, r_ / 2, f"{r_:.2f}%", ha='center', va='center', fontsize=9, color='white')
    if a_ >= 0.8:
        ax.text(x_, r_ + a_ / 2, f"{a_:.2f}%", ha='center', va='center', fontsize=9)
    else:   # thin segment: label beside the bar
        ax.text(x_ - 0.33, r_ + a_ / 2, f"{a_:.2f}%", ha='right', va='center', fontsize=9, color='#2b7bb9')
ax.set_xlim(-0.6, 3.6)
ax.set_xticks(xs)
ax.set_xticklabels([f"{short(v)}\n{int(tblA.loc[v, 'n_flips']):,} flips of {n:,}\n({level[v]})" for v in ORDER])
ax.set_ylabel('Decision flips at tau = 0.17 (% of test rows)', labelpad=10)
ax.set_ylim(0, max(tot) * 1.18)
ax.set_title('RQ2 extension, Part A: how often retracting a variable changes the lending decision')
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.grid(False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
fig.text(0.01, 0.005, textwrap.fill(
    "Decision = reject if P(Default=1) >= 0.17 (tau_BN, 5:1 FN:FP cost ratio). Bootstrap 95% confidence intervals for the total flip "
    "rates are in rq2_ext_partA_table.csv.", 140), fontsize=8, va='bottom')
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.savefig('rq2_ext_flip_rates.png', dpi=300, bbox_inches='tight'); plt.show()
print("Saved:", os.path.abspath('rq2_ext_flip_rates.png'))

In [ ]:
# Cell 74: Phase 0a -- inventory of the project root (read-only; writes nothing)
import os, time, hashlib
from collections import defaultdict

ROOT = os.environ["CREDIT_BN_ROOT"]
SKIP = {'.' + 'venv', '.git', '__pycache__', '.ipynb_checkpoints'}

def sha12(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()[:12]

rows, skipped = [], []
for dp, dns, fns in os.walk(ROOT):
    for d in list(dns):
        if d in SKIP:
            skipped.append(os.path.join(dp, d)); dns.remove(d)
    for fn in fns:
        p = os.path.join(dp, fn)
        try:
            st = os.lstat(p)
            rows.append((os.path.relpath(p, ROOT), st.st_size,
                         time.strftime('%Y-%m-%d %H:%M', time.localtime(st.st_mtime)), sha12(p)))
        except OSError as e:
            rows.append((os.path.relpath(p, ROOT), -1, f"ERROR {type(e).__name__}", ''))
rows.sort()

by_ext = defaultdict(lambda: [0, 0])
for rel, size, _, _ in rows:
    e = os.path.splitext(rel)[1].lower() or '(none)'
    by_ext[e][0] += 1; by_ext[e][1] += max(size, 0)
print(f"Files under {ROOT} (excluding {sorted(SKIP)}): {len(rows)}")
print("\nBy extension:")
for e, (n_, s_) in sorted(by_ext.items(), key=lambda kv: -kv[1][1]):
    print(f"  {e:10s} {n_:4d} files {s_/1e6:9.2f} MB")
print("\nbytes | modified | sha256[:12] | path")
for rel, size, mt, h in rows:
    print(f"{size:>12,}  {mt}  {h:12s}  {rel}")
print("\nExcluded directories (listed, not hashed):")
for d in skipped:
    n_f = n_b = 0
    for dp, _, fns in os.walk(d):
        for fn in fns:
            try:
                n_b += os.lstat(os.path.join(dp, fn)).st_size; n_f += 1
            except OSError:
                pass
    print(f"  {os.path.relpath(d, ROOT)}: {n_f:,} files, {n_b/1e6:,.1f} MB")

In [ ]:
# Cell 75: Phase 0a -- list objects living in the live kernel (READS ONLY; writes nothing)
import sys, types, numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

def approx_size(o):
    try:
        if pd is not None and isinstance(o, (pd.DataFrame, pd.Series)):
            return int(o.memory_usage(deep=True).sum()) if isinstance(o, pd.DataFrame) else int(o.memory_usage(deep=True))
        if isinstance(o, np.ndarray):
            return int(o.nbytes)
        return int(sys.getsizeof(o))
    except Exception:
        return -1

def shape_or_len(o):
    try:
        if hasattr(o, 'shape'):
            return str(tuple(o.shape))
        if hasattr(o, '__len__'):
            return f"len={len(o)}"
    except Exception:
        pass
    return '-'

ns = globals()
skip_types = (types.ModuleType, types.FunctionType, types.BuiltinFunctionType, type)
rows, funcs, mods = [], [], []
for name, obj in list(ns.items()):
    if name.startswith('_') or name in {'In', 'Out', 'get_ipython', 'exit', 'quit'}:
        continue
    if isinstance(obj, types.ModuleType):
        mods.append(name); continue
    if isinstance(obj, (types.FunctionType, types.BuiltinFunctionType)):
        funcs.append(name); continue
    if isinstance(obj, type):
        funcs.append(name + " (class)"); continue
    rows.append((name, type(obj).__name__, shape_or_len(obj), approx_size(obj)))

rows.sort(key=lambda r: -r[3])
print(f"Data/model objects in kernel: {len(rows)}")
print(f"{'name':36s} {'type':28s} {'shape/len':18s} {'~bytes':>14s}")
for n_, t_, s_, b_ in rows:
    print(f"{n_:36s} {t_[:28]:28s} {s_[:18]:18s} {b_:>14,}")

print(f"\nFunctions/classes defined in kernel ({len(funcs)}):")
print("  " + ", ".join(sorted(funcs)))
print(f"\nImported modules ({len(mods)}):")
print("  " + ", ".join(sorted(mods)))

In [ ]:
# Cell 76: Phase 0a -- list contents of the 9 .pkl files (READS ONLY; writes nothing; no new globals except _c76)
def _c76():
    import os, glob, pickle
    import numpy as np, pandas as pd
    d = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit"

    def desc(o):
        if isinstance(o, pd.DataFrame):
            cols = list(o.columns)
            return f"DataFrame {o.shape} cols={cols[:8]}{'...' if len(cols) > 8 else ''}"
        if isinstance(o, pd.Series):
            return f"Series len={len(o)} dtype={o.dtype}"
        if isinstance(o, np.ndarray):
            return f"ndarray {o.shape} {o.dtype}"
        if isinstance(o, dict):
            return f"dict len={len(o)}"
        if isinstance(o, (list, tuple, set)):
            return f"{type(o).__name__} len={len(o)}"
        if hasattr(o, 'nodes') and hasattr(o, 'edges'):
            try:
                return f"{type(o).__name__} nodes={len(o.nodes())} edges={len(o.edges())}"
            except Exception:
                pass
        if isinstance(o, (int, float, str, bool, np.generic)):
            return f"{type(o).__name__} = {str(o)[:60]}"
        return type(o).__name__

    for p in sorted(glob.glob(os.path.join(d, "*.pkl"))):
        print(f"\n== {os.path.basename(p)} ({os.path.getsize(p):,} bytes)")
        try:
            with open(p, 'rb') as fh:
                obj = pickle.load(fh)
        except Exception as ex:
            print("  LOAD ERROR:", type(ex).__name__, str(ex)[:150])
            continue
        print("  top:", desc(obj))
        if isinstance(obj, dict):
            for k, v in list(obj.items())[:25]:
                print(f"    [{str(k)[:40]}] {desc(v)}")
            if len(obj) > 25:
                print(f"    ... {len(obj) - 25} more keys")
        del obj

_c76()

In [ ]:
# Cell 77: Phase 0a -- cross-check kernel objects vs pkls; print key values; check recoverable source (READS ONLY; no globals except _c77)
def _c77():
    import pickle, inspect, re
    import numpy as np, pandas as pd
    g = globals()
    import os; d = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit/"
    L = lambda f: pickle.load(open(d + f, 'rb'))
    abl, bnp, art, mdl = L("ablation_predictions.pkl"), L("bn_test_predictions.pkl"), L("rq2_feature_importance_artifacts.pkl"), L("bn_final_model_iteration79.pkl")

    def cmp(label, a, b):
        try:
            a, b = np.asarray(a, float), np.asarray(b, float)
            print(f"  {label:34s} shape_ok={a.shape == b.shape}  exact={np.array_equal(a, b)}  max|diff|={np.max(np.abs(a - b)) if a.shape == b.shape else 'n/a'}")
        except Exception as ex:
            print(f"  {label:34s} ERROR {type(ex).__name__}: {str(ex)[:80]}")

    print("1) Kernel vs pkl predictions")
    for lab, k, v in [("lr_test_probs vs LR_all", 'lr_test_probs', abl['LR_all']),
                      ("rf_restricted_test_probs vs RF_all", 'rf_restricted_test_probs', abl['RF_all']),
                      ("bn_test_probs vs pkl", 'bn_test_probs', bnp['bn_test_probs']),
                      ("bn_test_probs vs ablation BN", 'bn_test_probs', abl['BN'])]:
        cmp(lab, g[k], v) if k in g else print(f"  {lab}: kernel object missing")

    print("\n2) Best params (on disk) and kernel scalars")
    print("  lr_best_params:", art['lr_best_params'])
    print("  rf_best_params:", art['rf_best_params'])
    for k in ['tau_lr', 'tau_rf', 'tau_bn', 'LR_BASE', 'RF_BASE', 'lr_test_auc', 'rf_restricted_test_auc', 'bn_test_auc', 'SEED', 'N_BINS']:
        print(f"  {k:24s}", g.get(k, 'MISSING'))
    if 'lr_final' in g:
        print("  lr_final params:", {k: v for k, v in g['lr_final'].get_params().items() if k in ('C', 'penalty', 'solver', 'class_weight', 'max_iter', 'l1_ratio')})

    print("\n3) Kernel BN objects vs iteration-79 file")
    disk_edges = sorted(map(tuple, mdl['model_edges']))
    for k in ['bn_final', 'bn', 'bn0']:
        if k in g:
            e = sorted(g[k].edges())
            print(f"  {k:9s} nodes={len(g[k].nodes())} edges={len(e)} edges_equal_to_pkl={e == disk_edges}")
    print("  pkl edges:", len(disk_edges), " pkl cpds:", len(mdl['cpds']))

    print("\n4) Split / data sanity (kernel)")
    for k in ['train_idx', 'test_idx', 'y_train', 'y_test']:
        if k in g:
            o = g[k]; print(f"  {k:10s} len={len(o)}  first5={list(o[:5])}")
    if 'y_train' in g and 'y_test' in g:
        print(f"  default rate train={np.mean(g['y_train']):.4%} test={np.mean(g['y_test']):.4%}")
    print("  excluded_row_indices (test):", bnp['excluded_row_indices'])
    if 'bin_edges' in g:
        print("  bin_edges keys:", list(g['bin_edges'].keys()))

    print("\n5) Recoverable source for key functions")
    for fn in ['run_custom_em', 'compute_log_likelihood', 'compute_new_cpds', 'compute_row_weights', 'row_sensitivity',
               'make_posterior_fn', 'posterior_default', 'build_evidence', 'paired_bootstrap', 'bootstrap_auc_diff',
               'find_optimal_threshold', 'cost_sensitive_loss', 'compute_ks', 'analyze_blanket', 'bin_natural_category']:
        try:
            src = inspect.getsource(g[fn]); print(f"  {fn:26s} source OK ({len(src.splitlines())} lines)")
        except Exception as ex:
            print(f"  {fn:26s} NO SOURCE ({type(ex).__name__})")

    print("\n6) Cell history: where HillClimbSearch / restarts appear")
    hist = g.get('In')
    if hist:
        hits = [(i, c) for i, c in enumerate(hist) if isinstance(c, str) and 'HillClimbSearch(' in c]
        print(f"  history cells: {len(hist)}; cells containing 'HillClimbSearch(': {[i for i, _ in hits]}")
        for i, c in hits[-2:]:
            print(f"  --- In[{i}] first 12 lines ---")
            print("\n".join(c.splitlines()[:12]))
    else:
        print("  In[] history not available")

_c77()

In [ ]:
# Cell 78: Phase 0a -- save KERNEL-ONLY objects (DRY RUN by default).
# READS ONLY while DRY_RUN=True. If DRY_RUN=False it WRITES NEW files only under:
#   <ROOT>/full_variable/{models,tables}, <ROOT>/data/processed, <ROOT>/shared/src, <ROOT>/archive/kernel
def _c78(DRY_RUN=True):
    import os, sys, json, pickle, inspect, hashlib, platform, importlib, types
    import numpy as np, pandas as pd
    g = globals()
    R = os.environ["CREDIT_BN_ROOT"]
    print(f"DRY_RUN = {DRY_RUN}")

    # ---------- pre-flight (read-only) ----------
    print("\nPre-flight checks")
    bnk = np.asarray(g['bn_test_probs'], float)
    bnp = pickle.load(open(R + "/Data/GiveMeSomeCredit/bn_test_predictions.pkl", 'rb'))
    bnd = np.asarray(bnp['bn_test_probs'], float)
    nk, nd = np.isnan(bnk), np.isnan(bnd)
    ok = ~nk & ~nd
    print(f"  BN: NaN kernel={nk.sum()} disk={nd.sum()} same NaN positions={np.array_equal(nk, nd)} "
          f"max|diff| non-NaN={np.max(np.abs(bnk[ok] - bnd[ok])):.3g}")
    print(f"  NaN rows by label: {list(np.asarray(g['test_idx'])[nk])}  vs excluded_row_indices {bnp['excluded_row_indices']}")
    try:
        tr, ks = set(np.asarray(g['train_idx']).tolist()), set(np.asarray(g['keys']).tolist())
        print(f"  train-side: |train_idx|={len(tr)} |keys|={len(ks)} train-not-in-keys={sorted(tr - ks)[:20]} keys-not-in-train={len(ks - tr)}")
    except Exception as ex:
        print("  train-side exclusion check ERROR:", type(ex).__name__, str(ex)[:100])
    for e_, d_ in [('X_train_encoded', 'X_train_df'), ('X_test_encoded', 'X_test_df')]:
        try:
            print(f"  {e_} == {d_}.values: {np.allclose(g[e_], g[d_].values)}")
        except Exception as ex:
            print(f"  {e_} check ERROR: {type(ex).__name__}")

    # ---------- builders ----------
    def grid():
        out = {}
        for k, nm in [('lr', 'lr_grid_search'), ('rf', 'rf_grid_search_restricted')]:
            o = g[nm]
            out[k] = dict(best_params=o.best_params_, best_score=float(o.best_score_), cv_results=o.cv_results_,
                          scoring=o.scoring, param_grid=o.param_grid)
        return out

    def versions():
        v = {'python': sys.version, 'platform': platform.platform(), 'PYTHONHASHSEED': os.environ.get('PYTHONHASHSEED', 'UNSET')}
        for m in ['numpy', 'pandas', 'scipy', 'sklearn', 'pgmpy', 'joblib', 'matplotlib']:
            try:
                v[m] = importlib.import_module(m).__version__
            except Exception:
                v[m] = 'n/a'
        return v

    def scalars():
        ks_ = ['tau_lr', 'tau_rf', 'tau_bn', 'LR_BASE', 'RF_BASE', 'lr_test_auc', 'rf_restricted_test_auc',
               'bn_test_auc', 'bn_gini', 'lr_gini', 'rf_gini', 'SEED', 'N_BINS']
        return {k: (float(g[k]) if isinstance(g[k], (int, float, np.generic)) else str(g[k])) for k in ks_ if k in g}

    SKIPF = {'sha12', 'approx_size', 'shape_or_len'}
    def recovered():
        parts = ["# UNEDITED dump of function source recovered from the live kernel (inspect.getsource).",
                 "# Order is alphabetical, NOT definition order. Not importable as-is; to be refactored into shared/src modules.\n"]
        for nm in sorted(g):
            o = g[nm]
            if isinstance(o, types.FunctionType) and o.__module__ == '__main__' and nm not in SKIPF and not nm.startswith('_c'):
                try:
                    parts.append(f"# ---- {nm} ----\n" + inspect.getsource(o))
                except Exception:
                    parts.append(f"# ---- {nm}: NO SOURCE ----")
        return "\n".join(parts)

    def history():
        return "\n\n".join(f"# ===== In[{i}] =====\n{c}" for i, c in enumerate(g['In']) if isinstance(c, str))

    P = "data/processed/"
    df_names = ['df_clean', 'df_binned', 'df_train', 'df_test', 'df_train_complete', 'df_train_imputed',
                'df_test_imputed', 'df_train_lr_rf', 'df_test_lr_rf', 'X_train_df', 'X_test_df']
    items = [
        ("full_variable/models/lr_final.pkl", 'pkl', lambda: g['lr_final']),
        ("full_variable/models/rf_final_restricted.pkl", 'pkl', lambda: g['rf_final_restricted']),
        ("full_variable/models/imputer.pkl", 'pkl', lambda: g['imputer']),
        ("full_variable/models/encoder.pkl", 'pkl', lambda: g['encoder']),
        ("full_variable/models/grid_search_summaries.pkl", 'pkl', grid),
        ("full_variable/models/env_versions.json", 'json', versions),
        ("full_variable/tables/kernel_scalars.json", 'json', scalars),
        ("full_variable/tables/metrics_table_kernel.csv", 'csv', lambda: g['metrics_table']),
        (P + "split_indices.pkl", 'pkl', lambda: {k: g[k] for k in ['train_idx', 'test_idx', 'y_train', 'y_test']}),
        (P + "analysis_row_sets.pkl", 'pkl', lambda: {k: g[k] for k in ['keys', 'analysis_idx', 'shared_index']}),
        (P + "preprocessing_artifacts.pkl", 'pkl', lambda: {k: g[k] for k in
            ['bin_edges', 'income_edges', 'income_edges_adj', 'encoded_feature_names', 'FEATURE_BINS', 'bin_cols', 'N_BINS', 'SEED']}),
    ] + [(P + f"{n}.pkl", 'pkl', (lambda n=n: g[n])) for n in df_names] + [
        ("shared/src/_recovered_from_kernel.py", 'txt', recovered),
        ("archive/kernel/In_history_at_cell78.txt", 'txt', history),
    ]

    def ser(kind, o):
        if kind == 'pkl':  return pickle.dumps(o, protocol=4)
        if kind == 'json': return json.dumps(o, indent=2, default=str).encode()
        if kind == 'csv':  return o.to_csv().encode()
        return o.encode()

    # ---------- plan / execute ----------
    print(f"\n{'destination (relative to Research/)':62s} {'status':14s} {'MB':>8s}  sha12")
    total = 0
    for rel, kind, fb in items:
        dest = os.path.join(R, rel)
        if os.path.exists(dest):
            print(f"{rel:62s} {'EXISTS->SKIP':14s}"); continue
        try:
            data = ser(kind, fb())
        except Exception as ex:
            print(f"{rel:62s} {'ERROR':14s} {type(ex).__name__}: {str(ex)[:60]}"); continue
        h = hashlib.sha256(data).hexdigest()[:12]
        status = 'would write'
        if not DRY_RUN:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with open(dest, 'xb') as fh:          # 'x' = refuses to overwrite
                fh.write(data)
            with open(dest, 'rb') as fh:
                back = hashlib.sha256(fh.read()).hexdigest()[:12]
            status = 'WRITTEN ok' if back == h else 'SHA MISMATCH'
        print(f"{rel:62s} {status:14s} {len(data)/1e6:8.2f}  {h}")
        total += len(data); del data
    print(f"\nTotal {total/1e6:,.1f} MB across {len(items)} planned files")
    print("Deliberately NOT saved: *_str frames, *_encoded arrays, caches (cache/cacheB/wcache/evid_all), toy/scratch objects, "
          "unfitted lr_base/rf_base, whole GridSearchCV objects.")

_c78()   # DRY_RUN=True by default; nothing is written

In [ ]:
# Cell 79: Phase 0a -- (1) retry train-side exclusion check, (2) print full In[29] (READS ONLY; writes nothing; no new globals except _c79)
def _c79():
    import numpy as np
    g = globals()
    print("1) train-side exclusion check")
    keys = g['keys']
    print(f"  type(keys)={type(keys).__name__} len={len(keys)}  first 3: {keys[:3]}  elem type={type(keys[0]).__name__}")
    tr = np.asarray(g['train_idx']).tolist()
    def rid(k):
        return k[0] if isinstance(k, (list, tuple, np.ndarray)) else k
    try:
        kset = {rid(k) for k in keys}
        trset = set(tr)
        print(f"  unique row ids in keys (taking element 0 if sequence): {len(kset)}")
        print(f"  |train_idx|={len(trset)}  train-not-in-keys (first 20)={sorted(trset - kset)[:20]}  count={len(trset - kset)}")
        print(f"  keys-not-in-train count={len(kset - trset)}")
    except Exception as ex:
        print("  ERROR:", type(ex).__name__, str(ex)[:120])
    print("\n2) In[29] in full")
    print(g['In'][29])

_c79()

In [ ]:
# Cell 80: Phase 0a -- targeted verification of LOCKED outputs (READS ONLY; writes nothing; no new globals except _c80)
def _c80():
    import os, glob, pickle, inspect, time
    import numpy as np, pandas as pd
    from scipy.stats import ks_2samp
    from sklearn.metrics import roc_auc_score
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    g = globals()
    D = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit/"
    L = lambda f: pickle.load(open(D + f, 'rb'))
    fails = []
    def chk(label, cond, detail=""):
        print(f"  [{'PASS' if cond else 'FAIL'}] {label}  {detail}")
        if not cond: fails.append(label)
    TOK = ['Credit', '90+', '30-59', '60-89']
    toks = lambda names: {t for n in names for t in TOK if t in n}

    print("A) bn_final_model_iteration79.pkl")
    try:
        mdl = L("bn_final_model_iteration79.pkl")
        m = DiscreteBayesianNetwork(list(map(tuple, mdl['model_edges']))); m.add_cpds(*mdl['cpds'])
        chk("check_model()", bool(m.check_model()))
        chk("13 nodes", len(m.nodes()) == 13, f"({len(m.nodes())})")
        chk("24 edges", len(m.edges()) == 24, f"({len(m.edges())})")
        dn = [n for n in m.nodes() if n == 'Default'] or [n for n in m.nodes() if n.startswith('Default')]
        d = dn[0]
        pa, ch = list(m.get_parents(d)), list(m.get_children(d))
        chk("Default parents {90+, CreditUtil}", len(pa) == 2 and toks(pa) == {'Credit', '90+'}, str(pa))
        chk("Default children {30-59, 60-89}", len(ch) == 2 and toks(ch) == {'30-59', '60-89'}, str(ch))
        sn = m.get_cpds(d).state_names[d]
        i1 = list(sn).index(1) if 1 in sn else list(sn).index('1')
        p1 = float(VariableElimination(m).query([d], show_progress=False).values[i1])
        chk("marginal P(Default=1) ~ 6.68%", abs(p1 - 0.0668) < 5e-4, f"({p1:.4%})")
        chk("peak_iteration == 79", mdl['peak_iteration'] == 79)
        print(f"  train_ll_at_peak={mdl['train_ll_at_peak']:.6f} (expect -1316746.532478)  test_ll_at_peak={mdl['test_ll_at_peak']:.6f} (expect -329111.347037)")
        chk("peak LL values match planning numbers", abs(mdl['train_ll_at_peak'] + 1316746.532478) < 1e-3 and abs(mdl['test_ll_at_peak'] + 329111.347037) < 1e-3)
        for fn in ['compute_log_likelihood', 'run_custom_em']:
            print(f"  signature {fn}{inspect.signature(g[fn])}" if fn in g else f"  {fn} missing")
        print("  (LL recompute from the model skipped: signature must be inspected first)")
    except Exception as ex:
        chk("section A ran", False, f"{type(ex).__name__}: {str(ex)[:120]}")

    y = np.asarray(g['y_test'])
    abl = L("ablation_predictions.pkl")
    print("\nB) bn_test_predictions.pkl")
    try:
        bp = L("bn_test_predictions.pkl"); s = bp['bn_test_probs']; v = np.asarray(s, float)
        nanpos = np.where(np.isnan(v))[0]; ok = ~np.isnan(v)
        chk("30,000 rows", len(v) == 30000, f"({len(v)})")
        lab = sorted(np.asarray(g['test_idx'])[nanpos].tolist())
        chk("NaN exactly at 125238 and 7312", lab == sorted([125238, 7312]), str(lab))
        auc = roc_auc_score(y[ok], v[ok]); ks = ks_2samp(v[ok][y[ok] == 1], v[ok][y[ok] == 0]).statistic
        chk("AUC 0.8519", round(auc, 4) == 0.8519, f"({auc:.6f})")
        chk("Gini 0.7038", round(2 * auc - 1, 4) == 0.7038, f"({2*auc-1:.6f})")
        chk("KS 0.5562", round(ks, 4) == 0.5562, f"({ks:.6f})")
        chk("202 unique values", len(np.unique(v[ok])) == 202, f"({len(np.unique(v[ok]))})")
        chk("mean 0.0676", round(v[ok].mean(), 4) == 0.0676, f"({v[ok].mean():.5f})")
        chk("median 0.0161", round(float(np.median(v[ok])), 4) == 0.0161, f"({np.median(v[ok]):.5f})")
        chk("2.41% >= 0.5", round(100 * (v[ok] >= 0.5).mean(), 2) == 2.41, f"({100*(v[ok]>=0.5).mean():.3f}%)")
    except Exception as ex:
        chk("section B ran", False, f"{type(ex).__name__}: {str(ex)[:120]}"); v = np.asarray(abl['BN'], float); ok = ~np.isnan(v)

    print("\nC) Confusion counts at own thresholds (prob >= tau; '>' shown only on FAIL)")
    def cm(p, yy, tau, strict=False):
        pr = (p > tau) if strict else (p >= tau)
        return (int((pr & (yy == 1)).sum()), int((pr & (yy == 0)).sum()), int((~pr & (yy == 1)).sum()), int((~pr & (yy == 0)).sum()))
    for nm, p, yy, tau, exp, cost in [("LR 0.69", np.asarray(abl['LR_all'], float), y, 0.69, (1139, 2143, 866, 25852), 6473),
                                      ("RF 0.67", np.asarray(abl['RF_all'], float), y, 0.67, (1128, 2089, 877, 25906), 6474),
                                      ("BN 0.17", v[ok], y[ok], 0.17, (1130, 2127, 873, 25868), 6492)]:
        c = cm(p, yy, tau); cs = 5 * c[2] + c[1]
        chk(f"{nm} TP/FP/FN/TN", c == exp, f"{c} cost={cs}" + ("" if c == exp else f" | strict '>' gives {cm(p, yy, tau, True)}"))
        chk(f"{nm} cost {cost}", cs == cost)

    print("\nD) ablation_predictions.pkl AUCs (test, n=30,000)")
    for k, e in [('LR_4var', 0.8508), ('LR_noID', 0.8612), ('RF_noID', 0.8599)]:
        a = roc_auc_score(y, np.asarray(abl[k], float)); chk(f"{k} AUC {e}", round(a, 4) == e, f"({a:.6f})")

    print("\nE) per-row pickles")
    try:
        r2 = L("rq2_sensitivity_per_row.pkl")
        vars_ = list(r2['variable'].unique())
        chk("rq2: 29,998 unique rows", r2['row_index'].nunique() == 29998, f"({r2['row_index'].nunique()}; {len(r2)} rows, {len(vars_)} variables)")
        nb = [x for x in vars_ if not any(t in x for t in TOK)]
        chk("rq2: 8 non-blanket variables", len(nb) == 8, f"({len(nb)})")
        chk("rq2: non-blanket S exactly 0", bool((r2[r2['variable'].isin(nb)]['S'].fillna(0) == 0).all()))
        print("  rq2 mean S by variable:", {x[:14]: round(float(r2[r2.variable == x]['S'].mean()), 4) for x in vars_ if x not in nb})
        vo = L("vo_blanket_original_per_row.pkl")
        spec = {'Credit': (0.0280, 0.0847, 0.0234, 1374, 4.58), '30-59': (0.0245, 0.1648, 0.0226, 1390, 4.63),
                '90+': (0.0223, 0.4388, 0.0214, 927, 3.09), '60-89': (0.0111, 0.5786, 0.0106, 419, 1.40)}
        for t, (se, sm, dl, fl, fp_) in spec.items():
            vn = [x for x in vo['variable'].unique() if t in x]
            if len(vn) != 1: chk(f"vo_original {t}", False, f"matched {vn}"); continue
            q = vo[vo.variable == vn[0]]; n = len(q)
            got = (round(q.S_exp.mean(), 4), round(q.S_max.mean(), 4), round(q.delta.mean(), 4), int(q.flip.sum()), round(100 * q.flip.sum() / n, 2))
            chk(f"vo_original {t}: S_exp/S_max/Delta/flips/flip%", got == (se, sm, dl, fl, fp_), f"{got} vs {(se, sm, dl, fl, fp_)} n={n}")
        base_rank = list(vo.groupby('variable').flip.sum().sort_values(ascending=False).index)
        for f_ in ["vo_blanket_ess1_per_row.pkl", "vo_blanket_ess10_per_row.pkl"]:
            q = L(f_); fs = q.groupby('variable').flip.sum()
            chk(f"{f_} loads, flip ranking matches original", list(fs.sort_values(ascending=False).index) == base_rank, str(fs.to_dict()))
    except Exception as ex:
        chk("section E ran", False, f"{type(ex).__name__}: {str(ex)[:120]}")

    print("\nF) Modified times: locked files and PNGs")
    names = ["rq2_sensitivity_per_row.pkl", "vo_blanket_original_per_row.pkl", "vo_blanket_ess1_per_row.pkl", "vo_blanket_ess10_per_row.pkl",
             "ablation_predictions.pkl", "bn_test_predictions.pkl", "bn_final_model_iteration79.pkl", "rq2_sensitivity_table.csv",
             "rq2_pattern_level_check.csv", "rq2_ext_partA_table.csv", "rq2_ext_partB_table.csv"]
    names += [os.path.basename(p) for p in sorted(glob.glob(D + "*.png"))]
    for n_ in names:
        p = D + n_
        print(f"  {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))}  {n_}" if os.path.exists(p) else f"  MISSING  {n_}")

    print(f"\nSUMMARY: {len(fails)} FAIL" + (f" -> STOP and report: {fails}" if fails else " -> all checks passed"))

_c80()

In [ ]:
# Cell 81: Phase 0a -- extra verification: notebook execution record, fitted LR/RF audit, EM PNG metadata (READS ONLY; writes nothing; no new globals except _c81)
def _c81():
    import os, json, time
    import numpy as np
    from sklearn.metrics import roc_auc_score
    g = globals()
    D = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit/"
    fails = []
    def chk(label, cond, detail=""):
        print(f"  [{'PASS' if cond else 'FAIL'}] {label}  {detail}")
        if not cond: fails.append(label)

    print("1) Notebook execution record")
    print("  PYTHONHASHSEED in this kernel:", repr(os.environ.get('PYTHONHASHSEED')))
    print("  kernel In[] length:", len(g['In']) if 'In' in g else 'n/a')
    p = D + "Expirements.ipynb"
    print("  ipynb mtime:", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))))
    nb = json.load(open(p))
    cells = [c for c in nb['cells'] if c['cell_type'] == 'code']
    src = lambda c: ''.join(c['source']) if isinstance(c['source'], list) else c['source']
    ecs = [c.get('execution_count') for c in cells]
    ran = [e for e in ecs if e is not None]
    print(f"  code cells: {len(cells)}  executed (have a count): {len(ran)}  never executed: {len(cells) - len(ran)}")
    if ran:
        print(f"  execution_count min={min(ran)} max={max(ran)}  strictly increasing in notebook order: {all(a < b for a, b in zip(ran, ran[1:]))}")
        inv = [(i, ecs[i]) for i in range(len(ecs) - 1) if ecs[i] is not None and ecs[i + 1] is not None and ecs[i] >= ecs[i + 1]]
        print(f"  out-of-order positions (cell idx, count): {inv[:10]}")
    KW = ['GridSearchCV(', '_search.fit(', 'run_custom_em(', 'HillClimbSearch(', 'DiscreteEM(']
    print("  cells containing key calls (ACTIVE = uncommented line, COMM = commented line):")
    found = False
    for i, c in enumerate(cells):
        lines = src(c).splitlines()
        for kw in KW:
            act = sum(1 for l in lines if kw in l and not l.lstrip().startswith('#'))
            com = sum(1 for l in lines if kw in l and l.lstrip().startswith('#'))
            if act or com:
                found = True
                first = next((l.strip() for l in lines if l.strip()), '')[:50]
                print(f"    code-cell {i:3d} exec_count={ecs[i]}  {kw:16s} ACTIVE={act} COMM={com}  | {first}")
    if not found:
        print("    none found")

    print("\n2) Fitted LR / RF audit")
    for nm, gs_n in [('LR', 'lr_grid_search'), ('RF', 'rf_grid_search_restricted')]:
        gs = g[gs_n]
        print(f"  {nm} grid: best_params_={gs.best_params_}  best_score_={gs.best_score_:.6f}  scoring={gs.scoring}  cv={gs.cv}")
    chk("RF best_params 500/15/10", g['rf_grid_search_restricted'].best_params_ == {'max_depth': 15, 'min_samples_leaf': 10, 'n_estimators': 500}, str(g['rf_grid_search_restricted'].best_params_))
    chk("RF CV 0.8559", round(g['rf_grid_search_restricted'].best_score_, 4) == 0.8559, f"({g['rf_grid_search_restricted'].best_score_:.6f})")
    chk("LR CV 0.8582", round(g['lr_grid_search'].best_score_, 4) == 0.8582, f"({g['lr_grid_search'].best_score_:.6f})")
    lr, rf = g['lr_final'], g['rf_final_restricted']
    print("  lr_final: ", {k: v for k, v in lr.get_params().items() if k in ('C', 'penalty', 'solver', 'class_weight', 'max_iter', 'random_state', 'tol')})
    print("  rf_final: ", {k: v for k, v in rf.get_params().items() if k in ('n_estimators', 'max_depth', 'min_samples_leaf', 'class_weight', 'random_state', 'n_jobs', 'max_features')})
    for nm, fin, gs_n in [('LR', lr, 'lr_grid_search'), ('RF', rf, 'rf_grid_search_restricted')]:
        try:
            be = g[gs_n].best_estimator_
            chk(f"{nm} final params == grid best_estimator_ params", fin.get_params() == be.get_params())
        except Exception as ex:
            print(f"  {nm} best_estimator_ comparison skipped: {type(ex).__name__}")
    Xte, y = g['X_test_df'], np.asarray(g['y_test'])
    for nm, fin, ref_n, exp in [('LR', lr, 'lr_test_probs', 0.8618), ('RF', rf, 'rf_restricted_test_probs', 0.8603)]:
        try:
            pr = fin.predict_proba(Xte.values)[:, 1]
            a = roc_auc_score(y, pr)
            chk(f"{nm} test AUC {exp}", round(a, 4) == exp, f"({a:.6f})")
            print(f"     {nm} predict_proba vs saved kernel probs: max|diff|={np.max(np.abs(pr - np.asarray(g[ref_n], float))):.3g}")
        except Exception as ex:
            chk(f"{nm} test AUC computed", False, f"{type(ex).__name__}: {str(ex)[:100]}")

    print("\n3) EM convergence PNGs (metadata only; not opened for writing)")
    for n_ in ["em_convergence_final_with_zoom.png", "em_convergence_final.png"]:
        q = D + n_
        if os.path.exists(q):
            print(f"  {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(q)))}  {os.path.getsize(q):>9,} bytes  {n_}")
        else:
            print(f"  MISSING  {n_}")
    print("  -> You: open em_convergence_final_with_zoom.png and confirm two panels with the peak at iteration 79.")

    print(f"\nSUMMARY: {len(fails)} FAIL" + (f" -> STOP and report: {fails}" if fails else " -> LR/RF audit passed"))

_c81()

In [ ]:
# Cell 82: Phase 0a -- recheck rq2 non-blanket count by EXACT name (READS ONLY; writes nothing; no new globals except _c82)
def _c82():
    import pickle
    import numpy as np
    import os; D = os.environ["CREDIT_BN_ROOT"] + "/Data/GiveMeSomeCredit/"
    r2 = pickle.load(open(D + "rq2_sensitivity_per_row.pkl", 'rb'))
    BL = ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)']
    allv = list(r2['variable'].unique())
    nb = [v for v in allv if v not in BL]
    print(f"{len(allv)} variables; blanket present: {[v for v in BL if v in allv]}; non-blanket ({len(nb)}):")
    for v in allv:
        q = r2[r2.variable == v]
        print(f"  {'BLANKET' if v in BL else 'non-bl '} {v:34s} rows={len(q):6d} meanS={q.S.mean():.4f} maxS={q.S.max():.4f} all_S_zero={bool((q.S.fillna(0) == 0).all())}")
    print("non-blanket count == 8:", len(nb) == 8, "| all non-blanket S == 0:", bool((r2[r2.variable.isin(nb)].S.fillna(0) == 0).all()))
    print("rows per variable distinct counts:", sorted(set(r2.groupby('variable').size())))

_c82()

In [ ]:
# Cell 78 (REAL RUN, approved): save KERNEL-ONLY objects.
# WRITES NEW files only under:
#   <ROOT>/full_variable/{models,tables}, <ROOT>/data/processed, <ROOT>/shared/src, <ROOT>/archive/kernel
# Never overwrites (exclusive-create). Existing paths are skipped.
def _c78(DRY_RUN=False):
    import os, sys, json, pickle, inspect, hashlib, platform, importlib, types
    import numpy as np, pandas as pd
    g = globals()
    R = os.environ["CREDIT_BN_ROOT"]
    print(f"DRY_RUN = {DRY_RUN}")

    # ---------- pre-flight (read-only) ----------
    print("\nPre-flight checks")
    bnk = np.asarray(g['bn_test_probs'], float)
    bnp = pickle.load(open(R + "/Data/GiveMeSomeCredit/bn_test_predictions.pkl", 'rb'))
    bnd = np.asarray(bnp['bn_test_probs'], float)
    nk, nd = np.isnan(bnk), np.isnan(bnd)
    ok = ~nk & ~nd
    print(f"  BN: NaN kernel={nk.sum()} disk={nd.sum()} same NaN positions={np.array_equal(nk, nd)} "
          f"max|diff| non-NaN={np.max(np.abs(bnk[ok] - bnd[ok])):.3g}")
    for e_, d_ in [('X_train_encoded', 'X_train_df'), ('X_test_encoded', 'X_test_df')]:
        try:
            print(f"  {e_} == {d_}.values: {np.allclose(g[e_], g[d_].values)}")
        except Exception as ex:
            print(f"  {e_} check ERROR: {type(ex).__name__}")

    # ---------- builders ----------
    def grid():
        out = {}
        for k, nm in [('lr', 'lr_grid_search'), ('rf', 'rf_grid_search_restricted')]:
            o = g[nm]
            out[k] = dict(best_params=o.best_params_, best_score=float(o.best_score_), cv_results=o.cv_results_,
                          scoring=o.scoring, param_grid=o.param_grid)
        return out

    def versions():
        v = {'python': sys.version, 'platform': platform.platform(), 'PYTHONHASHSEED': os.environ.get('PYTHONHASHSEED', 'UNSET')}
        for m in ['numpy', 'pandas', 'scipy', 'sklearn', 'pgmpy', 'joblib', 'matplotlib']:
            try:
                v[m] = importlib.import_module(m).__version__
            except Exception:
                v[m] = 'n/a'
        return v

    def scalars():
        ks_ = ['tau_lr', 'tau_rf', 'tau_bn', 'LR_BASE', 'RF_BASE', 'lr_test_auc', 'rf_restricted_test_auc',
               'bn_test_auc', 'bn_gini', 'lr_gini', 'rf_gini', 'SEED', 'N_BINS']
        return {k: (float(g[k]) if isinstance(g[k], (int, float, np.generic)) else str(g[k])) for k in ks_ if k in g}

    SKIPF = {'sha12', 'approx_size', 'shape_or_len'}
    def recovered():
        parts = ["# UNEDITED dump of function source recovered from the live kernel (inspect.getsource).",
                 "# Order is alphabetical, NOT definition order. Not importable as-is; to be refactored into shared/src modules.\n"]
        for nm in sorted(g):
            o = g[nm]
            if isinstance(o, types.FunctionType) and o.__module__ == '__main__' and nm not in SKIPF and not nm.startswith('_c'):
                try:
                    parts.append(f"# ---- {nm} ----\n" + inspect.getsource(o))
                except Exception:
                    parts.append(f"# ---- {nm}: NO SOURCE ----")
        return "\n".join(parts)

    def history():
        return "\n\n".join(f"# ===== In[{i}] =====\n{c}" for i, c in enumerate(g['In']) if isinstance(c, str))

    P = "data/processed/"
    df_names = ['df_clean', 'df_binned', 'df_train', 'df_test', 'df_train_complete', 'df_train_imputed',
                'df_test_imputed', 'df_train_lr_rf', 'df_test_lr_rf', 'X_train_df', 'X_test_df']
    items = [
        ("full_variable/models/lr_final.pkl", 'pkl', lambda: g['lr_final']),
        ("full_variable/models/rf_final_restricted.pkl", 'pkl', lambda: g['rf_final_restricted']),
        ("full_variable/models/imputer.pkl", 'pkl', lambda: g['imputer']),
        ("full_variable/models/encoder.pkl", 'pkl', lambda: g['encoder']),
        ("full_variable/models/grid_search_summaries.pkl", 'pkl', grid),
        ("full_variable/models/env_versions.json", 'json', versions),
        ("full_variable/tables/kernel_scalars.json", 'json', scalars),
        ("full_variable/tables/metrics_table_kernel.csv", 'csv', lambda: g['metrics_table']),
        (P + "split_indices.pkl", 'pkl', lambda: {k: g[k] for k in ['train_idx', 'test_idx', 'y_train', 'y_test']}),
        (P + "analysis_row_sets.pkl", 'pkl', lambda: {k: g[k] for k in ['keys', 'analysis_idx', 'shared_index']}),
        (P + "preprocessing_artifacts.pkl", 'pkl', lambda: {k: g[k] for k in
            ['bin_edges', 'income_edges', 'income_edges_adj', 'encoded_feature_names', 'FEATURE_BINS', 'bin_cols', 'N_BINS', 'SEED']}),
    ] + [(P + f"{n}.pkl", 'pkl', (lambda n=n: g[n])) for n in df_names] + [
        ("shared/src/_recovered_from_kernel.py", 'txt', recovered),
        ("archive/kernel/In_history_at_cell78.txt", 'txt', history),
    ]

    def ser(kind, o):
        if kind == 'pkl':  return pickle.dumps(o, protocol=4)
        if kind == 'json': return json.dumps(o, indent=2, default=str).encode()
        if kind == 'csv':  return o.to_csv().encode()
        return o.encode()

    # ---------- plan / execute ----------
    print(f"\n{'destination (relative to Research/)':62s} {'status':14s} {'MB':>8s}  sha12")
    total = 0
    for rel, kind, fb in items:
        dest = os.path.join(R, rel)
        if os.path.exists(dest):
            print(f"{rel:62s} {'EXISTS->SKIP':14s}"); continue
        try:
            data = ser(kind, fb())
        except Exception as ex:
            print(f"{rel:62s} {'ERROR':14s} {type(ex).__name__}: {str(ex)[:60]}"); continue
        h = hashlib.sha256(data).hexdigest()[:12]
        status = 'would write'
        if not DRY_RUN:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with open(dest, 'xb') as fh:          # 'x' = refuses to overwrite
                fh.write(data)
            with open(dest, 'rb') as fh:
                back = hashlib.sha256(fh.read()).hexdigest()[:12]
            status = 'WRITTEN ok' if back == h else 'SHA MISMATCH'
        print(f"{rel:62s} {status:14s} {len(data)/1e6:8.2f}  {h}")
        total += len(data); del data
    print(f"\nTotal {total/1e6:,.1f} MB across {len(items)} planned files")
    print("Deliberately NOT saved: *_str frames, *_encoded arrays, caches (cache/cacheB/wcache/evid_all), toy/scratch objects, "
          "unfitted lr_base/rf_base, whole GridSearchCV objects.")

_c78(DRY_RUN=False)

In [ ]:
# Cell 83: Phase 0a -- save node orders + best params JSON (DRY RUN by default).
# READS ONLY while DRY_RUN=True. If DRY_RUN=False it WRITES NEW files only:
#   <ROOT>/data/processed/node_orders.json, <ROOT>/full_variable/tables/best_params.json (exclusive-create, never overwrites)
def _c83(DRY_RUN=True):
    import os, json, hashlib
    g = globals()
    R = os.environ["CREDIT_BN_ROOT"]
    print(f"DRY_RUN = {DRY_RUN}")

    DROP_VARS = ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)']
    orig = list(g['original_11_nodes'])
    print("original_11_nodes:", orig)
    print("df_train_complete columns:", list(g['df_train_complete'].columns))
    print("  original_11_nodes == df_train_complete.columns (same order):", orig == list(g['df_train_complete'].columns))
    missing = [v for v in DROP_VARS if v not in orig]
    print("  dropped names missing from original list:", missing)
    assert not missing, "Drop names do not match node names; fix before saving"
    derived = [n for n in orig if n not in DROP_VARS]
    print("derived 7-node order:", derived, "| n =", len(derived))
    assert len(derived) == 7 and 'Default' in derived
    KEEP_VARS = [n for n in derived if n != 'Default']

    node_orders = {
        'original_11_nodes': orig,
        'derived_7_nodes_selection': derived,
        'DROP_VARS': DROP_VARS,
        'KEEP_VARS': KEEP_VARS,
        'col_order': list(g['col_order']) if 'col_order' in g else None,
        'model_cols': list(g['model_cols']) if 'model_cols' in g else None,
        'VARS12': list(g['VARS12']) if 'VARS12' in g else None,
        'note': "Derived order = original_11_nodes with DROP_VARS removed, relative order preserved (planning-instance decision 1a).",
    }
    bp = {}
    for k, nm, fin in [('LR', 'lr_grid_search', 'lr_final'), ('RF', 'rf_grid_search_restricted', 'rf_final_restricted')]:
        gs = g[nm]
        bp[k] = {'best_params': gs.best_params_, 'cv_best_score_auc': float(gs.best_score_),
                 'scoring': gs.scoring, 'cv': str(gs.cv), 'random_state': g[fin].get_params().get('random_state'),
                 'final_model_params': {a: (b if isinstance(b, (int, float, str, bool, type(None))) else str(b))
                                        for a, b in g[fin].get_params().items()}}
    items = [("data/processed/node_orders.json", node_orders), ("full_variable/tables/best_params.json", bp)]

    print(f"\n{'destination':48s} {'status':14s} {'bytes':>8s}  sha12")
    for rel, obj in items:
        dest = os.path.join(R, rel)
        if os.path.exists(dest):
            print(f"{rel:48s} {'EXISTS->SKIP':14s}"); continue
        data = json.dumps(obj, indent=2, default=str).encode()
        h = hashlib.sha256(data).hexdigest()[:12]
        status = 'would write'
        if not DRY_RUN:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with open(dest, 'xb') as fh:
                fh.write(data)
            with open(dest, 'rb') as fh:
                status = 'WRITTEN ok' if hashlib.sha256(fh.read()).hexdigest()[:12] == h else 'SHA MISMATCH'
        print(f"{rel:48s} {status:14s} {len(data):8d}  {h}")
    print("\nbest params:", {k: v['best_params'] for k, v in bp.items()})

_c83()   # DRY_RUN=True by default; nothing is written

In [ ]:
# Cell 84: Phase 0a -- verify Cell 78 saves touched only NEW paths and no locked file changed (READS ONLY; writes nothing; no new globals except _c84)
def _c84():
    import os, time, hashlib
    R = os.environ["CREDIT_BN_ROOT"]
    G = "Data/GiveMeSomeCredit/"
    base = {  # sha256[:12] from the Cell 74 inventory (taken AFTER the 14:10-14:22 rerun)
        "Data/Data_Exploration.py": "f8a44bdcf571", G+"Data Dictionary.xls": "d388d6b11ed6", G+"ablation_predictions.pkl": "6535605d1ccd",
        G+"bn_13node_dag_final.png": "e80e2d0c5a45", G+"bn_final_model_iteration79.pkl": "3b75db5acf3e", G+"bn_test_predictions.pkl": "f7880eb5a03d",
        G+"cs-test.csv": "bab363a2a807", G+"cs-training.csv": "1bd46da486a5", G+"diagnostic_prob_histograms.png": "43d9c81471e9",
        G+"em_convergence_final.png": "cd25837f9394", G+"em_convergence_final_with_zoom.png": "ae02061f206a", G+"final_dag.png": "e80e2d0c5a45",
        G+"rq1_chart1_threshold_curves.png": "53bb3d4ed119", G+"rq1_chart2_confusion_matrices.png": "41c45354c75a",
        G+"rq1_chart3_roc_curves.png": "0548af828ca9", G+"rq1_chart4_pr_curves.png": "f5b234ef5ec9", G+"rq1_chart5_ks_plots.png": "1ab5e76078d3",
        G+"rq2_ext_delta_distributions.png": "25b76ab896db", G+"rq2_ext_delta_vs_baseline.png": "8a7d2e7a1ff2",
        G+"rq2_ext_flip_rates.png": "60a61537dd9a", G+"rq2_ext_heatmap.png": "b2aaed17ab07", G+"rq2_ext_partA_table.csv": "00a760545289",
        G+"rq2_ext_partB_table.csv": "aa85f44d9e0c", G+"rq2_feature_importance_artifacts.pkl": "0f346dd79774",
        G+"rq2_pattern_level_check.csv": "3d87012be62c", G+"rq2_sensitivity_distributions.png": "44ee8b4068c7",
        G+"rq2_sensitivity_per_row.pkl": "3eb191237049", G+"rq2_sensitivity_ranking.png": "b9e24fa11686", G+"rq2_sensitivity_table.csv": "7bb5a51078a8",
        G+"sampleEntry.csv": "578b4b01d0f6", G+"vo_blanket_ess10_per_row.pkl": "b74652c6c6db", G+"vo_blanket_ess1_per_row.pkl": "747b71ed1cae",
        G+"vo_blanket_original_per_row.pkl": "c8daf42c5a0f", G+"vo_partA_retraction_per_row.pkl": "07126197c399"}
    def sha12(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()[:12]
    bad = 0
    print("1) Existing Data/ files vs Cell 74 inventory hashes")
    for rel, exp in base.items():
        p = os.path.join(R, rel)
        if not os.path.exists(p):
            print(f"  [FAIL] MISSING {rel}"); bad += 1; continue
        got = sha12(p)
        if got != exp:
            print(f"  [FAIL] CHANGED {rel}: {exp} -> {got}"); bad += 1
    print(f"  checked {len(base)} files, {bad} problem(s)" + ("" if bad else " -> all identical to inventory"))

    nb = os.path.join(R, G + "Expirements.ipynb")
    print(f"\n2) Notebook (expected to differ): sha12={sha12(nb)} (inventory 31294638ca7c) mtime={time.strftime('%H:%M', time.localtime(os.path.getmtime(nb)))} size={os.path.getsize(nb):,}")

    print("\n3) Files under Research/ modified since 15:30 today (excl venv/.git)")
    cut = time.mktime(time.strptime(time.strftime('%Y-%m-%d', time.localtime()) + " 15:30", "%Y-%m-%d %H:%M"))
    n = 0
    for dp, dns, fns in os.walk(R):
        dns[:] = [d for d in dns if d not in {'.' + 'venv', '.git', '__pycache__', '.ipynb_checkpoints'}]
        for fn in fns:
            p = os.path.join(dp, fn)
            if os.path.getmtime(p) >= cut:
                n += 1; print(f"  {time.strftime('%H:%M', time.localtime(os.path.getmtime(p)))} {os.path.getsize(p):>11,}  {os.path.relpath(p, R)}")
    print(f"  {n} file(s)  (expected: the notebook + the 24 Cell 78 files; none in Data/ other than the notebook)")

    print("\n4) Cell 83 outputs (expected ABSENT until approved)")
    for rel in ["data/processed/node_orders.json", "full_variable/tables/best_params.json"]:
        print(f"  {'EXISTS' if os.path.exists(os.path.join(R, rel)) else 'absent'}  {rel}")
    print(f"\nSUMMARY: {'PASS' if not bad else 'FAIL -> STOP and report'}")

_c84()

In [ ]:
# Cell 83 (REAL RUN, approved): save node orders + best params JSON.
# WRITES NEW files only (exclusive-create, never overwrites):
#   <ROOT>/data/processed/node_orders.json   (resolves to Data/processed/ on case-insensitive macOS)
#   <ROOT>/full_variable/tables/best_params.json
def _c83(DRY_RUN=False):
    import os, json, hashlib
    g = globals()
    R = os.environ["CREDIT_BN_ROOT"]
    print(f"DRY_RUN = {DRY_RUN}")

    DROP_VARS = ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)']
    orig = list(g['original_11_nodes'])
    print("original_11_nodes == df_train_complete.columns (same order):", orig == list(g['df_train_complete'].columns))
    assert orig == list(g['df_train_complete'].columns), "Node order does not match df_train_complete columns"
    missing = [v for v in DROP_VARS if v not in orig]
    assert not missing, "Drop names do not match node names; fix before saving"
    derived = [n for n in orig if n not in DROP_VARS]
    print("derived 7-node order:", derived, "| n =", len(derived))
    assert len(derived) == 7 and 'Default' in derived
    KEEP_VARS = [n for n in derived if n != 'Default']

    node_orders = {
        'original_11_nodes': orig,
        'derived_7_nodes_selection': derived,
        'DROP_VARS': DROP_VARS,
        'KEEP_VARS': KEEP_VARS,
        'col_order': list(g['col_order']) if 'col_order' in g else None,
        'model_cols': list(g['model_cols']) if 'model_cols' in g else None,
        'VARS12': list(g['VARS12']) if 'VARS12' in g else None,
        'note': "Derived order = original_11_nodes with DROP_VARS removed, relative order preserved (planning-instance decision 1a).",
    }
    bp = {}
    for k, nm, fin in [('LR', 'lr_grid_search', 'lr_final'), ('RF', 'rf_grid_search_restricted', 'rf_final_restricted')]:
        gs = g[nm]
        bp[k] = {'best_params': gs.best_params_, 'cv_best_score_auc': float(gs.best_score_),
                 'scoring': gs.scoring, 'cv': str(gs.cv), 'random_state': g[fin].get_params().get('random_state'),
                 'final_model_params': {a: (b if isinstance(b, (int, float, str, bool, type(None))) else str(b))
                                        for a, b in g[fin].get_params().items()}}
    items = [("data/processed/node_orders.json", node_orders), ("full_variable/tables/best_params.json", bp)]

    print(f"\n{'destination':48s} {'status':14s} {'bytes':>8s}  sha12")
    for rel, obj in items:
        dest = os.path.join(R, rel)
        if os.path.exists(dest):
            print(f"{rel:48s} {'EXISTS->SKIP':14s}"); continue
        data = json.dumps(obj, indent=2, default=str).encode()
        h = hashlib.sha256(data).hexdigest()[:12]
        status = 'would write'
        if not DRY_RUN:
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            with open(dest, 'xb') as fh:
                fh.write(data)
            with open(dest, 'rb') as fh:
                status = 'WRITTEN ok' if hashlib.sha256(fh.read()).hexdigest()[:12] == h else 'SHA MISMATCH'
        print(f"{rel:48s} {status:14s} {len(data):8d}  {h}")
    print("\nbest params:", {k: v['best_params'] for k, v in bp.items()})

_c83(DRY_RUN=False)